# Assignement INTRO


```
gDrive_home/
├── Colab Notebooks/
│   └── NLP_assignment/
│       ├── PoliMillionaire.ipynb <-- Your notebook
│       └── millionaire_client/ <-- Directory provided
```


URL: configured via the `api-url` Colab Secret (HTTPS required)

### Game interaction

In [ ]:
from google.colab import drive
import os
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


In [ ]:
import sys
# Define the path to the directory containing your package
package_parent_dir = '/content/gdrive/MyDrive/NLP/PoliMillionaire/NLP_assignment/'

# Append to sys.path if it is not already present
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

# Verify the path was added
print(sys.path)

['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/gdrive/MyDrive/NLP/PoliMillionaire/NLP_assignment/']


Let's import the client classes

In [ ]:
from millionaire_client import MillionaireClient, AuthenticationError

You can save your password in a Colab secret (the "key" icon on the tab on the left) and import it into your notebook.

In [ ]:
from google.colab import userdata
pwd = userdata.get('poli-millionaire')
usr = userdata.get('username')

Now keep the API_URL as stated, but please change the username and password to be the ones you used during sign up session.

In [ ]:
API_URL = userdata.get('api-url')
if not API_URL or not API_URL.startswith("https://"):
    raise ValueError("The quiz API URL must use HTTPS.")
username = usr
password = pwd

Now we can instantiate a MillionaireClient object and call the login method, which takes as parameters username and password.

In [ ]:
client = MillionaireClient(API_URL)
try:
    user = client.login(username, password)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")


Welcome, AleAssini! (Role: student)


Types of competition


In [ ]:
# List available competitions
print("\n=== Available Competitions ===")
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"  {comp.id}: {comp.name} ({comp.max_levels} questions)")


=== Available Competitions ===
  0: Entertainment (15 questions)
  1: Ancient History and Politics (15 questions)
  2: Science and Nature (15 questions)
  3: Maths (15 questions)
  4: Philosophy and Psychology (15 questions)
  5: News (15 questions)


##Competition

In [ ]:
# Choose a competition ID
comp_id = 1
chosen_comp = competitions[comp_id].name
print(f"Chosen competition: {chosen_comp}")

Chosen competition: Ancient History and Politics


## Show Leaderboard


In [ ]:
# Show leaderboard position
lb = client.leaderboard.get(competition_id=comp_id, limit=10)
print(f"\n=== Leaderboard for {lb.competition.name} ===")
for i, entry in enumerate(lb.entries[:10], 1):
    marker = " <-- YOU" if entry.username == username else ""
    print(f"  {i}. {entry.username}: ${entry.score:,.2f} (Level {entry.reached_level}){marker}")


=== Leaderboard for Ancient History and Politics ===
  1. luca_bordin: $1,024,000.00 (Level 15)
  2. zahra: $1,024,000.00 (Level 15)
  3. El_Marzel: $1,024,000.00 (Level 15)
  4. gvin: $1,024,000.00 (Level 15)
  5. Bilal_jaiel: $1,024,000.00 (Level 15)
  6. Zero37: $1,024,000.00 (Level 15)
  7. gabrielep: $1,024,000.00 (Level 15)
  8. dany: $1,024,000.00 (Level 15)
  9. MatteoVitali: $1,024,000.00 (Level 15)
  10. Mario: $1,024,000.00 (Level 15)


# LLMs

## General settings

Models we want to try by category:

Small (< 5B parameters):

*   microsoft/Phi-3.5-mini-instruct (thinking)
*   google/gemma-3-4b-it
*   Qwen/Qwen2.5-3B-Instruct
*   meta-llama/Llama-3.2-3B-Instruct


Medium (5B - 10B parameters):

*   meta-llama/Meta-Llama-3-8B-Instruct (thinking)
*   mistralai/Mistral-7B-v0.3 (non istruct)
*   mistralai/Mistral-7B-Instruct-v0.3
*   google/gemma-2-9b
*   google/gemma-7b-it
*   Qwen/Qwen3.5-9B
*   Qwen/Qwen2.5-7B-Instruct

Large (10B+ parameters):

*   Qwen/Qwen-2.5-14B (non instr)
*   Qwen/Qwen2.5-14B-Instruct
*   google/gemma-2-27b-it

Thinking/Reasoning Models:
* deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
* deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
* microsoft/Phi-3.5-mini-instruct  (GIA PROVATO )
* Qwen/Qwen2.5-7B-Instruct (GIA PROVATO )

Mixture of Experts / Dense

* mistralai/Mixtral-8x7B-Instruct-v0.1 (pesante 47B)
* Qwen/Qwen1.5-MoE-A2.7B (A-> Active params)
* google/gemma-4-E4B-it (E-> Effective Params)


In [ ]:
MODELS_LIST = [
    "microsoft/Phi-3.5-mini-instruct",
    "google/gemma-3-4b-it",
    "Qwen/Qwen2.5-3B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "meta-llama/Meta-Llama-3-8B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "Qwen/Qwen2.5-14B-Instruct",
]

STRATEGIES_LIST = [
    "zero_shot",
    "few_shot",
    "cot",
]

##CONFIG

In [ ]:
import torch

CONFIG = {
    # SELECT HERE THE MODEL TO RUN
    "model_id": "Qwen/Qwen2.5-3B-Instruct",

    #quantization
    "quantization_bits": None, # 4, 8 or None to default 16

    # Select here the strategy to apply
    "prompt_strategy": "few_shot",
    "n-shots": 1, # set number of examples to give in the prompt
    "temperature": 1.0,  # sampling temperature (1.0 = default)
    "do_sample": False,

    # generation
    "max_new_tokens": 512,
    "reasoning_max_new_tokens": 512,

    "model_kwargs": {
        "device_map": "auto", # "auto" , "cuda"
        "torch_dtype": "auto",  # "auto", torch.float16, torch.bfloat16, or torch.float32
        "trust_remote_code": True, # Autorizzo Hugging Face
        "attn_implementation": "eager",
    }
}

#LLM loader

##Helper commands


###Phi helpers

In [ ]:
model_id_lower = CONFIG["model_id"].lower()

if model_id_lower.startswith("microsoft/phi-3.5"):
    !pip uninstall -y transformers accelerate
    !pip install transformers==4.43.3 accelerate==0.33.0
    !rm -rf ~/.cache/huggingface/modules

###Llama helper

In [ ]:
model_id_lower = CONFIG["model_id"].lower()

if (model_id_lower.startswith("meta-llama/llama-3.2") or model_id_lower.startswith("meta-llama/meta-llama-3")):
    CONFIG["model_kwargs"]["attn_implementation"] = "sdpa"

###Quantization helper

In [ ]:
import torch
import subprocess
import sys

bits = CONFIG.get("quantization_bits")

if bits in {4, 8}:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-U", "transformers", "accelerate", "bitsandbytes"],
        check=True
    )

## Model Loader

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def load_model(config):
    model_id = config["model_id"]
    kwargs = config["model_kwargs"].copy()

    # attn_implementation controlla come viene calcolata l’attenzione dentro il Transformer.
    if model_id.lower().startswith("microsoft/phi"):
        kwargs["attn_implementation"] = "eager"


    # quantization
    bits = config.get("quantization_bits")

    if bits == 4:
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True)
    elif bits == 8:
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)


    display_bits = f"{bits}-bit" if bits else "None (Native/16-bit)"
    print(f"Loading model: {model_id} | Quantization: {display_bits}")

    # 3. Load Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=kwargs.get("trust_remote_code", False)
    )

    # 4. Load Model
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        **kwargs
    )

    return tokenizer, model

In [ ]:
# ============================================================
# LOAD MODEL (USING CONFIG)
# ============================================================

llm_tokenizer, llm_model = load_model(CONFIG)

print("Model loaded successfully.")
print("Model name:", CONFIG["model_id"])
print("CUDA available:", torch.cuda.is_available())

try:
    print("Model device:", llm_model.device)
except:
    print("Model device: device_map='auto'")

Loading model: Qwen/Qwen2.5-3B-Instruct | Quantization: None (Native/16-bit)


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully.
Model name: Qwen/Qwen2.5-3B-Instruct
CUDA available: True
Model device: cuda:0


# CHATBOT 🤖

## Prompting

In [ ]:
# Small DB where examples can be added and then provided for n-shots prompting strategies.

EXAMPLES_DB = {
    "Entertainment": [
        {"q": "Who directed the movie 'Inception'?", "o": "[0] Steven Spielberg\n[1] Christopher Nolan\n[2] Quentin Tarantino\n[3] James Cameron", "a": "1", "r": "'Inception' is a 2010 sci-fi film directed by Christopher Nolan. It is option 1."},
        {"q": "Which band released the album 'Abbey Road'?", "o": "[0] The Rolling Stones\n[1] Led Zeppelin\n[2] The Beatles\n[3] Pink Floyd", "a": "2", "r": "'Abbey Road' is the eleventh studio album by the English rock band The Beatles. It is option 2."},
        {"q": "Which actor played the character of Iron Man in the Marvel Cinematic Universe?", "o": "[0] Chris Evans\n[1] Robert Downey Jr.\n[2] Chris Hemsworth\n[3] Mark Ruffalo", "a": "1", "r": "Robert Downey Jr. portrayed Tony Stark, also known as Iron Man, starting in 2008. It is option 1."}
    ],
    "Ancient History and Politics": [
        {"q": "In which year did the French Revolution begin?", "o": "[0] 1776\n[1] 1789\n[2] 1812\n[3] 1492", "a": "1", "r": "The French Revolution began in 1789 with the Storming of the Bastille. It is option 1."},
        {"q": "Who was the first President of the United States?", "o": "[0] Thomas Jefferson\n[1] Abraham Lincoln\n[2] George Washington\n[3] John Adams", "a": "2", "r": "George Washington served as the first president from 1789 to 1797. It is option 2."},
        {"q": "Which ancient civilization is famous for building the Great Pyramid of Giza?", "o": "[0] The Romans\n[1] The Greeks\n[2] The Egyptians\n[3] The Sumerians", "a": "2", "r": "The Great Pyramid was built as a tomb for the Pharaoh Khufu by the Ancient Egyptians. It is option 2."}
    ],
    "Science and Nature": [
        {"q": "What is the chemical symbol for Gold?", "o": "[0] Gd\n[1] Ag\n[2] Au\n[3] Fe", "a": "2", "r": "The symbol Au comes from the Latin word for gold, 'aurum'. It is option 2."},
        {"q": "Which part of the cell is known as the powerhouse?", "o": "[0] Nucleus\n[1] Ribosome\n[2] Mitochondria\n[3] Golgi apparatus", "a": "2", "r": "Mitochondria generate most of the cell's supply of adenosine triphosphate (ATP). It is option 2."},
        {"q": "What is the closest planet to the Sun?", "o": "[0] Venus\n[1] Mars\n[2] Mercury\n[3] Earth", "a": "2", "r": "Mercury is the smallest and innermost planet in the Solar System. It is option 2."}
    ],
    "Maths": [
        {"q": "What is the square root of 144?", "o": "[0] 10\n[1] 11\n[2] 12\n[3] 14", "a": "2", "r": "12 multiplied by 12 equals 144. It is option 2."},
        {"q": "Solve for x: 2x + 5 = 15", "o": "[0] 5\n[1] 10\n[2] 15\n[3] 20", "a": "0", "r": "Subtract 5 from 15 to get 10, then divide by 2. x = 5. It is option 0."},
        {"q": "What is the value of 5 factorial (5!)?", "o": "[0] 60\n[1] 100\n[2] 120\n[3] 150", "a": "2", "r": "The factorial of 5 is 5 * 4 * 3 * 2 * 1, which equals 120. It is option 2."}
    ],
    "Philosophy and Psychology": [
        {"q": "Who is considered the founder of psychoanalysis?", "o": "[0] Carl Rogers\n[1] Sigmund Freud\n[2] B. F. Skinner\n[3] Jean Piaget", "a": "1", "r": "Sigmund Freud developed psychoanalysis as a theory and therapeutic method. It is option 1."},
        {"q": "What term describes learning through rewards and punishments?", "o": "[0] Classical conditioning\n[1] Operant conditioning\n[2] Free association\n[3] Introspection", "a": "1", "r": "Operant conditioning is learning shaped by reinforcement and punishment. It is option 1."},
        {"q": "Which philosopher wrote 'The Republic'?", "o": "[0] Aristotle\n[1] Plato\n[2] Socrates\n[3] Descartes", "a": "1", "r": "'The Republic' is a philosophical work by Plato. It is option 1."}
    ],
    "News": [
        {"q": "What organization is responsible for international public health coordination?", "o": "[0] NATO\n[1] WHO\n[2] IMF\n[3] WTO", "a": "1", "r": "The World Health Organization coordinates international public health. It is option 1."},
        {"q": "Which institution is the central bank of the United States?", "o": "[0] World Bank\n[1] Federal Reserve\n[2] European Central Bank\n[3] International Monetary Fund", "a": "1", "r": "The Federal Reserve is the central banking system of the United States. It is option 1."},
        {"q": "What does UN stand for?", "o": "[0] United Nations\n[1] Union Network\n[2] Universal Nation\n[3] United Neutrality", "a": "0", "r": "UN stands for United Nations, an international organization founded in 1945. It is option 0."}
    ]
}

In [ ]:
import random
def get_examples(n=1):
    category_list = EXAMPLES_DB.get(chosen_comp)
    # Return n samples, or the whole list if n is larger than available questions
    return random.sample(category_list, min(n, len(category_list)))

In [ ]:
def format_options(options):
    return "\n".join([f"[{opt.id}] {opt.text}" for opt in options])

In [ ]:
# Prompt builders

def build_zero_shot_prompt(question_text, options):
    options_text = format_options(options)

    system_message = (
        "You are playing a multiple-choice quiz game.\n"
        f"You are an expert of {chosen_comp}.\n"
        "Choose the correct answer for the question delimited by triple backticks.\n\n"
        "Rules:\n"
        "- Return only the option ID.\n"
        "- Do not explain.\n"
        "- Do not write anything except one number."
    )

    user_message = (
        f"```{question_text}```\n\n"
        f"Options:\n{options_text}\n\n"
        "Answer:"
    )
    return system_message, user_message

def build_few_shot_prompt(question_text, options):
    options_text = format_options(options)
    n_shots = CONFIG["n-shots"]
    examples = get_examples(n_shots)

    example_blocks = []
    for ex in examples:
        block = f"```{ex['q']}```\nOptions:\n{ex['o']}\nAnswer: {ex['a']}"
        example_blocks.append(block)

    system_message = (
        "You are playing a multiple-choice quiz game.\n"
        f"You are an expert of {chosen_comp}.\n"
        "Choose the correct answer for the question delimited by triple backticks.\n\n"
        "Rules:\n"
        "- Return only the option ID.\n"
        "- Do not explain.\n"
        "- Do not write anything except one number.\n\n"
        "Examples:\n\n" + "\n\n".join(example_blocks)
    )

    user_message = (
        f"```{question_text}```\n\n"
        f"Options:\n{options_text}\n\n"
        "Answer:"
    )

    return system_message, user_message


def build_cot_prompt(question_text, options):
    options_text = format_options(options)
    n_shots = CONFIG["n-shots"]
    examples = get_examples(n_shots)

    example_blocks = []
    for ex in examples:
        block = (
            f"```{ex['q']}```\n"
            f"Options:\n{ex['o']}\n"
            f"Reasoning: {ex['r']}\n"
            f"Answer: {ex['a']}"
        )
        example_blocks.append(block)

    system_message = (
        "You are playing a multiple-choice quiz game.\n"
        f"You are an expert of {chosen_comp}.\n"
        "To answer the question delimited by triple backticks, follow these steps:\n"
        "Step 1 - Read the question carefully.\n"
        "Step 2 - Consider each option and reason about it.\n"
        "Step 3 - Select the most plausible answer.\n"
        "Step 4 - Output your answer using this format:\n"
        "Reasoning: <your reasoning>\n"
        "Answer: <option ID>\n\n"
        "Examples:\n\n" + "\n\n".join(example_blocks)
    )

    user_message = (
        f"```{question_text}```\n\n"
        f"Options:\n{options_text}\n\n"
        "Reasoning:"
    )

    return system_message, user_message

In [ ]:
PROMPT_BUILDERS = {
    "zero_shot": build_zero_shot_prompt,
    "few_shot": build_few_shot_prompt,
    "cot": build_cot_prompt,
}

def build_prompt(question_text, options):
    prompt_strategy = CONFIG["prompt_strategy"]

    if prompt_strategy not in PROMPT_BUILDERS:
        #Default to zero-shot prompt strategy
        prompt_strategy = "zero_shot"

    if prompt_strategy in {"few_shot", "cot"}:
        print(f"Using {prompt_strategy} prompting strategy with {CONFIG["n-shots"]} examples")
        return PROMPT_BUILDERS[prompt_strategy](question_text, options)

    print(f"Using {prompt_strategy} prompting strategy")
    return PROMPT_BUILDERS[prompt_strategy](question_text, options)


###Dynamic prompting

In [ ]:
# Dynamic prompting configuration

DYNAMIC_PROMPTING = {
    "enabled": False,

    # Prompt used from level 1 up to "switch_after_level"
    "first_prompt_strategy": "zero_shot",

    # Last level in which the first prompt is used
    "switch_after_level": 7,

    # Prompt used from the next level onward
    "second_prompt_strategy": "cot",
}


def get_prompt_strategy_for_level(current_level):
    if not DYNAMIC_PROMPTING["enabled"]:
        return CONFIG["prompt_strategy"]

    if current_level <= DYNAMIC_PROMPTING["switch_after_level"]:
        return DYNAMIC_PROMPTING["first_prompt_strategy"]

    return DYNAMIC_PROMPTING["second_prompt_strategy"]

## Extraction options

For different Prompting Strategies


In [ ]:
import json

def extract_option_id_json(output, valid_ids):
    try:
        match = re.search(r'\{.*\}', output, re.DOTALL)
        if match:
            data = json.loads(match.group())
            ans = int(data.get("answer"))
            if ans in valid_ids:
                return ans
    except:
        pass
    return None

def extract_option_id_cot(output, valid_ids):
    match = re.search(r'(?:Answer|Answer is)[\s:]*([0-9]+)', output, re.IGNORECASE)
    if match:
        ans = int(match.group(1))
        if ans in valid_ids:
            return ans
    numbers = re.findall(r'\b\d+\b', output)
    if numbers:
        ans = int(numbers[-1])
        if ans in valid_ids:
            return ans
    return None

def extract_option_id(output, valid_ids):
    match = re.search(r'(?:Answer|Answer is)[\s:]*([0-9]+)', output, re.IGNORECASE)
    if match:
        ans = int(match.group(1))
        if ans in valid_ids:
            return ans
    numbers = re.findall(r'\b\d+\b', output)
    if numbers:
        ans = int(numbers[0])
        if ans in valid_ids:
            return ans
    return None

## Answer preparation

In [ ]:
import re
def generate_answer(prompt, tokenizer, model, do_sample=True, max_new_tokens=15, temperature=1.0):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[-1]
    #prompt_len = len(prompt)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            #use_cache=False
        )

    # We cut-off the "raw part" and return only the reasoning +
    # Answer: []

    pruned_output = tokenizer.decode(
        outputs[0][prompt_len:],
        skip_special_tokens=True
    ).strip()

    return pruned_output


def parse_answer(output, valid_ids):

    prompt_strategy = CONFIG["prompt_strategy"]

    if prompt_strategy == "zero_shot_cot":
        return extract_option_id_json(output, valid_ids)

    if prompt_strategy == "cot":
        return extract_option_id_cot(output, valid_ids)

    return extract_option_id(output, valid_ids)

def get_llm_answer(question, tokenizer, model, max_retries=3):

    valid_ids = [opt.id for opt in question.options]
    system_msg, user_msg = build_prompt(question.text , question.options)

    #For Debug
    #print(f"System message:\n {system_msg}")

    # Applica il Chat Template
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    max_new_tokens = CONFIG.get("max_new_tokens", 15)

    # Assign higher number of tokens if reasoning is required
    if CONFIG["prompt_strategy"] in {"zero_shot_cot", "cot"}:
        max_new_tokens = CONFIG.get("reasoning_max_new_tokens", 256)


    # retry if the llm fails 3 times
    for _ in range(max_retries):

        output = generate_answer(prompt, tokenizer, model, do_sample=CONFIG.get("do_sample",True), max_new_tokens=max_new_tokens, temperature=CONFIG.get("temperature", 1.0))

        print(f"Output: (Reasoning, if selected, + Answer):\n {output}")
        answer = parse_answer(output, valid_ids)

        #print(f"\nLA RISPOSTA QUA è: {answer}")
        if answer is not None:
            return answer

    # fallback to
    print("\nFallback...")
    return valid_ids[0]

In [ ]:
def play_game_llm(game, tokenizer, model):
    while game.in_progress:
        question = game.current_question
        if not question:
            print("No question available.")
            break

        print(f"\n--- Level {game.current_level} ---")
        print(f"Q: {question.text}\n")

        for opt in question.options:
            print(f"  [{opt.id}] {opt.text}")

        time_left = game.time_remaining
        if time_left:
            print(f"\nTime remaining: {time_left:.1f}s")

        # Select prompt strategy dynamically according to the current level
        CONFIG["prompt_strategy"] = get_prompt_strategy_for_level(game.current_level)
        print(f"Active prompt strategy: {CONFIG['prompt_strategy']}")

        # LLM answering
        answer_id = get_llm_answer(question, tokenizer, model)

        result = game.answer(answer_id)

        if result.correct:
            print("CORRECT!")
            if result.game_over:
                print("\nCONGRATULATIONS!")
                print(f"Final earnings: ${result.earned_amount:,.2f}")
            else:
                print(f"Earned so far: ${result.earned_amount:,.2f}")

        elif result.timed_out:
            print("TIMED OUT!")
            print(f"\nGame Over!")
            print(f"Final earnings: ${result.earned_amount:,.2f}")

        else:
            print("WRONG ANSWER!")
            print(f"\nGame Over!")
            print(f"Final earnings: ${result.earned_amount:,.2f}")

    print("\n=== Game Summary ===")
    print(f"Reached Level: {game.current_level}")
    print(f"Total Earnings: ${game.earned_amount:,.2f}")

# Play game with LLM

In [ ]:
game = client.game.start(competition_id=comp_id)
play_game_llm(game,llm_tokenizer, llm_model)


--- Level 1 ---
Q: How did the legal status of a freed slave (libertus) differ from that of a freeborn Roman citizen?

  [0] A freed slave retained their former master's power over them (potestas).
  [1] A freed slave could not vote but could hold public office, unlike a freeborn citizen.
  [2] A freed slave could vote and hold public office, just like a freeborn citizen.
  [3] A freed slave could vote but could not hold public office, unlike a freeborn citizen.

Time remaining: 29.9s
Active prompt strategy: few_shot
Using few_shot prompting strategy with 1 examples
Output: (Reasoning, if selected, + Answer):
 3
CORRECT!
Earned so far: $100.00

--- Level 2 ---
Q: What is the primary basis for moral and social obligation in Plato's Republic?

  [0] The pursuit of pleasure and happiness.
  [1] Following the laws and customs of society.
  [2] Power and control over others.
  [3] The knowledge of the Form of the Good.

Time remaining: 29.9s
Active prompt strategy: few_shot
Using few_shot 

#RAG 🌱

# Common Setup

These cells build the shared local Wikipedia retrieval index and define the common RAG prompt, answer parser integration, and one-game helper used by all variants.


In [ ]:
BASE_RAG_CONFIG = {
    "retrieval_method": "wikipedia_local",

    # retrieval
    "top_k_context": 3,

    # generation
    "rag_max_new_tokens": 64,

    # debug
    "debug": True,
}

In [ ]:
import os
import json
import gzip
import pickle
import torch

In [ ]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 74.8 MB/s eta 0:00:00


In [ ]:
import faiss
import numpy as np
import os

In [ ]:
SEMANTIC_SEARCH_DIR = "/content/gdrive/MyDrive/NLP/PoliMillionaire/SemanticSearch"

os.makedirs(SEMANTIC_SEARCH_DIR, exist_ok=True)

print("Semantic Search cache folder:")
print(SEMANTIC_SEARCH_DIR)

Semantic Search cache folder:
/content/gdrive/MyDrive/NLP/PoliMillionaire/SemanticSearch


In [ ]:
from sentence_transformers import SentenceTransformer, util

SEMB_MODEL_NAME = 'multi-qa-MiniLM-L6-cos-v1'
semb_model = SentenceTransformer(SEMB_MODEL_NAME)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/383 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
WIKIPEDIA_FILENAME = "simplewiki-2020-11-01.jsonl.gz"
wikipedia_filepath = os.path.join(SEMANTIC_SEARCH_DIR, WIKIPEDIA_FILENAME)

if not os.path.exists(wikipedia_filepath):
    print("Wikipedia dump not found on Drive. Downloading...")
    util.http_get(
        "http://sbert.net/datasets/simplewiki-2020-11-01.jsonl.gz",
        wikipedia_filepath
    )
else:
    print("Wikipedia dump already found on Drive. Skipping download.")

print("Wikipedia file path:")
print(wikipedia_filepath)

Wikipedia dump already found on Drive. Skipping download.
Wikipedia file path:
/content/gdrive/MyDrive/NLP/PoliMillionaire/SemanticSearch/simplewiki-2020-11-01.jsonl.gz


In [ ]:
MAX_PARAGRAPHS_PER_ARTICLE = 10

In [ ]:
# Load paragraphs from SimpleWiki

import json
import gzip

passages = []
passage_metadata = []

with gzip.open(wikipedia_filepath, 'rt', encoding='utf8') as f:
    for line in f:
        data = json.loads(line.strip())

        title = data.get("title", "").strip()
        paragraphs = data.get("paragraphs", [])

        for paragraph_id, paragraph in enumerate(paragraphs[:MAX_PARAGRAPHS_PER_ARTICLE]):
            paragraph = paragraph.strip()

            if paragraph:
                passages.append(f"{title}\n\n{paragraph}")
                passage_metadata.append({
                    "title": title,
                    "paragraph_id": paragraph_id
                })

print(f"Retrieved {len(passages)} passages")

Retrieved 452648 passages


In [ ]:
EMBEDDINGS_CACHE_PATH = os.path.join(
    SEMANTIC_SEARCH_DIR,
    f"corpus_embeddings_{SEMB_MODEL_NAME}_first_{MAX_PARAGRAPHS_PER_ARTICLE}_paragraphs_title_plus_paragraph.pt"
)
EMBEDDINGS_SIGNATURE_PATH = EMBEDDINGS_CACHE_PATH.replace(".pt", ".sig.json")

_CACHE_SIGNATURE = {
    "model": SEMB_MODEL_NAME,
    "max_paragraphs_per_article": MAX_PARAGRAPHS_PER_ARTICLE,
    "wikipedia_file": WIKIPEDIA_FILENAME,
    "passage_format": "title_plus_paragraph",
}

def _signature_matches():
    if not os.path.exists(EMBEDDINGS_SIGNATURE_PATH):
        return False
    with open(EMBEDDINGS_SIGNATURE_PATH) as f:
        return json.load(f) == _CACHE_SIGNATURE

if os.path.exists(EMBEDDINGS_CACHE_PATH) and _signature_matches():
    print("Embeddings already found. Loading from Drive...")

    corpus_embeddings = torch.load(
        EMBEDDINGS_CACHE_PATH,
        map_location="cuda" if torch.cuda.is_available() else "cpu"
    )

else:
    print("Embeddings not found. Computing embeddings...")

    corpus_embeddings = semb_model.encode(
        passages,
        convert_to_tensor=True,
        show_progress_bar=True
    )

    print("Saving embeddings to Drive...")
    torch.save(corpus_embeddings, EMBEDDINGS_CACHE_PATH)
    with open(EMBEDDINGS_SIGNATURE_PATH, "w") as f:
        json.dump(_CACHE_SIGNATURE, f)

Embeddings already found. Loading from Drive...


In [ ]:
# Create or load FAISS index for local Wikipedia retrieval

import faiss
import numpy as np
import os

FAISS_INDEX_PATH = os.path.join(
    SEMANTIC_SEARCH_DIR,
    f"faiss_index_{SEMB_MODEL_NAME}_first_{MAX_PARAGRAPHS_PER_ARTICLE}_paragraphs.index"
)

# SentenceTransformer embeddings with multi-qa-MiniLM-L6-cos-v1 are usually compared with cosine similarity.
# FAISS IndexFlatIP uses inner product, so we normalize embeddings to make inner product equivalent to cosine similarity.

if os.path.exists(FAISS_INDEX_PATH):
    print("FAISS index already found. Loading from Drive...")
    faiss_index = faiss.read_index(FAISS_INDEX_PATH)

else:
    print("FAISS index not found. Creating FAISS index...")

    # Move embeddings to CPU and convert to numpy
    corpus_embeddings_np = corpus_embeddings.detach().cpu().numpy().astype("float32")

    # Normalize embeddings for cosine similarity
    faiss.normalize_L2(corpus_embeddings_np)

    embedding_dim = corpus_embeddings_np.shape[1]

    # IndexFlatIP = exact inner product search
    faiss_index = faiss.IndexFlatIP(embedding_dim)
    faiss_index.add(corpus_embeddings_np)

    print("Saving FAISS index to Drive...")
    faiss.write_index(faiss_index, FAISS_INDEX_PATH)

print("FAISS index size:", faiss_index.ntotal)

FAISS index already found. Loading from Drive...
FAISS index size: 452648


In [ ]:
from sentence_transformers import util
def retrieve_local_wikipedia_context(question, top_k=3):
    """
    Baseline local Wikipedia retrieval using FAISS.

    Given a quiz question, retrieve the top-k most similar passages
    from the local SimpleWiki FAISS index.
    """

    query = question.text

    query_embedding = semb_model.encode(
        query,
        convert_to_tensor=False
    ).astype("float32")

    # FAISS expects shape: (num_queries, embedding_dim)
    query_embedding = np.expand_dims(query_embedding, axis=0)

    # Normalize query embedding for cosine similarity
    faiss.normalize_L2(query_embedding)

    scores, indices = faiss_index.search(query_embedding, top_k)

    context_blocks = []
    hits = []

    for rank, (corpus_id, score) in enumerate(zip(indices[0], scores[0]), start=1):
        corpus_id = int(corpus_id)
        score = float(score)

        title = passage_metadata[corpus_id]["title"]
        paragraph_id = passage_metadata[corpus_id]["paragraph_id"]
        passage_text = passages[corpus_id]

        hits.append({
            "corpus_id": corpus_id,
            "score": score,
            "query": query,
            "used_options": False,
        })

        context_blocks.append(
            f"[Source {rank}] Title: {title} | Paragraph: {paragraph_id} | Score: {score:.4f}\n"
            f"{passage_text}"
        )

    context = "\n\n".join(context_blocks)

    return context, hits

In [ ]:
def build_rag_prompt(question_text, options, context):
    options_text = format_options(options)

    system_message = (
        "You are playing a multiple-choice quiz game.\n"
        f"You are an expert of {chosen_comp}.\n"
        "You are given external context retrieved from a local Wikipedia corpus.\n"
        "Use the context only if it is relevant to the question and the answer options.\n"
        "If the context is weak or unrelated, rely on your own knowledge.\n\n"
        "Rules:\n"
        "- Choose the correct option ID.\n"
        "- Return only the option ID.\n"
        "- Do not explain.\n"
        "- Do not write anything except one number."
    )

    user_message = (
        f"Context:\n{context}\n\n"
        f"Question:\n```{question_text}```\n\n"
        f"Options:\n{options_text}\n\n"
        "Answer:"
    )

    return system_message, user_message

In [ ]:
def get_rag_answer(question, tokenizer, model, max_retries=3):
    valid_ids = [opt.id for opt in question.options]

    context, hits = retrieve_local_wikipedia_context(
        question,
        top_k=BASE_RAG_CONFIG.get("top_k_context", 3)
    )

    if BASE_RAG_CONFIG.get("debug", True):
        print("\nRetrieved context:")
        print("-" * 80)

        for rank, hit in enumerate(hits, start=1):
            corpus_id = hit["corpus_id"]
            score = float(hit["score"])

            title = passage_metadata[corpus_id]["title"]
            paragraph_id = passage_metadata[corpus_id]["paragraph_id"]
            preview = passages[corpus_id].replace("\n", " ")

            print(f"[{rank}] {title} | paragraph {paragraph_id} | score={score:.4f}")
            print(f"{preview}...")
            print()

    system_msg, user_msg = build_rag_prompt(
        question.text,
        question.options,
        context
    )

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    max_new_tokens = BASE_RAG_CONFIG.get("rag_max_new_tokens", 64)

    for _ in range(max_retries):
        output = generate_answer(
            prompt,
            tokenizer,
            model,
            do_sample=CONFIG.get("do_sample", False),
            max_new_tokens=max_new_tokens,
            temperature=CONFIG.get("temperature", 1.0)
        )

        print(f"Output:\n{output}")

        answer = extract_option_id(output, valid_ids)

        if answer is not None:
            return answer

    print("\nRAG parsing fallback...")
    return valid_ids[0]

In [ ]:
def play_game_rag(game, tokenizer, model):
    while game.in_progress:
        question = game.current_question

        if not question:
            print("No question available.")
            break

        print(f"\n--- Level {game.current_level} ---")
        print(f"Q: {question.text}\n")

        for opt in question.options:
            print(f"  [{opt.id}] {opt.text}")

        time_left = game.time_remaining
        if time_left is not None:
            print(f"\nTime remaining: {time_left:.1f}s")

        # For RAG baseline we keep a fixed RAG prompt.
        # Dynamic prompting can be added later if needed.
        print("Active strategy: local_wikipedia_rag")

        answer_id = get_rag_answer(question, tokenizer, model)

        print(f"\nSelected answer id: {answer_id}")

        result = game.answer(answer_id)

        if result.correct:
            print("CORRECT!")

            if result.game_over:
                print("\nCONGRATULATIONS!")
                print(f"Final earnings: ${result.earned_amount:,.2f}")
            else:
                print(f"Earned so far: ${result.earned_amount:,.2f}")

        elif result.timed_out:
            print("TIMED OUT!")
            print("\nGame Over!")
            print(f"Final earnings: ${result.earned_amount:,.2f}")

        else:
            print("WRONG ANSWER!")
            print("\nGame Over!")
            print(f"Final earnings: ${result.earned_amount:,.2f}")

    print("\n=== Game Summary ===")
    print(f"Reached Level: {game.current_level}")
    print(f"Total Earnings: ${game.earned_amount:,.2f}")

In [ ]:
# Baseline aliases used by the final configuration selector.
# They allow returning to the original baseline after later variants override globals.
BASELINE_RETRIEVE_LOCAL_WIKIPEDIA_CONTEXT = retrieve_local_wikipedia_context
BASELINE_GET_RAG_ANSWER = get_rag_answer


# 🔵 Baseline

## 🟢 Question-Only Retrieval

## 🟢 Question + Answer Options Retrieval


In [ ]:
def retrieve_local_wikipedia_context_with_options(question, top_k=3):
    question_text = question.text.strip()

    if hasattr(question, "options") and question.options is not None:
        options_text = " ".join([opt.text for opt in question.options])
    else:
        options_text = ""

    # Build retrieval query
    # "Who discovered penicillin? Isaac Newton Alexander Fleming Marie Curie Charles Darwin"
    query = f"{question_text} {options_text}".strip()

    query_embedding = semb_model.encode(
        query,
        convert_to_tensor=False
    ).astype("float32")

    query_embedding = np.expand_dims(query_embedding, axis=0)
    faiss.normalize_L2(query_embedding)

    # Retrieve top-k most similar Wikipedia passages
    # indices = [[1523, 9871, 330]]
    # scores  = [[0.8123, 0.7641, 0.7018]]
    scores, indices = faiss_index.search(query_embedding, top_k)

    context_blocks = []
    hits = []

    for rank, (corpus_id, score) in enumerate(zip(indices[0], scores[0]), start=1):
        corpus_id = int(corpus_id)
        score = float(score)

        # Get passage metadata and text
        title = passage_metadata[corpus_id]["title"]
        paragraph_id = passage_metadata[corpus_id]["paragraph_id"]
        passage_text = passages[corpus_id]

        hits.append({
            "corpus_id": corpus_id,
            "score": score,
            "query": query,
            "used_options": True,
        })

        context_blocks.append(
            f"[Source {rank}] Title: {title} | Paragraph: {paragraph_id} | Score: {score:.4f}\n"
            f"{passage_text}"
        )

    context = "\n\n".join(context_blocks)
    return context, hits


## 🟢 Reliable-RAG: Relevance Filtering

In [ ]:
import numpy as np

In [ ]:
# Helpers

def shorten_text(text, max_chars=900):
    # Shorten long passages for the relevance judge
    text = text.replace("\n", " ")
    if len(text) > max_chars:
        text = text[:max_chars].rstrip() + "..."
    return text


def get_question_text(question):
    # "Who discovered penicillin?"
    return question.text.strip()


def get_options_text(question):
    # "Isaac Newton Alexander Fleming Marie Curie Charles Darwin"
    if hasattr(question, "options") and question.options is not None:
        return " ".join([opt.text for opt in question.options])
    return ""


def build_retrieval_query(question):
    # Build the query used for FAISS retrieval
    # question_only:
    # "Who discovered penicillin?"
    #
    # question_plus_options:
    # "Who discovered penicillin? Isaac Newton Alexander Fleming Marie Curie Charles Darwin"

    question_text = get_question_text(question)

    query_mode = BASE_RAG_CONFIG.get(
        "retrieval_query_mode",
        "question_plus_options"
    )

    if query_mode == "question_only":
        return question_text, False

    options_text = get_options_text(question)
    query = f"{question_text} {options_text}".strip()

    return query, True


def build_chat_prompt(system_msg, user_msg, tokenizer):
    # Convert system/user messages to model chat format
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

In [ ]:
# Prompt builders

def build_relevance_prompt(question, hit):
    # Build prompt for checking if one source is useful

    corpus_id = int(hit["corpus_id"])
    source_label = hit.get("source_label", "?")
    score = float(hit["score"])

    title = passage_metadata[corpus_id]["title"]
    paragraph_id = passage_metadata[corpus_id]["paragraph_id"]
    passage_text = shorten_text(passages[corpus_id], max_chars=900)
    options_text = format_options(question.options)

    system_msg = (
        "You are checking if a Wikipedia source is useful for answering a quiz question.\n"
        "Return only YES or NO.\n"
        "Return YES if the source is related to the question or to one answer option.\n"
        "Return NO only if the source is clearly unrelated."
    )

    user_msg = (
        f"Source {source_label}\n"
        f"Title: {title}\n"
        f"Paragraph: {paragraph_id}\n"
        f"Score: {score:.4f}\n\n"
        f"{passage_text}\n\n"
        f"Question:\n{question.text}\n\n"
        f"Options:\n{options_text}\n\n"
        "Is this source relevant?"
    )

    return system_msg, user_msg


def build_final_answer_prompt(question, context):
    # Use RAG prompt if context exists, otherwise fallback to base prompt

    if context.strip():
        system_msg, user_msg = build_rag_prompt(
            question.text,
            question.options,
            context
        )

        answer_parser = extract_option_id
        max_new_tokens = BASE_RAG_CONFIG.get("rag_max_new_tokens", 64)
        prompt_type = "rag"

    else:
        system_msg, user_msg = build_prompt(
            question.text,
            question.options
        )

        answer_parser = parse_answer
        max_new_tokens = CONFIG.get("max_new_tokens", 15)
        prompt_type = "llm_only"

        if CONFIG.get("prompt_strategy") in {"zero_shot_cot", "cot"}:
            max_new_tokens = CONFIG.get("reasoning_max_new_tokens", 256)

    return system_msg, user_msg, answer_parser, max_new_tokens, prompt_type

In [ ]:
# Retrieval

def retrieve_local_wikipedia_candidates(question, top_k=4):
    # Build query
    query, used_options = build_retrieval_query(question)

    # Encode query
    query_embedding = semb_model.encode(
        query,
        convert_to_tensor=False
    ).astype("float32")

    # FAISS expects 2D input
    query_embedding = np.expand_dims(query_embedding, axis=0)

    # Normalize for cosine-like inner product search
    faiss.normalize_L2(query_embedding)

    # Retrieve top-k passages
    scores, indices = faiss_index.search(query_embedding, top_k)

    hits = []

    for rank in range(top_k):
        corpus_id = int(indices[0][rank])
        score = float(scores[0][rank])
        source_label = chr(65 + rank)  # A, B, C, D...

        title = passage_metadata[corpus_id]["title"]
        paragraph_id = passage_metadata[corpus_id]["paragraph_id"]
        passage_text = passages[corpus_id]

        hits.append({
            "corpus_id": corpus_id,
            "score": score,
            "query": query,
            "used_options": used_options,
            "source_label": source_label,

            # Fields used by the advanced debugger
            "title": title,
            "paragraph_id": paragraph_id,
            "text": passage_text,
        })

    return hits

In [ ]:
# Relevance check

def check_source_relevance(question, hit, tokenizer, model, debug=False):
    # Ask the LLM if a retrieved source is relevant

    system_msg, user_msg = build_relevance_prompt(question, hit)

    prompt = build_chat_prompt(
        system_msg,
        user_msg,
        tokenizer
    )

    raw_output = generate_answer(
        prompt,
        tokenizer,
        model,
        do_sample=False,
        max_new_tokens=BASE_RAG_CONFIG.get("relevance_max_new_tokens", 16),
        temperature=1.0,
    )

    if debug:
        print("\n" + "=" * 80)
        print(f"RELEVANCE CHECK - SOURCE {hit.get('source_label')}")
        print("=" * 80)
        print(raw_output)

    answer = raw_output.strip().lower()

    if "yes" in answer:
        return True

    if "no" in answer:
        return False

    return None


def check_all_sources_relevance(question, hits, tokenizer, model, debug=False):
    # Check all retrieved sources

    relevant_labels = []
    parsing_failed = False

    for hit in hits:
        is_relevant = check_source_relevance(
            question,
            hit,
            tokenizer,
            model,
            debug=debug
        )

        if is_relevant is True:
            relevant_labels.append(hit.get("source_label"))

        if is_relevant is None:
            parsing_failed = True

    if parsing_failed:
        return None, True

    return relevant_labels, False

In [ ]:
# Context construction

def build_context_from_filtered_hits(
    hits,
    relevant_source_labels,
    parsing_failed=False,
    max_final_sources=3
):
    # If parsing failed, keep first sources as fallback
    if parsing_failed:
        selected_hits = hits[:max_final_sources]

    # If relevant sources exist, keep only those
    elif relevant_source_labels:
        selected_hits = [
            hit for hit in hits
            if hit.get("source_label") in relevant_source_labels
        ][:max_final_sources]

    # If no source is relevant, return empty context
    else:
        return ""

    context_blocks = []

    for hit in selected_hits:
        corpus_id = int(hit["corpus_id"])
        source_label = hit.get("source_label", "?")
        score = float(hit["score"])

        title = passage_metadata[corpus_id]["title"]
        paragraph_id = passage_metadata[corpus_id]["paragraph_id"]
        passage_text = passages[corpus_id]

        context_blocks.append(
            f"Context Source {source_label}\n"
            f"Title: {title} | Paragraph: {paragraph_id} | Retrieval score: {score:.4f}\n"
            f"{passage_text}"
        )

    return "\n\n".join(context_blocks)

In [ ]:
# Main answer function

def get_reliable_rag_answer_logged(
    question,
    tokenizer,
    model,
    max_retries=3,
    debug=False
):
    start_total = time.perf_counter()

    valid_ids = [opt.id for opt in question.options]

    # 1. Retrieve candidate sources
    retrieval_start = time.perf_counter()

    candidate_hits = retrieve_local_wikipedia_candidates(
        question,
        top_k=BASE_RAG_CONFIG.get("top_k_context", 4)
    )

    retrieval_time_sec = time.perf_counter() - retrieval_start

    # 2. Check source relevance
    relevance_start = time.perf_counter()

    if BASE_RAG_CONFIG.get("enable_llm_relevance_check", True):
        relevant_source_labels, relevance_parsing_failed = check_all_sources_relevance(
            question,
            candidate_hits,
            tokenizer,
            model,
            debug=False
        )
    else:
        relevant_source_labels = [
            hit.get("source_label")
            for hit in candidate_hits[:BASE_RAG_CONFIG.get("max_final_sources", 3)]
            if hit.get("source_label")
        ]
        relevance_parsing_failed = False

    relevance_time_sec = time.perf_counter() - relevance_start

    # 3. Build final context
    context_start = time.perf_counter()

    context = build_context_from_filtered_hits(
        candidate_hits,
        relevant_source_labels,
        parsing_failed=relevance_parsing_failed,
        max_final_sources=BASE_RAG_CONFIG.get("max_final_sources", 3)
    )

    context_time_sec = time.perf_counter() - context_start

    # 4. Build final answer prompt
    system_msg, user_msg, answer_parser, max_new_tokens, prompt_type = build_final_answer_prompt(
        question,
        context
    )

    prompt = build_chat_prompt(
        system_msg,
        user_msg,
        tokenizer
    )

    # 5. Generate answer with retries
    generation_start = time.perf_counter()

    final_answer = None
    attempts = []

    for attempt_idx in range(1, max_retries + 1):
        output = generate_answer(
            prompt,
            tokenizer,
            model,
            do_sample=CONFIG.get("do_sample", False),
            max_new_tokens=max_new_tokens,
            temperature=CONFIG.get("temperature", 1.0),
        )

        parsed_answer = answer_parser(output, valid_ids)

        attempts.append({
            "attempt": attempt_idx,
            "raw_output": output,
            "parsed_answer": parsed_answer,
        })

        if parsed_answer is not None:
            final_answer = parsed_answer
            break

    generation_time_sec = time.perf_counter() - generation_start

    # 6. Fallback if parsing failed
    fallback_used = False
    fallback_reason = None
    parsing_failed = False

    if final_answer is None:
        parsing_failed = True
        fallback_used = True
        fallback_reason = "default_first_valid_option"
        final_answer = valid_ids[0]

    # 7. Keep only selected hits for debug
    if relevance_parsing_failed:
        selected_hits = candidate_hits[:BASE_RAG_CONFIG.get("max_final_sources", 3)]
    elif relevant_source_labels:
        selected_hits = [
            hit for hit in candidate_hits
            if hit.get("source_label") in relevant_source_labels
        ][:BASE_RAG_CONFIG.get("max_final_sources", 3)]
    else:
        selected_hits = []

    # 8. Metadata compatible with the advanced debugger
    details = {
        "active_rag_variant": "reliable_rag",
        "rag_type": "reliable_local_rag",

        "prompt_mode": prompt_type,
        "context_status": "context_used" if context.strip() else "llm_only_fallback",
        "selection_mode": "llm_relevance_filtering",

        "query": candidate_hits[0]["query"] if candidate_hits else None,
        "used_options": candidate_hits[0]["used_options"] if candidate_hits else None,

        "hits": candidate_hits,
        "candidate_hits": candidate_hits,
        "selected_hits": selected_hits,

        "context": context,
        "context_used": bool(context.strip()),

        "relevant_source_labels": relevant_source_labels,
        "relevance_parsing_failed": relevance_parsing_failed,

        "attempts": attempts,
        "raw_output": attempts[-1]["raw_output"] if attempts else "",

        "answer_id": final_answer,
        "parsing_failed": parsing_failed,
        "fallback_used": fallback_used,
        "fallback_reason": fallback_reason,

        "retrieval_time_sec": retrieval_time_sec,
        "relevance_time_sec": relevance_time_sec,
        "context_time_sec": context_time_sec,
        "generation_time_sec": generation_time_sec,
        "total_answer_time_sec": time.perf_counter() - start_total,
    }

    return final_answer, details

In [ ]:
# Shared RAG dispatcher

def get_rag_answer(question, tokenizer, model, max_retries=3):
    return get_reliable_rag_answer_logged(
        question,
        tokenizer,
        model,
        max_retries=max_retries
    )

In [ ]:
# Reliable-RAG activation

def activate_reliable_rag_variant(
    top_k_context=4,
    retrieval_query_mode="question_plus_options",
    enable_llm_relevance_check=True,
    max_final_sources=3,
):
    global retrieve_local_wikipedia_context
    global get_rag_answer_logged

    retrieve_local_wikipedia_context = retrieve_local_wikipedia_candidates
    get_rag_answer_logged = get_reliable_rag_answer_logged

    BASE_RAG_CONFIG.update({
        "top_k_context": top_k_context,
        "max_final_sources": max_final_sources,
        "relevance_max_new_tokens": 16,
        "enable_llm_relevance_check": enable_llm_relevance_check,
        "retrieval_query_mode": retrieval_query_mode,
    })

    print("Reliable local RAG variant activated.")
    print(f"top_k_context: {top_k_context}")
    print(f"max_final_sources: {max_final_sources}")
    print(f"retrieval_query_mode: {retrieval_query_mode}")
    print(f"enable_llm_relevance_check: {enable_llm_relevance_check}")

# 🔵 Query Enhancement

## 🟢 Query Transformation RAG


In [ ]:
import json
import re
import time
import numpy as np

In [ ]:
# Helpers

def qt_clean_text(value):
    # Clean one generated search query
    text = str(value or "").strip()

    text = re.sub(r"^\s*[-*\d.]+\s*", "", text).strip()

    text = re.sub(
        r"^(rewritten query|step-back query|step back query|sub-query|subquery)\s*:\s*",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()

    return text


def qt_normalize_query_key(query):
    # Used for deduplication
    return re.sub(r"\s+", " ", str(query or "").lower()).strip()


def qt_get_question_text(question):
    # "Who discovered penicillin?"
    return question.text.strip()


def qt_get_options_text(question):
    # "Isaac Newton Alexander Fleming Marie Curie Charles Darwin"
    if hasattr(question, "options") and question.options is not None:
        return " ".join([opt.text for opt in question.options])
    return ""


def qt_build_fallback_query(question):
    # Build original/fallback query
    # question_only:
    # "Who discovered penicillin?"
    #
    # question_plus_options:
    # "Who discovered penicillin? Isaac Newton Alexander Fleming Marie Curie Charles Darwin"

    question_text = qt_get_question_text(question)

    original_mode = BASE_RAG_CONFIG.get(
        "query_transform_original_mode",
        "question_only"
    )

    if original_mode == "question_only":
        return question_text

    options_text = qt_get_options_text(question)
    return f"{question_text} {options_text}".strip()


def qt_build_chat_prompt(system_msg, user_msg, tokenizer):
    # Convert system/user messages to model chat format
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


def qt_get_selected_hits(candidate_hits, relevant_source_labels, relevance_parsing_failed):
    # Keep the same source selection logic used for context construction

    max_final_sources = BASE_RAG_CONFIG.get("max_final_sources", 3)

    if relevance_parsing_failed:
        return candidate_hits[:max_final_sources]

    if relevant_source_labels:
        return [
            hit for hit in candidate_hits
            if hit.get("source_label") in relevant_source_labels
        ][:max_final_sources]

    return []

In [ ]:
# Prompt builders

def qt_build_query_transformation_prompt(question):
    # Ask the LLM to generate search queries, not answers

    options_text = format_options(question.options)

    system_msg = (
        "You generate search queries for a local Wikipedia retrieval system.\n"
        "Given one multiple-choice quiz question, create query transformations "
        "that help retrieve useful evidence from Wikipedia.\n\n"
        "Return only valid JSON with this exact schema:\n"
        "{\n"
        '  "rewritten_query": "a specific and detailed retrieval query",\n'
        '  "step_back_query": "a broader background query",\n'
        '  "sub_queries": ["simple query 1", "simple query 2", "simple query 3"]\n'
        "}\n\n"
        "Rules:\n"
        "- Use the question and the answer options to understand the topic.\n"
        "- Do not answer the quiz question.\n"
        "- Do not include option IDs.\n"
        "- Keep each query short and useful for semantic search.\n"
        "- Return at most 3 sub_queries.\n"
        "- Do not write anything outside the JSON object."
    )

    user_msg = (
        f"Question:\n{question.text.strip()}\n\n"
        f"Answer options:\n{options_text}\n\n"
        "Query transformation JSON:"
    )

    return system_msg, user_msg


def qt_build_final_answer_prompt(question, context):
    # Use RAG prompt if context exists, otherwise fallback to base LLM prompt

    if context.strip():
        system_msg, user_msg = build_rag_prompt(
            question.text,
            question.options,
            context
        )

        answer_parser = extract_option_id
        max_new_tokens = BASE_RAG_CONFIG.get("rag_max_new_tokens", 64)
        prompt_type = "rag"

    else:
        system_msg, user_msg = build_prompt(
            question.text,
            question.options
        )

        answer_parser = parse_answer
        max_new_tokens = CONFIG.get("max_new_tokens", 15)
        prompt_type = "llm_only"

        if CONFIG.get("prompt_strategy") in {"zero_shot_cot", "cot"}:
            max_new_tokens = CONFIG.get("reasoning_max_new_tokens", 256)

    return system_msg, user_msg, answer_parser, max_new_tokens, prompt_type


In [ ]:
# Output parsing

def qt_parse_query_transformation_output(raw_output, max_subqueries=3):
    # Parse JSON produced by the query transformation LLM

    raw = str(raw_output or "").strip()

    raw = re.sub(r"^```(?:json)?", "", raw, flags=re.IGNORECASE).strip()
    raw = re.sub(r"```$", "", raw).strip()

    match = re.search(r"\{.*\}", raw, flags=re.DOTALL)

    if not match:
        return None

    try:
        data = json.loads(match.group())
    except Exception:
        return None

    rewritten_query = qt_clean_text(data.get("rewritten_query", ""))
    step_back_query = qt_clean_text(data.get("step_back_query", ""))

    sub_queries_raw = data.get("sub_queries", [])

    if isinstance(sub_queries_raw, str):
        sub_queries_raw = [
            line.strip()
            for line in re.split(r"\n|;", sub_queries_raw)
            if line.strip()
        ]

    if not isinstance(sub_queries_raw, list):
        sub_queries_raw = []

    sub_queries = []

    for item in sub_queries_raw:
        cleaned = qt_clean_text(item)

        if cleaned and cleaned not in sub_queries:
            sub_queries.append(cleaned)

        if len(sub_queries) >= max_subqueries:
            break

    if not rewritten_query and not step_back_query and not sub_queries:
        return None

    return {
        "rewritten_query": rewritten_query,
        "step_back_query": step_back_query,
        "sub_queries": sub_queries,
    }

In [ ]:
# Query generation

def qt_generate_query_transformations(question, tokenizer, model):
    # Generate rewritten, step-back and sub-query variants

    fallback_query = qt_build_fallback_query(question)

    if tokenizer is None or model is None:
        return {
            "rewritten_query": "",
            "step_back_query": "",
            "sub_queries": [],
            "fallback_query": fallback_query,
            "raw_output": "",
        }, True

    system_msg, user_msg = qt_build_query_transformation_prompt(question)

    prompt = qt_build_chat_prompt(
        system_msg,
        user_msg,
        tokenizer
    )

    raw_output = generate_answer(
        prompt,
        tokenizer,
        model,
        do_sample=False,
        max_new_tokens=BASE_RAG_CONFIG.get("query_transform_max_new_tokens", 160),
        temperature=1.0,
    )

    transformations = qt_parse_query_transformation_output(
        raw_output,
        max_subqueries=BASE_RAG_CONFIG.get("query_transform_max_subqueries", 3),
    )

    if transformations is None:
        transformations = {
            "rewritten_query": "",
            "step_back_query": "",
            "sub_queries": [],
        }
        fallback_used = True
    else:
        fallback_used = False

    transformations["fallback_query"] = fallback_query
    transformations["raw_output"] = raw_output

    return transformations, fallback_used


def qt_build_search_queries(question, transformations, fallback_used=False):
    # Build final list of queries to search in FAISS

    fallback_query = transformations.get("fallback_query")

    if not fallback_query:
        fallback_query = qt_build_fallback_query(question)

    query_items = []

    # Add original/fallback query
    if BASE_RAG_CONFIG.get("query_transform_include_original", True):
        original_mode = BASE_RAG_CONFIG.get(
            "query_transform_original_mode",
            "question_only"
        )

        query_items.append({
            "query": fallback_query,
            "query_type": f"original_{original_mode}",
        })

    # Add LLM-generated transformed queries
    if not fallback_used:
        rewritten_query = transformations.get("rewritten_query")
        step_back_query = transformations.get("step_back_query")
        sub_queries = transformations.get("sub_queries", [])

        if rewritten_query:
            query_items.append({
                "query": rewritten_query,
                "query_type": "rewritten_query",
            })

        if step_back_query:
            query_items.append({
                "query": step_back_query,
                "query_type": "step_back_query",
            })

        for idx, sub_query in enumerate(sub_queries, start=1):
            query_items.append({
                "query": sub_query,
                "query_type": f"sub_query_{idx}",
            })

    # Safety fallback
    if not query_items and BASE_RAG_CONFIG.get("query_transform_fallback_to_question_options", True):
        original_mode = BASE_RAG_CONFIG.get(
            "query_transform_original_mode",
            "question_only"
        )

        query_items.append({
            "query": fallback_query,
            "query_type": f"fallback_{original_mode}",
        })

    # Deduplicate queries
    deduped = []
    seen = set()

    for item in query_items:
        query = qt_clean_text(item.get("query", ""))
        key = qt_normalize_query_key(query)

        if not query or key in seen:
            continue

        seen.add(key)

        deduped.append({
            "query": query,
            "query_type": item["query_type"],
        })

    return deduped

In [ ]:
# Multi-query FAISS retrieval

def retrieve_local_wikipedia_context_query_transform(
    question,
    tokenizer=None,
    model=None,
    top_k=None
):
    # Backward compatibility:
    # retrieve_local_wikipedia_context_query_transform(question, 6)
    if isinstance(tokenizer, int) and model is None and top_k is None:
        top_k = tokenizer
        tokenizer = None

    if tokenizer is None and "llm_tokenizer" in globals():
        tokenizer = llm_tokenizer

    if model is None and "llm_model" in globals():
        model = llm_model

    if top_k is None:
        top_k = BASE_RAG_CONFIG.get("query_transform_max_candidate_hits", 6)

    # 1. Generate transformed queries
    transformations, transform_fallback_used = qt_generate_query_transformations(
        question,
        tokenizer,
        model
    )

    # 2. Build final search query list
    query_items = qt_build_search_queries(
        question,
        transformations,
        fallback_used=transform_fallback_used,
    )

    if not query_items:
        fallback_query = qt_build_fallback_query(question)
        original_mode = BASE_RAG_CONFIG.get(
            "query_transform_original_mode",
            "question_only"
        )

        query_items = [{
            "query": fallback_query,
            "query_type": f"fallback_{original_mode}",
        }]

    # 3. Encode all queries
    queries = [item["query"] for item in query_items]

    top_k_per_query = BASE_RAG_CONFIG.get(
        "query_transform_top_k_per_query",
        2
    )

    query_embeddings = semb_model.encode(
        queries,
        convert_to_tensor=False
    ).astype("float32")

    if query_embeddings.ndim == 1:
        query_embeddings = np.expand_dims(query_embeddings, axis=0)

    faiss.normalize_L2(query_embeddings)

    # 4. Search FAISS for each query
    scores, indices = faiss_index.search(
        query_embeddings,
        top_k_per_query
    )

    # 5. Deduplicate hits, keeping best score per passage
    best_hits_by_corpus_id = {}

    for query_idx, query_item in enumerate(query_items):
        for query_rank, (corpus_id, score) in enumerate(
            zip(indices[query_idx], scores[query_idx]),
            start=1
        ):
            corpus_id = int(corpus_id)
            score = float(score)

            if corpus_id < 0:
                continue

            old_hit = best_hits_by_corpus_id.get(corpus_id)

            if old_hit is not None and old_hit["score"] >= score:
                continue

            best_hits_by_corpus_id[corpus_id] = {
                "corpus_id": corpus_id,
                "score": score,
                "query": query_item["query"],
                "query_type": query_item["query_type"],
                "_query_index": query_idx,
                "_query_rank": query_rank,
            }

    candidate_hits = sorted(
        best_hits_by_corpus_id.values(),
        key=lambda h: (-h["score"], h["_query_index"], h["_query_rank"])
    )[:top_k]

    # 6. Build context and debugger-friendly hits
    context_blocks = []
    hits = []

    for rank, hit in enumerate(candidate_hits, start=1):
        corpus_id = int(hit["corpus_id"])
        source_label = chr(64 + rank)  # A, B, C...

        title = passage_metadata[corpus_id]["title"]
        paragraph_id = passage_metadata[corpus_id]["paragraph_id"]
        passage_text = passages[corpus_id]

        clean_hit = {
            "corpus_id": corpus_id,
            "score": float(hit["score"]),
            "query": hit["query"],
            "query_type": hit["query_type"],
            "source_label": source_label,

            # Fields for the advanced debugger
            "title": title,
            "paragraph_id": paragraph_id,
            "text": passage_text,

            # Internal ranking info
            "_query_index": hit["_query_index"],
            "_query_rank": hit["_query_rank"],
        }

        hits.append(clean_hit)

        context_blocks.append(
            f"Context Source {source_label}\n"
            f"Title: {title} | Paragraph: {paragraph_id} | Retrieval score: {hit['score']:.4f}\n"
            f"Matched query type: {hit['query_type']}\n"
            f"Matched query: {hit['query']}\n"
            f"{passage_text}"
        )

    details = {
        "query_transformations": transformations,
        "query_transform_raw_output": transformations.get("raw_output", ""),
        "query_transform_fallback_used": transform_fallback_used,
        "query_items": query_items,
    }

    return "\n\n".join(context_blocks), hits, details

In [ ]:
# Main answer function

def get_query_transform_rag_answer_logged(
    question,
    tokenizer,
    model,
    max_retries=3
):
    start_total = time.perf_counter()

    valid_ids = [opt.id for opt in question.options]

    # 1. Multi-query retrieval
    retrieval_start = time.perf_counter()

    _, candidate_hits, retrieval_details = retrieve_local_wikipedia_context_query_transform(
        question,
        tokenizer,
        model,
        top_k=BASE_RAG_CONFIG.get("query_transform_max_candidate_hits", 6)
    )

    retrieval_time_sec = time.perf_counter() - retrieval_start

    # 2. LLM relevance filtering
    relevance_start = time.perf_counter()

    if BASE_RAG_CONFIG.get("enable_llm_relevance_check", True):
        relevant_source_labels, relevance_parsing_failed = check_all_sources_relevance(
            question,
            candidate_hits,
            tokenizer,
            model
        )
    else:
        relevant_source_labels = [
            hit.get("source_label")
            for hit in candidate_hits[:BASE_RAG_CONFIG.get("max_final_sources", 3)]
            if hit.get("source_label")
        ]
        relevance_parsing_failed = False

    relevance_time_sec = time.perf_counter() - relevance_start

    # 3. Build final context from selected hits
    context_start = time.perf_counter()

    context = build_context_from_filtered_hits(
        candidate_hits,
        relevant_source_labels,
        parsing_failed=relevance_parsing_failed,
        max_final_sources=BASE_RAG_CONFIG.get("max_final_sources", 3)
    )

    context_time_sec = time.perf_counter() - context_start

    selected_hits = qt_get_selected_hits(
        candidate_hits,
        relevant_source_labels,
        relevance_parsing_failed
    )

    # 4. Build answer prompt
    system_msg, user_msg, answer_parser, max_new_tokens, prompt_type = qt_build_final_answer_prompt(
        question,
        context
    )

    prompt = qt_build_chat_prompt(
        system_msg,
        user_msg,
        tokenizer
    )

    # 5. Generate answer with retries
    generation_start = time.perf_counter()

    final_answer = None
    attempts = []

    for attempt_idx in range(1, max_retries + 1):
        output = generate_answer(
            prompt,
            tokenizer,
            model,
            do_sample=CONFIG.get("do_sample", False),
            max_new_tokens=max_new_tokens,
            temperature=CONFIG.get("temperature", 1.0),
        )

        parsed_answer = answer_parser(output, valid_ids)

        attempts.append({
            "attempt": attempt_idx,
            "raw_output": output,
            "parsed_answer": parsed_answer,
        })

        if parsed_answer is not None:
            final_answer = parsed_answer
            break

    generation_time_sec = time.perf_counter() - generation_start

    # 6. Fallback if answer parsing failed
    parsing_failed = False
    fallback_used = False
    fallback_reason = None

    if final_answer is None:
        parsing_failed = True
        fallback_used = True
        fallback_reason = "default_first_valid_option"
        final_answer = valid_ids[0]

    # 7. Metadata compatible with Advanced RAG Debugger
    details = {
        "active_rag_variant": "query_transform_rag",
        "rag_type": "query_transform_local_rag",

        "prompt_mode": prompt_type,
        "context_status": "context_used" if context.strip() else "llm_only_fallback",
        "selection_mode": "query_transformation_multi_query_retrieval",

        "query_transformations": retrieval_details.get("query_transformations"),
        "query_transform_raw_output": retrieval_details.get("query_transform_raw_output"),
        "query_transform_fallback_used": retrieval_details.get("query_transform_fallback_used"),
        "query_items": retrieval_details.get("query_items"),

        "hits": candidate_hits,
        "candidate_hits": candidate_hits,
        "selected_hits": selected_hits,

        "context": context,
        "context_used": bool(context.strip()),

        "relevant_source_labels": relevant_source_labels,
        "relevance_parsing_failed": relevance_parsing_failed,

        "attempts": attempts,
        "raw_output": attempts[-1]["raw_output"] if attempts else "",

        "answer_id": final_answer,
        "parsing_failed": parsing_failed,
        "fallback_used": fallback_used,
        "fallback_reason": fallback_reason,

        "retrieval_time_sec": retrieval_time_sec,
        "relevance_time_sec": relevance_time_sec,
        "context_time_sec": context_time_sec,
        "generation_time_sec": generation_time_sec,
        "total_answer_time_sec": time.perf_counter() - start_total,
    }

    return final_answer, details

In [ ]:
# Shared dispatcher

def get_query_transform_rag_answer(question, tokenizer, model, max_retries=3):
    answer_id, _ = get_query_transform_rag_answer_logged(
        question,
        tokenizer,
        model,
        max_retries=max_retries,
    )
    return answer_id

In [ ]:
# Activation

def activate_query_transformation_rag_variant(
    top_k_context=6,
    top_k_per_query=2,
    max_subqueries=3,
    max_final_sources=3,
    query_transform_original_mode="question_only",
    enable_llm_relevance_check=True,
):
    global retrieve_local_wikipedia_context
    global get_rag_answer_logged

    retrieve_local_wikipedia_context = retrieve_local_wikipedia_context_query_transform
    get_rag_answer_logged = get_query_transform_rag_answer_logged

    BASE_RAG_CONFIG.update({
        "top_k_context": top_k_context,

        "query_transform_max_new_tokens": 160,
        "query_transform_max_subqueries": max_subqueries,
        "query_transform_top_k_per_query": top_k_per_query,
        "query_transform_max_candidate_hits": top_k_context,
        "query_transform_include_original": True,
        "query_transform_fallback_to_question_options": True,
        "query_transform_original_mode": query_transform_original_mode,

        "max_final_sources": max_final_sources,
        "enable_llm_relevance_check": enable_llm_relevance_check,
        "relevance_max_new_tokens": 16,
        "rag_max_new_tokens": 64,
    })

    print("Query Transformation RAG variant activated.")
    print(f"top_k_context: {top_k_context}")
    print(f"top_k_per_query: {top_k_per_query}")
    print(f"max_subqueries: {max_subqueries}")
    print(f"max_final_sources: {max_final_sources}")
    print(f"query_transform_original_mode: {query_transform_original_mode}")
    print(f"enable_llm_relevance_check: {enable_llm_relevance_check}")

## 🟢 HyDE

In [ ]:
import re
import time
import numpy as np

In [ ]:
# Helpers

def hyde_clean_document(raw_output):
    # Clean the generated hypothetical document
    text = str(raw_output or "").strip()

    text = re.sub(r"^```(?:text)?", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"```$", "", text).strip()

    text = re.sub(
        r"^(hypothetical\s+(?:wikipedia\s+)?(?:document|passage)|passage|document)\s*:\s*",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()

    return text


def hyde_get_question_text(question):
    # "Who discovered penicillin?"
    return question.text.strip()


def hyde_get_options_text(question):
    # "Isaac Newton Alexander Fleming Marie Curie Charles Darwin"
    if hasattr(question, "options") and question.options is not None:
        return " ".join([opt.text for opt in question.options])
    return ""


def hyde_build_fallback_query(question):
    # Build fallback query if HyDE generation fails
    # question_only:
    # "Who discovered penicillin?"
    #
    # question_plus_options:
    # "Who discovered penicillin? Isaac Newton Alexander Fleming Marie Curie Charles Darwin"

    question_text = hyde_get_question_text(question)

    fallback_mode = BASE_RAG_CONFIG.get(
        "hyde_fallback_query_mode",
        "question_only"
    )

    if fallback_mode == "question_only":
        return question_text

    options_text = hyde_get_options_text(question)
    return f"{question_text} {options_text}".strip()


def hyde_build_chat_prompt(system_msg, user_msg, tokenizer):
    # Convert system/user messages to model chat format
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


def hyde_get_selected_hits(candidate_hits, relevant_source_labels, relevance_parsing_failed):
    # Keep the same source selection logic used for context construction

    max_final_sources = BASE_RAG_CONFIG.get("max_final_sources", 3)

    if relevance_parsing_failed:
        return candidate_hits[:max_final_sources]

    if relevant_source_labels:
        return [
            hit for hit in candidate_hits
            if hit.get("source_label") in relevant_source_labels
        ][:max_final_sources]

    return []

# Backward compatibility with older HyDE code
def build_hyde_fallback_query(question):
    return hyde_build_fallback_query(question)


def build_hyde_generation_prompt(question):
    return hyde_build_generation_prompt(question)


def clean_hyde_document(raw_output):
    return hyde_clean_document(raw_output)

In [ ]:
# Prompt builders

def hyde_build_generation_prompt(question):
    # Build prompt that asks the LLM to generate a hypothetical Wikipedia passage

    include_options = BASE_RAG_CONFIG.get("hyde_include_options", True)

    options_text = ""
    if include_options and hasattr(question, "options") and question.options is not None:
        options_text = format_options(question.options)

    system_msg = (
        "You generate search passages for a local Wikipedia retrieval system.\n"
        "Given a multiple-choice quiz question, write a short hypothetical "
        "Wikipedia-style passage that would likely contain the information needed "
        "to answer it.\n\n"
        "Rules:\n"
        "- Return only the hypothetical passage.\n"
        "- Do not return an option ID.\n"
        "- Do not explain your reasoning.\n"
        "- Keep it factual in style, concise, and useful for semantic search.\n"
        "- Prefer key entities, dates, places, concepts, and relationships."
    )

    user_msg = f"Question:\n{question.text.strip()}"

    if options_text:
        user_msg += f"\n\nAnswer options:\n{options_text}"

    user_msg += "\n\nHypothetical Wikipedia-style passage:"

    return system_msg, user_msg


def hyde_build_final_answer_prompt(question, context):
    # Use RAG prompt if context exists, otherwise fallback to base LLM prompt

    if context.strip():
        system_msg, user_msg = build_rag_prompt(
            question.text,
            question.options,
            context
        )

        answer_parser = extract_option_id
        max_new_tokens = BASE_RAG_CONFIG.get("rag_max_new_tokens", 64)
        prompt_type = "rag"

    else:
        system_msg, user_msg = build_prompt(
            question.text,
            question.options
        )

        answer_parser = parse_answer
        max_new_tokens = CONFIG.get("max_new_tokens", 15)
        prompt_type = "llm_only"

        if CONFIG.get("prompt_strategy") in {"zero_shot_cot", "cot"}:
            max_new_tokens = CONFIG.get("reasoning_max_new_tokens", 256)

    return system_msg, user_msg, answer_parser, max_new_tokens, prompt_type

In [ ]:
# HyDE document generation

def hyde_generate_hypothetical_document(question, tokenizer, model):
    # Generate the HyDE document used as semantic retrieval query

    fallback_query = hyde_build_fallback_query(question)

    if tokenizer is None or model is None:
        return {
            "hypothetical_document": fallback_query,
            "raw_output": "",
            "fallback_query": fallback_query,
            "fallback_used": True,
            "query_type": "hyde_fallback_query",
        }

    system_msg, user_msg = hyde_build_generation_prompt(question)

    prompt = hyde_build_chat_prompt(
        system_msg,
        user_msg,
        tokenizer
    )

    raw_output = generate_answer(
        prompt,
        tokenizer,
        model,
        do_sample=False,
        max_new_tokens=BASE_RAG_CONFIG.get("hyde_max_new_tokens", 96),
        temperature=1.0,
    )

    hypothetical_document = hyde_clean_document(raw_output)

    if len(hypothetical_document) < 20:
        return {
            "hypothetical_document": fallback_query,
            "raw_output": raw_output,
            "fallback_query": fallback_query,
            "fallback_used": True,
            "query_type": "hyde_fallback_query",
        }

    return {
        "hypothetical_document": hypothetical_document,
        "raw_output": raw_output,
        "fallback_query": fallback_query,
        "fallback_used": False,
        "query_type": "hyde_hypothetical_document",
    }


In [ ]:
# HyDE retrieval

def retrieve_local_wikipedia_context_hyde(
    question,
    tokenizer=None,
    model=None,
    top_k=None
):
    # Backward compatibility:
    # retrieve_local_wikipedia_context_hyde(question, 4)
    if isinstance(tokenizer, int) and model is None and top_k is None:
        top_k = tokenizer
        tokenizer = None

    if tokenizer is None and "llm_tokenizer" in globals():
        tokenizer = llm_tokenizer

    if model is None and "llm_model" in globals():
        model = llm_model

    if top_k is None:
        top_k = BASE_RAG_CONFIG.get("top_k_context", 4)

    # 1. Generate HyDE document
    hyde_details = hyde_generate_hypothetical_document(
        question,
        tokenizer,
        model
    )

    hypothetical_document = hyde_details["hypothetical_document"]
    query_type = hyde_details["query_type"]

    # 2. Embed HyDE document
    query_embedding = semb_model.encode(
        hypothetical_document,
        convert_to_tensor=False
    ).astype("float32")

    query_embedding = np.expand_dims(query_embedding, axis=0)

    # Normalize for cosine-like inner product search
    faiss.normalize_L2(query_embedding)

    # 3. Search FAISS
    scores, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    # 4. Build context and debugger-friendly hits
    context_blocks = []
    hits = []

    for rank, (corpus_id, score) in enumerate(zip(indices[0], scores[0]), start=1):
        corpus_id = int(corpus_id)
        score = float(score)

        if corpus_id < 0:
            continue

        source_label = chr(64 + rank)  # A, B, C...

        title = passage_metadata[corpus_id]["title"]
        paragraph_id = passage_metadata[corpus_id]["paragraph_id"]
        passage_text = passages[corpus_id]

        hits.append({
            "corpus_id": corpus_id,
            "score": score,
            "query": hypothetical_document,
            "query_type": query_type,
            "source_label": source_label,

            # Fields for the advanced debugger
            "title": title,
            "paragraph_id": paragraph_id,
            "text": passage_text,
        })

        context_blocks.append(
            f"Context Source {source_label}\n"
            f"Title: {title} | Paragraph: {paragraph_id} | Retrieval score: {score:.4f}\n"
            f"Matched query type: {query_type}\n"
            f"Matched query:\n{hypothetical_document}\n"
            f"{passage_text}"
        )

    retrieval_details = {
        "hyde_hypothetical_document": hypothetical_document,
        "hyde_raw_output": hyde_details.get("raw_output", ""),
        "hyde_fallback_query": hyde_details.get("fallback_query", ""),
        "hyde_fallback_used": hyde_details.get("fallback_used", False),
        "hyde_query_type": query_type,

        # Compatible with generic query printer
        "query": hypothetical_document,
        "query_type": query_type,
        "query_items": [
            {
                "query": hypothetical_document,
                "query_type": query_type,
            }
        ],
    }

    return "\n\n".join(context_blocks), hits, retrieval_details

In [ ]:
# Main answer function

def get_hyde_rag_answer_logged(
    question,
    tokenizer,
    model,
    max_retries=3
):
    start_total = time.perf_counter()

    valid_ids = [opt.id for opt in question.options]

    # 1. Retrieve candidate sources with HyDE
    retrieval_start = time.perf_counter()

    _, candidate_hits, retrieval_details = retrieve_local_wikipedia_context_hyde(
        question,
        tokenizer,
        model,
        top_k=BASE_RAG_CONFIG.get("top_k_context", 4)
    )

    retrieval_time_sec = time.perf_counter() - retrieval_start

    # 2. Source relevance filtering
    relevance_start = time.perf_counter()

    if BASE_RAG_CONFIG.get("enable_llm_relevance_check", True):
        relevant_source_labels, relevance_parsing_failed = check_all_sources_relevance(
            question,
            candidate_hits,
            tokenizer,
            model
        )
    else:
        relevant_source_labels = [
            hit.get("source_label")
            for hit in candidate_hits[:BASE_RAG_CONFIG.get("max_final_sources", 3)]
            if hit.get("source_label")
        ]
        relevance_parsing_failed = False

    relevance_time_sec = time.perf_counter() - relevance_start

    # 3. Build final context
    context_start = time.perf_counter()

    context = build_context_from_filtered_hits(
        candidate_hits,
        relevant_source_labels,
        parsing_failed=relevance_parsing_failed,
        max_final_sources=BASE_RAG_CONFIG.get("max_final_sources", 3)
    )

    context_time_sec = time.perf_counter() - context_start

    selected_hits = hyde_get_selected_hits(
        candidate_hits,
        relevant_source_labels,
        relevance_parsing_failed
    )

    # 4. Build answer prompt
    system_msg, user_msg, answer_parser, max_new_tokens, prompt_type = hyde_build_final_answer_prompt(
        question,
        context
    )

    prompt = hyde_build_chat_prompt(
        system_msg,
        user_msg,
        tokenizer
    )

    # 5. Generate final answer with retries
    generation_start = time.perf_counter()

    final_answer = None
    attempts = []

    for attempt_idx in range(1, max_retries + 1):
        output = generate_answer(
            prompt,
            tokenizer,
            model,
            do_sample=CONFIG.get("do_sample", False),
            max_new_tokens=max_new_tokens,
            temperature=CONFIG.get("temperature", 1.0),
        )

        parsed_answer = answer_parser(output, valid_ids)

        attempts.append({
            "attempt": attempt_idx,
            "raw_output": output,
            "parsed_answer": parsed_answer,
        })

        if parsed_answer is not None:
            final_answer = parsed_answer
            break

    generation_time_sec = time.perf_counter() - generation_start

    # 6. Fallback if parsing failed
    parsing_failed = False
    fallback_used = False
    fallback_reason = None

    if final_answer is None:
        parsing_failed = True
        fallback_used = True
        fallback_reason = "default_first_valid_option"
        final_answer = valid_ids[0]

    # 7. Metadata compatible with Advanced RAG Debugger
    details = {
        "active_rag_variant": "hyde_rag",
        "rag_type": "hyde_local_rag",

        "prompt_mode": prompt_type,
        "context_status": "context_used" if context.strip() else "llm_only_fallback",
        "selection_mode": "hyde_hypothetical_document_retrieval",

        "hyde_hypothetical_document": retrieval_details.get("hyde_hypothetical_document"),
        "hyde_raw_output": retrieval_details.get("hyde_raw_output"),
        "hyde_fallback_query": retrieval_details.get("hyde_fallback_query"),
        "hyde_fallback_used": retrieval_details.get("hyde_fallback_used"),
        "hyde_query_type": retrieval_details.get("hyde_query_type"),

        "query": retrieval_details.get("query"),
        "query_type": retrieval_details.get("query_type"),
        "query_items": retrieval_details.get("query_items"),

        "hits": candidate_hits,
        "candidate_hits": candidate_hits,
        "selected_hits": selected_hits,

        "context": context,
        "context_used": bool(context.strip()),

        "relevant_source_labels": relevant_source_labels,
        "relevance_parsing_failed": relevance_parsing_failed,

        "attempts": attempts,
        "raw_output": attempts[-1]["raw_output"] if attempts else "",

        "answer_id": final_answer,
        "parsing_failed": parsing_failed,
        "fallback_used": fallback_used,
        "fallback_reason": fallback_reason,

        "retrieval_time_sec": retrieval_time_sec,
        "relevance_time_sec": relevance_time_sec,
        "context_time_sec": context_time_sec,
        "generation_time_sec": generation_time_sec,
        "total_answer_time_sec": time.perf_counter() - start_total,
    }

    return final_answer, details

In [ ]:
# ============================================================
# Shared dispatcher
# ============================================================

def get_hyde_rag_answer(question, tokenizer, model, max_retries=3):
    answer_id, _ = get_hyde_rag_answer_logged(
        question,
        tokenizer,
        model,
        max_retries=max_retries,
    )
    return answer_id

In [ ]:
# ============================================================
# Activation
# ============================================================

def activate_hyde_rag_variant(
    top_k_context=4,
    max_final_sources=3,
    hyde_include_options=True,
    hyde_fallback_query_mode="question_only",
    enable_llm_relevance_check=True,
):
    global retrieve_local_wikipedia_context
    global get_rag_answer_logged

    retrieve_local_wikipedia_context = retrieve_local_wikipedia_context_hyde
    get_rag_answer_logged = get_hyde_rag_answer_logged

    BASE_RAG_CONFIG.update({
        "top_k_context": top_k_context,

        "hyde_max_new_tokens": 96,
        "hyde_include_options": hyde_include_options,
        "hyde_fallback_query_mode": hyde_fallback_query_mode,

        "max_final_sources": max_final_sources,
        "enable_llm_relevance_check": enable_llm_relevance_check,
        "relevance_max_new_tokens": 16,
        "rag_max_new_tokens": 64,
    })

    print("HyDE-RAG variant activated.")
    print(f"top_k_context: {top_k_context}")
    print(f"max_final_sources: {max_final_sources}")
    print(f"hyde_include_options: {hyde_include_options}")
    print(f"hyde_fallback_query_mode: {hyde_fallback_query_mode}")
    print(f"enable_llm_relevance_check: {enable_llm_relevance_check}")

# 🔵 Context Enrichment

## 🟢 Relevant Segment Extraction (RSE)



In [ ]:
import time

In [ ]:
# Paragraph lookup

def build_rse_paragraph_lookup():
    # Build lookup:
    # ("Sparta", 4) -> corpus_id
    lookup = {}

    for corpus_id, meta in enumerate(passage_metadata):
        title = meta["title"]
        paragraph_id = int(meta["paragraph_id"])

        lookup[(title, paragraph_id)] = int(corpus_id)

    return lookup


corpus_id_by_title_paragraph = build_rse_paragraph_lookup()

print(f"RSE paragraph lookup entries: {len(corpus_id_by_title_paragraph)}")


RSE paragraph lookup entries: 452648


In [ ]:
# Helpers

def rse_get_config():
    # Read RSE parameters from BASE_RAG_CONFIG
    return {
        "top_k_context": BASE_RAG_CONFIG.get("top_k_context", 6),
        "neighbor_window": BASE_RAG_CONFIG.get("rse_neighbor_window", 1),
        "max_segment_length": BASE_RAG_CONFIG.get("rse_max_segment_length", 3),
        "max_total_paragraphs": BASE_RAG_CONFIG.get("rse_max_total_paragraphs", 5),
        "min_segment_score": BASE_RAG_CONFIG.get("rse_min_segment_score", 0.15),
        "max_seed_hits": BASE_RAG_CONFIG.get("rse_max_seed_hits", 5),
        "max_final_sources": BASE_RAG_CONFIG.get("max_final_sources", 3),
        "enable_relevance_check": BASE_RAG_CONFIG.get("enable_llm_relevance_check", True),
        "rag_max_new_tokens": BASE_RAG_CONFIG.get("rag_max_new_tokens", 64),
    }


def rse_build_chat_prompt(system_msg, user_msg, tokenizer):
    # Convert system/user messages to model chat format
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def rse_compact_hits_for_debug(hits):
    # Compact version of hits for readable debug
    compact = []

    for hit in hits or []:
        corpus_id = int(hit["corpus_id"])

        compact.append({
            "source_label": hit.get("source_label", "?"),
            "corpus_id": corpus_id,
            "title": hit.get("title", passage_metadata[corpus_id]["title"]),
            "paragraph_id": hit.get("paragraph_id", passage_metadata[corpus_id]["paragraph_id"]),
            "score": float(hit.get("score", 0.0)),
            "query": hit.get("query", ""),
            "query_type": hit.get("query_type", ""),
            "used_options": bool(hit.get("used_options", False)),
            "text": hit.get("text", passages[corpus_id]),
        })

    return compact


def rse_build_final_answer_prompt(question, context):
    # Use RAG prompt if RSE context exists, otherwise fallback to base LLM prompt

    if context.strip():
        system_msg, user_msg = build_rag_prompt(
            question.text,
            question.options,
            context,
        )

        answer_parser = extract_option_id
        max_new_tokens = BASE_RAG_CONFIG.get("rag_max_new_tokens", 64)
        prompt_mode = "rag_with_rse_context"

    else:
        system_msg, user_msg = build_prompt(
            question.text,
            question.options,
        )

        answer_parser = parse_answer
        max_new_tokens = CONFIG.get("max_new_tokens", 15)
        prompt_mode = "llm_fallback_no_context"

        if CONFIG.get("prompt_strategy") in {"zero_shot_cot", "cot"}:
            max_new_tokens = CONFIG.get("reasoning_max_new_tokens", 256)

    return system_msg, user_msg, answer_parser, max_new_tokens, prompt_mode

In [ ]:
# Hit selection

def rse_select_hits_from_relevance(
    candidate_hits,
    relevant_source_labels,
    parsing_failed=False,
    max_seed_hits=5,
):
    # If relevance parsing fails, keep first hits
    if parsing_failed:
        return candidate_hits[:max_seed_hits]

    # If some sources are relevant, keep only those
    if relevant_source_labels:
        return [
            hit for hit in candidate_hits
            if hit.get("source_label") in relevant_source_labels
        ][:max_seed_hits]

    # If no source is relevant, return no seed hits
    return []


def rse_select_seed_hits(question, candidate_hits, tokenizer, model):
    # Select seed hits with optional LLM relevance filtering

    config = rse_get_config()

    relevance_start = time.perf_counter()
    relevance_parsing_failed = False

    if config["enable_relevance_check"]:
        relevant_source_labels, relevance_parsing_failed = check_all_sources_relevance(
            question,
            candidate_hits,
            tokenizer,
            model,
        )
    else:
        relevant_source_labels = [
            hit.get("source_label")
            for hit in candidate_hits[:config["max_seed_hits"]]
            if hit.get("source_label")
        ]

    selected_hits = rse_select_hits_from_relevance(
        candidate_hits,
        relevant_source_labels,
        parsing_failed=relevance_parsing_failed,
        max_seed_hits=config["max_seed_hits"],
    )

    relevance_time_sec = time.perf_counter() - relevance_start

    return selected_hits, {
        "relevant_source_labels": relevant_source_labels,
        "relevance_parsing_failed": relevance_parsing_failed,
        "relevance_time_sec": relevance_time_sec,
    }


In [ ]:
# Segment construction

def rse_get_neighbor_paragraph_ids(center_paragraph_id, neighbor_window, max_segment_length):
    # Example:
    # center = 6, window = 1 -> [5, 6, 7]
    start_paragraph = center_paragraph_id - neighbor_window
    end_paragraph = center_paragraph_id + neighbor_window

    paragraph_ids = list(range(start_paragraph, end_paragraph + 1))

    # Keep segment short
    if len(paragraph_ids) > max_segment_length:
        paragraph_ids = paragraph_ids[:max_segment_length]

    return paragraph_ids


def rse_build_segment_from_hit(
    hit,
    used_paragraph_keys,
    total_paragraphs,
    config,
):
    # Expand one retrieved paragraph into a local segment

    corpus_id = int(hit["corpus_id"])
    score = float(hit.get("score", 0.0))

    if score < config["min_segment_score"]:
        return None, total_paragraphs

    title = passage_metadata[corpus_id]["title"]
    center_paragraph_id = int(passage_metadata[corpus_id]["paragraph_id"])
    source_label = hit.get("source_label", "?")

    paragraph_ids = rse_get_neighbor_paragraph_ids(
        center_paragraph_id,
        config["neighbor_window"],
        config["max_segment_length"],
    )

    segment_paragraphs = []

    for paragraph_id in paragraph_ids:
        if total_paragraphs >= config["max_total_paragraphs"]:
            break

        key = (title, paragraph_id)

        if key in used_paragraph_keys:
            continue

        neighbor_corpus_id = corpus_id_by_title_paragraph.get(key)

        if neighbor_corpus_id is None:
            continue

        used_paragraph_keys.add(key)
        total_paragraphs += 1

        segment_paragraphs.append({
            "corpus_id": int(neighbor_corpus_id),
            "title": title,
            "paragraph_id": int(paragraph_id),
            "text": passages[neighbor_corpus_id],
            "is_retrieved_hit": paragraph_id == center_paragraph_id,
            "retrieved_score": score if paragraph_id == center_paragraph_id else None,
        })

    if not segment_paragraphs:
        return None, total_paragraphs

    segment = {
        "source_label": source_label,
        "title": title,
        "center_paragraph_id": center_paragraph_id,
        "start_paragraph_id": segment_paragraphs[0]["paragraph_id"],
        "end_paragraph_id": segment_paragraphs[-1]["paragraph_id"],
        "segment_score": score,
        "source_hit": hit,
        "paragraphs": segment_paragraphs,
    }

    return segment, total_paragraphs


def rse_build_segments_from_hits(hits):
    # Expand selected retrieved hits into RSE segments

    if not hits:
        return []

    config = rse_get_config()

    segments = []
    used_paragraph_keys = set()
    total_paragraphs = 0

    for hit in hits:
        if total_paragraphs >= config["max_total_paragraphs"]:
            break

        segment, total_paragraphs = rse_build_segment_from_hit(
            hit,
            used_paragraph_keys,
            total_paragraphs,
            config,
        )

        if segment is not None:
            segments.append(segment)

    return segments

In [ ]:
# Context formatting

def rse_format_context(rse_segments):
    # Format RSE segments into final context string

    if not rse_segments:
        return ""

    context_blocks = []

    for segment in rse_segments:
        source_label = segment.get("source_label", "?")
        title = segment["title"]
        start_pid = segment["start_paragraph_id"]
        end_pid = segment["end_paragraph_id"]
        segment_score = float(segment.get("segment_score", 0.0))

        block_lines = [
            f"Context Segment {source_label}",
            (
                f"Title: {title} | Paragraphs: {start_pid}-{end_pid} "
                f"| Segment score: {segment_score:.4f}"
            ),
        ]

        for paragraph in segment["paragraphs"]:
            paragraph_id = paragraph["paragraph_id"]
            paragraph_text = paragraph["text"]

            if paragraph.get("is_retrieved_hit"):
                retrieved_score = paragraph.get("retrieved_score")
                block_lines.append(
                    f"[Paragraph {paragraph_id}] Retrieved score: {retrieved_score:.4f}"
                )
            else:
                block_lines.append(f"[Paragraph {paragraph_id}]")

            block_lines.append(paragraph_text)

        context_blocks.append("\n".join(block_lines))

    return "\n\n".join(context_blocks)

In [ ]:
# Retrieval wrapper

def retrieve_local_wikipedia_context_rse(question, top_k=None):
    # Retrieve candidate paragraphs and enrich them with neighboring paragraphs

    config = rse_get_config()

    if top_k is None:
        top_k = config["top_k_context"]

    candidate_hits = retrieve_local_wikipedia_context_with_options_candidates(
        question,
        top_k=top_k,
    )

    selected_hits = candidate_hits[:config["max_seed_hits"]]
    rse_segments = rse_build_segments_from_hits(selected_hits)
    context = rse_format_context(rse_segments)

    return context


def retrieve_local_wikipedia_context_rse_logged(question, tokenizer=None, model=None, top_k=None):
    # Logged RSE retrieval used by the main answer function

    config = rse_get_config()

    if top_k is None:
        top_k = config["top_k_context"]

    retrieval_start = time.perf_counter()

    candidate_hits = retrieve_local_wikipedia_context_with_options_candidates(
        question,
        top_k=top_k,
    )

    retrieval_time_sec = time.perf_counter() - retrieval_start

    selected_hits, relevance_details = rse_select_seed_hits(
        question,
        candidate_hits,
        tokenizer,
        model,
    )

    rse_start = time.perf_counter()

    rse_segments = rse_build_segments_from_hits(selected_hits)
    context = rse_format_context(rse_segments)

    rse_time_sec = time.perf_counter() - rse_start

    details = {
        "candidate_hits": candidate_hits,
        "selected_hits": selected_hits,
        "rse_segments": rse_segments,
        "context": context,

        "retrieval_time_sec": retrieval_time_sec,
        "relevance_time_sec": relevance_details["relevance_time_sec"],
        "rse_time_sec": rse_time_sec,

        "relevant_source_labels": relevance_details["relevant_source_labels"],
        "relevance_parsing_failed": relevance_details["relevance_parsing_failed"],
    }

    return context, candidate_hits, selected_hits, rse_segments, details

In [ ]:
# Main answer function

def get_rse_rag_answer_logged(question, tokenizer, model, max_retries=3):
    start_total = time.perf_counter()

    valid_ids = [opt.id for opt in question.options]

    # 1. Retrieval + relevance selection + RSE enrichment
    context, candidate_hits, selected_hits, rse_segments, retrieval_details = (
        retrieve_local_wikipedia_context_rse_logged(
            question,
            tokenizer=tokenizer,
            model=model,
            top_k=BASE_RAG_CONFIG.get("top_k_context", 6),
        )
    )

    # 2. Build answer prompt
    system_msg, user_msg, answer_parser, max_new_tokens, prompt_mode = (
        rse_build_final_answer_prompt(question, context)
    )

    prompt = rse_build_chat_prompt(
        system_msg,
        user_msg,
        tokenizer,
    )

    # 3. Generate answer with retries
    generation_start = time.perf_counter()

    final_answer = None
    attempts = []
    last_output = ""

    for attempt_idx in range(1, max_retries + 1):
        output = generate_answer(
            prompt,
            tokenizer,
            model,
            do_sample=CONFIG.get("do_sample", False),
            max_new_tokens=max_new_tokens,
            temperature=CONFIG.get("temperature", 1.0),
        )

        last_output = output
        parsed_answer = answer_parser(output, valid_ids)

        attempts.append({
            "attempt": attempt_idx,
            "raw_output": output,
            "parsed_answer": parsed_answer,
        })

        if parsed_answer is not None:
            final_answer = parsed_answer
            break

    generation_time_sec = time.perf_counter() - generation_start

    # 4. Fallback if parsing failed
    parsing_failed = False
    fallback_used = False
    fallback_reason = None

    if final_answer is None:
        final_answer = valid_ids[0]
        parsing_failed = True
        fallback_used = True
        fallback_reason = "default_first_valid_option"

    total_answer_time_sec = time.perf_counter() - start_total

    # 5. Metadata for Advanced Debugger
    details = {
        "active_rag_variant": "rse_rag",
        "rag_type": "rse_local_rag",

        "prompt_mode": prompt_mode,
        "context_status": f"rse_segments_{len(rse_segments)}",
        "selection_mode": "rse_relevance_selection_then_segment_expansion",

        "relevant_source_labels": retrieval_details["relevant_source_labels"],
        "relevance_parsing_failed": retrieval_details["relevance_parsing_failed"],

        "hits": rse_compact_hits_for_debug(candidate_hits),
        "candidate_hits": rse_compact_hits_for_debug(candidate_hits),
        "selected_hits": rse_compact_hits_for_debug(selected_hits),

        "rse_segments": rse_segments,
        "context": context,
        "context_used": bool(context.strip()),

        "attempts": attempts,
        "raw_output": last_output,

        "answer_id": final_answer,
        "parsing_failed": parsing_failed,
        "fallback_used": fallback_used,
        "fallback_reason": fallback_reason,

        "retrieval_time_sec": retrieval_details["retrieval_time_sec"],
        "relevance_time_sec": retrieval_details["relevance_time_sec"],
        "rse_time_sec": retrieval_details["rse_time_sec"],
        "generation_time_sec": generation_time_sec,
        "total_answer_time_sec": total_answer_time_sec,
    }

    return final_answer, details

In [ ]:
# Shared dispatcher

def get_rse_rag_answer(question, tokenizer, model, max_retries=3):
    answer_id, _ = get_rse_rag_answer_logged(
        question,
        tokenizer,
        model,
        max_retries=max_retries,
    )
    return answer_id

In [ ]:
# Activation

def activate_rse_rag_variant(
    top_k_context=6,
    rse_neighbor_window=1,
    rse_max_segment_length=3,
    rse_max_total_paragraphs=3,
    rse_max_seed_hits=5,
    enable_llm_relevance_check=False,
):
    global retrieve_local_wikipedia_context
    global get_rag_answer_logged

    retrieve_local_wikipedia_context = retrieve_local_wikipedia_context_rse
    get_rag_answer_logged = get_rse_rag_answer_logged

    BASE_RAG_CONFIG.update({
        "active_rag_variant": "rse_rag",

        "top_k_context": top_k_context,

        "rse_neighbor_window": rse_neighbor_window,
        "rse_max_segment_length": rse_max_segment_length,
        "rse_max_total_paragraphs": rse_max_total_paragraphs,
        "rse_min_segment_score": 0.15,
        "rse_max_seed_hits": rse_max_seed_hits,

        "max_final_sources": rse_max_seed_hits,

        "enable_llm_relevance_check": enable_llm_relevance_check,
        "relevance_max_new_tokens": 16,

        "rag_max_new_tokens": 64,
        "debug": True,
    })

    print("RSE-RAG activated.")
    print(f"top_k_context: {top_k_context}")
    print(f"neighbor_window: {rse_neighbor_window}")
    print(f"max_segment_length: {rse_max_segment_length}")
    print(f"max_total_paragraphs: {rse_max_total_paragraphs}")
    print(f"max_seed_hits: {rse_max_seed_hits}")
    print(f"LLM relevance filtering: {enable_llm_relevance_check}")


print("RSE-RAG cell loaded successfully.")

RSE-RAG cell loaded successfully.


## 🟢 Contextual Compression



In [ ]:
import re
import time

In [ ]:
# Compression helpers

def compression_clean_output(raw_output):
    # Clean compressed evidence generated by the local LLM
    text = str(raw_output or "").strip()

    text = re.sub(
        r"^\s*(compressed context|compressed evidence|relevant evidence|evidence)\s*:\s*",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()

    forbidden_starts = (
        "therefore",
        "so the answer",
        "the answer",
        "answer",
        "final answer",
        "option",
        "selected option",
        "correct option",
        "evidence supports",
        "evidence suggests",
        "evidence points",
        "this supports",
        "this means",
    )

    lines = []

    for line in text.splitlines():
        line = line.strip()

        if not line:
            continue

        normalized_line = line.lstrip("-*0123456789. )").strip().lower()

        # Remove lines where the compressor accidentally answers the question
        if any(normalized_line.startswith(prefix) for prefix in forbidden_starts):
            continue

        lines.append(line)

    return "\n".join(lines).strip()


def compression_build_prompt(question, context):
    # Build prompt for context compression

    options_text = format_options(question.options)

    system_msg = (
        "You compress retrieved Wikipedia context.\n"
        "Your job is to keep only the evidence useful for the question.\n"
        "Do not answer the question.\n"
        "Do not choose an option.\n"
        "Do not invent information.\n\n"
        "Rules:\n"
        "- Copy or minimally shorten useful evidence from the context.\n"
        "- Use short bullet points.\n"
        "- Keep names, dates, definitions, and cause-effect relations.\n"
        "- If there is no useful evidence, return EMPTY."
    )

    user_msg = (
        f"Question:\n{question.text}\n\n"
        f"Options:\n{options_text}\n\n"
        f"Retrieved context:\n{context}\n\n"
        "Compressed evidence:"
    )

    return system_msg, user_msg


def compression_build_chat_prompt(system_msg, user_msg, tokenizer):
    # Convert system/user messages to model chat format
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )


def compress_context_with_local_llm_logged(question, context, tokenizer, model):
    # Compress retrieved context and keep debug info

    if not context or not context.strip():
        return "", {
            "compression_raw_output": "",
            "compression_used": False,
            "compression_fallback_used": False,
        }

    system_msg, user_msg = compression_build_prompt(question, context)

    prompt = compression_build_chat_prompt(
        system_msg,
        user_msg,
        tokenizer,
    )

    raw_output = generate_answer(
        prompt,
        tokenizer,
        model,
        do_sample=False,
        max_new_tokens=BASE_RAG_CONFIG.get("compression_max_new_tokens", 160),
        temperature=1.0,
    )

    compressed_context = compression_clean_output(raw_output)
    normalized = compressed_context.strip().lstrip("-* ").strip().upper()

    # If compression fails or returns EMPTY, keep original context
    if not compressed_context or normalized == "EMPTY":
        return context, {
            "compression_raw_output": raw_output,
            "compression_used": False,
            "compression_fallback_used": True,
        }

    return compressed_context, {
        "compression_raw_output": raw_output,
        "compression_used": True,
        "compression_fallback_used": False,
    }


# Backward-compatible non-logged version
def compress_context_with_local_llm(question, context, tokenizer, model):
    compressed_context, _ = compress_context_with_local_llm_logged(
        question,
        context,
        tokenizer,
        model,
    )
    return compressed_context

In [ ]:
# Safe hit/context helpers

def compression_select_hits_from_relevance(
    candidate_hits,
    relevant_source_labels,
    parsing_failed=False,
    max_final_sources=3,
):
    # If relevance parsing fails, keep first hits
    if parsing_failed:
        return candidate_hits[:max_final_sources]

    # If some sources are relevant, keep only those
    if relevant_source_labels:
        return [
            hit for hit in candidate_hits
            if hit.get("source_label") in relevant_source_labels
        ][:max_final_sources]

    # If no source is relevant, use no context
    return []


def compression_build_basic_context_from_hits(hits, max_final_sources=3):
    # Build basic context from selected hits without RSE

    selected_hits = hits[:max_final_sources]
    context_blocks = []

    for hit in selected_hits:
        corpus_id = int(hit["corpus_id"])
        source_label = hit.get("source_label", "?")
        score = float(hit.get("score", 0.0))

        title = passage_metadata[corpus_id]["title"]
        paragraph_id = passage_metadata[corpus_id]["paragraph_id"]
        passage_text = passages[corpus_id]

        context_blocks.append(
            f"Context Source {source_label}\n"
            f"Title: {title} | Paragraph: {paragraph_id} | Retrieval score: {score:.4f}\n"
            f"{passage_text}"
        )

    return "\n\n".join(context_blocks), selected_hits


def compression_compact_hits_for_debug(hits):
    # Compact version of hits for readable debug

    compact = []

    for hit in hits or []:
        corpus_id = int(hit["corpus_id"])

        compact.append({
            "source_label": hit.get("source_label", "?"),
            "corpus_id": corpus_id,
            "title": hit.get("title", passage_metadata[corpus_id]["title"]),
            "paragraph_id": hit.get("paragraph_id", passage_metadata[corpus_id]["paragraph_id"]),
            "score": float(hit.get("score", 0.0)),
            "query": hit.get("query", ""),
            "query_type": hit.get("query_type", ""),
            "used_options": bool(hit.get("used_options", False)),
            "text": hit.get("text", passages[corpus_id]),
        })

    return compact


def compression_build_final_answer_prompt(question, context):
    # Use RAG prompt if context exists, otherwise fallback to base LLM prompt

    if context.strip():
        system_msg, user_msg = build_rag_prompt(
            question.text,
            question.options,
            context,
        )

        answer_parser = extract_option_id
        max_new_tokens = BASE_RAG_CONFIG.get("rag_max_new_tokens", 64)
        prompt_mode = "rag_with_compressed_context"

    else:
        system_msg, user_msg = build_prompt(
            question.text,
            question.options,
        )

        answer_parser = parse_answer
        max_new_tokens = CONFIG.get("max_new_tokens", 15)
        prompt_mode = "llm_fallback_no_context"

        if CONFIG.get("prompt_strategy") in {"zero_shot_cot", "cot"}:
            max_new_tokens = CONFIG.get("reasoning_max_new_tokens", 256)

    return system_msg, user_msg, answer_parser, max_new_tokens, prompt_mode

In [ ]:
# Retrieval and relevance selection

def compression_retrieve_candidate_hits(question):
    # Retrieve candidate paragraphs from local Wikipedia

    candidate_hits = retrieve_local_wikipedia_context_with_options_candidates(
        question,
        top_k=BASE_RAG_CONFIG.get("top_k_context", 6),
    )

    return candidate_hits


def compression_select_seed_hits(question, candidate_hits, tokenizer, model):
    # Optional LLM relevance filtering before RSE/compression

    relevance_start = time.perf_counter()
    relevance_parsing_failed = False

    if BASE_RAG_CONFIG.get("enable_llm_relevance_check", True):
        relevant_source_labels, relevance_parsing_failed = check_all_sources_relevance(
            question,
            candidate_hits,
            tokenizer,
            model,
        )
    else:
        relevant_source_labels = [
            hit.get("source_label")
            for hit in candidate_hits[:BASE_RAG_CONFIG.get("rse_max_seed_hits", 5)]
            if hit.get("source_label")
        ]

    selected_seed_hits = compression_select_hits_from_relevance(
        candidate_hits,
        relevant_source_labels,
        parsing_failed=relevance_parsing_failed,
        max_final_sources=BASE_RAG_CONFIG.get("rse_max_seed_hits", 5),
    )

    relevance_time_sec = time.perf_counter() - relevance_start

    return selected_seed_hits, {
        "relevant_source_labels": relevant_source_labels,
        "relevance_parsing_failed": relevance_parsing_failed,
        "relevance_time_sec": relevance_time_sec,
    }

In [ ]:
# Context construction

def compression_build_context_before_compression(selected_seed_hits):
    # Build context with either RSE enrichment or basic selected hits

    rse_start = time.perf_counter()
    rse_segments = []

    if BASE_RAG_CONFIG.get("enable_rse", True):
        rse_segments = rse_build_segments_from_hits(selected_seed_hits)
        context_before_compression = rse_format_context(rse_segments)
        context_status = f"rse_segments_{len(rse_segments)}"

    else:
        context_before_compression, selected_seed_hits = compression_build_basic_context_from_hits(
            selected_seed_hits,
            max_final_sources=BASE_RAG_CONFIG.get("max_final_sources", 3),
        )
        context_status = f"basic_context_{len(selected_seed_hits)}_sources"

    rse_time_sec = time.perf_counter() - rse_start

    return context_before_compression, selected_seed_hits, rse_segments, {
        "context_status": context_status,
        "rse_time_sec": rse_time_sec,
    }

In [ ]:
# Compression step

def compression_maybe_compress_context(question, context_before_compression, tokenizer, model, context_status):
    # Optionally compress the context before answer generation

    compression_time_sec = 0.0

    if (
        BASE_RAG_CONFIG.get("enable_contextual_compression", True)
        and context_before_compression.strip()
    ):
        compression_start = time.perf_counter()

        context, compression_details = compress_context_with_local_llm_logged(
            question,
            context_before_compression,
            tokenizer,
            model,
        )

        compression_time_sec = time.perf_counter() - compression_start

        if compression_details.get("compression_used"):
            context_status = context_status + "_compressed"
        else:
            context_status = context_status + "_compression_fallback"

    else:
        context = context_before_compression
        compression_details = {
            "compression_raw_output": "",
            "compression_used": False,
            "compression_fallback_used": False,
        }

    compression_details["compression_time_sec"] = compression_time_sec
    compression_details["context_status"] = context_status

    return context, compression_details

In [ ]:
# Main answer function

def get_rse_compressed_rag_answer_logged(question, tokenizer, model, max_retries=3):
    start_total = time.perf_counter()
    valid_ids = [opt.id for opt in question.options]

    # --------------------------------------------------------
    # 1. Retrieve candidate paragraphs
    # --------------------------------------------------------

    retrieval_start = time.perf_counter()

    candidate_hits = compression_retrieve_candidate_hits(question)

    retrieval_time_sec = time.perf_counter() - retrieval_start

    # --------------------------------------------------------
    # 2. Relevance selection
    # --------------------------------------------------------

    selected_seed_hits, relevance_details = compression_select_seed_hits(
        question,
        candidate_hits,
        tokenizer,
        model,
    )

    # --------------------------------------------------------
    # 3. RSE/basic context construction
    # --------------------------------------------------------

    (
        context_before_compression,
        selected_seed_hits,
        rse_segments,
        context_details,
    ) = compression_build_context_before_compression(selected_seed_hits)

    # --------------------------------------------------------
    # 4. Optional contextual compression
    # --------------------------------------------------------

    context, compression_details = compression_maybe_compress_context(
        question,
        context_before_compression,
        tokenizer,
        model,
        context_details["context_status"],
    )

    # --------------------------------------------------------
    # 5. Build final answer prompt
    # --------------------------------------------------------

    (
        system_msg,
        user_msg,
        answer_parser,
        max_new_tokens,
        prompt_mode,
    ) = compression_build_final_answer_prompt(question, context)

    prompt = compression_build_chat_prompt(
        system_msg,
        user_msg,
        tokenizer,
    )

    # --------------------------------------------------------
    # 6. Generate answer with retries
    # --------------------------------------------------------

    generation_start = time.perf_counter()

    final_answer = None
    attempts = []
    last_output = ""

    for attempt_idx in range(1, max_retries + 1):
        output = generate_answer(
            prompt,
            tokenizer,
            model,
            do_sample=CONFIG.get("do_sample", False),
            max_new_tokens=max_new_tokens,
            temperature=CONFIG.get("temperature", 1.0),
        )

        last_output = output
        parsed_answer = answer_parser(output, valid_ids)

        attempts.append({
            "attempt": attempt_idx,
            "raw_output": output,
            "parsed_answer": parsed_answer,
        })

        if parsed_answer is not None:
            final_answer = parsed_answer
            break

    generation_time_sec = time.perf_counter() - generation_start

    # --------------------------------------------------------
    # 7. Fallback if parsing failed
    # --------------------------------------------------------

    parsing_failed = False
    fallback_used = False
    fallback_reason = None

    if final_answer is None:
        final_answer = valid_ids[0]
        parsing_failed = True
        fallback_used = True
        fallback_reason = "default_first_valid_option"

    total_answer_time_sec = time.perf_counter() - start_total

    # --------------------------------------------------------
    # 8. Metadata for Advanced Debugger
    # --------------------------------------------------------

    details = {
        "active_rag_variant": "rse_contextual_compression_rag",
        "rag_type": "rse_contextual_compression_local_rag",

        "prompt_mode": prompt_mode,
        "context_status": compression_details["context_status"],
        "selection_mode": "relevance_selection_rse_then_contextual_compression",

        "relevant_source_labels": relevance_details["relevant_source_labels"],
        "relevance_parsing_failed": relevance_details["relevance_parsing_failed"],

        "hits": compression_compact_hits_for_debug(candidate_hits),
        "candidate_hits": compression_compact_hits_for_debug(candidate_hits),
        "selected_seed_hits": compression_compact_hits_for_debug(selected_seed_hits),
        "selected_hits": compression_compact_hits_for_debug(selected_seed_hits),

        "rse_segments": rse_segments,
        "context_before_compression": context_before_compression,
        "context": context,
        "context_used": bool(context.strip()),

        "compression_raw_output": compression_details["compression_raw_output"],
        "compression_used": compression_details["compression_used"],
        "compression_fallback_used": compression_details["compression_fallback_used"],

        "attempts": attempts,
        "raw_output": last_output,

        "answer_id": final_answer,
        "parsing_failed": parsing_failed,
        "fallback_used": fallback_used,
        "fallback_reason": fallback_reason,

        "retrieval_time_sec": retrieval_time_sec,
        "relevance_time_sec": relevance_details["relevance_time_sec"],
        "rse_time_sec": context_details["rse_time_sec"],
        "compression_time_sec": compression_details["compression_time_sec"],
        "generation_time_sec": generation_time_sec,
        "total_answer_time_sec": total_answer_time_sec,
    }

    return final_answer, details

In [ ]:
# ============================================================
# Shared dispatcher
# ============================================================

def get_rse_compressed_rag_answer(question, tokenizer, model, max_retries=3):
    answer_id, _ = get_rse_compressed_rag_answer_logged(
        question,
        tokenizer,
        model,
        max_retries=max_retries,
    )
    return answer_id

In [ ]:
# ============================================================
# Activation
# ============================================================

def activate_rse_compressed_rag_variant(
    top_k_context=6,
    max_final_sources=3,
    enable_rse=True,
    rse_neighbor_window=1,
    rse_max_segment_length=3,
    rse_max_total_paragraphs=5,
    rse_max_seed_hits=5,
    enable_llm_relevance_check=True,
    enable_contextual_compression=True,
):
    global retrieve_local_wikipedia_context
    global get_rag_answer_logged

    retrieve_local_wikipedia_context = retrieve_local_wikipedia_context_rse
    get_rag_answer_logged = get_rse_compressed_rag_answer_logged

    BASE_RAG_CONFIG.update({
        "active_rag_variant": "rse_contextual_compression_rag",

        "top_k_context": top_k_context,
        "max_final_sources": max_final_sources,

        "enable_rse": enable_rse,
        "rse_neighbor_window": rse_neighbor_window,
        "rse_max_segment_length": rse_max_segment_length,
        "rse_max_total_paragraphs": rse_max_total_paragraphs,
        "rse_min_segment_score": 0.15,
        "rse_max_seed_hits": rse_max_seed_hits,

        "enable_llm_relevance_check": enable_llm_relevance_check,
        "relevance_max_new_tokens": 16,

        "enable_contextual_compression": enable_contextual_compression,
        "compression_max_new_tokens": 160,

        "rag_max_new_tokens": 64,
    })

    print("RSE + Contextual Compression RAG activated.")
    print(f"top_k_context: {top_k_context}")
    print(f"max_final_sources: {max_final_sources}")
    print(f"RSE enabled: {enable_rse}")
    print(f"neighbor_window: {rse_neighbor_window}")
    print(f"max_segment_length: {rse_max_segment_length}")
    print(f"max_total_paragraphs: {rse_max_total_paragraphs}")
    print(f"max_seed_hits: {rse_max_seed_hits}")
    print(f"Contextual compression enabled: {enable_contextual_compression}")
    print(f"LLM relevance filtering: {enable_llm_relevance_check}")


print("RSE + Contextual Compression RAG cell loaded successfully.")

RSE + Contextual Compression RAG cell loaded successfully.


# 🔵 Advanced Retrieval

## 🟢 Fusion Retrieval


In [ ]:
import math
import re
import pickle
import time
import numpy as np
from pathlib import Path
from collections import Counter, defaultdict

In [ ]:
# BM25 cache and global state

BM25_CACHE_PATH = (
    Path(SEMANTIC_SEARCH_DIR)
    / f"simplewiki_bm25_first_{MAX_PARAGRAPHS_PER_ARTICLE}_paragraphs.pkl"
)

BM25_SIGNATURE = {
    **_CACHE_SIGNATURE,
    "num_passages": len(passages),
    "tokenizer": r"[a-zA-Z0-9]+",
    "bm25_version": 1,
}

bm25_doc_freqs = []
bm25_doc_lengths = []
bm25_avg_doc_length = 0.0
bm25_idf = {}
bm25_postings = defaultdict(set)

In [ ]:
# BM25 tokenization/indexing

def fusion_tokenize_for_bm25(text):
    # "Roman Republic and patricians" -> ["roman", "republic", "and", "patricians"]
    return re.findall(r"[a-zA-Z0-9]+", str(text or "").lower())


def fusion_build_bm25_index():
    # Build local BM25 structures from passages

    global bm25_doc_freqs
    global bm25_doc_lengths
    global bm25_avg_doc_length
    global bm25_idf
    global bm25_postings

    bm25_doc_freqs = []
    bm25_doc_lengths = []
    bm25_idf = {}
    bm25_postings = defaultdict(set)

    document_frequency = defaultdict(int)

    for corpus_id, passage in enumerate(passages):
        tokens = fusion_tokenize_for_bm25(passage)
        word_counts = Counter(tokens)

        bm25_doc_freqs.append(word_counts)
        bm25_doc_lengths.append(len(tokens))

        for token in word_counts:
            document_frequency[token] += 1
            bm25_postings[token].add(corpus_id)

    num_docs = len(passages)
    bm25_avg_doc_length = sum(bm25_doc_lengths) / max(num_docs, 1)

    for token, df in document_frequency.items():
        bm25_idf[token] = math.log(
            1 + (num_docs - df + 0.5) / (df + 0.5)
        )

    print(f"Local BM25 index built for {num_docs} passages")
    print(f"BM25 vocabulary size: {len(bm25_idf)}")


# Backward-compatible alias
def tokenize_for_bm25(text):
    return fusion_tokenize_for_bm25(text)


def build_local_bm25_index():
    return fusion_build_bm25_index()

In [ ]:
# BM25 save/load

def fusion_save_bm25_index(cache_path=BM25_CACHE_PATH):
    # Save BM25 index to disk

    cache_path = Path(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    with cache_path.open("wb") as f:
        pickle.dump({
            "signature": BM25_SIGNATURE,
            "bm25_doc_freqs": bm25_doc_freqs,
            "bm25_doc_lengths": bm25_doc_lengths,
            "bm25_avg_doc_length": bm25_avg_doc_length,
            "bm25_idf": bm25_idf,
            "bm25_postings": dict(bm25_postings),
        }, f)

    print(f"BM25 index saved to: {cache_path}")


def fusion_load_bm25_index(cache_path=BM25_CACHE_PATH):
    # Load BM25 index if the signature matches the current corpus/config

    global bm25_doc_freqs
    global bm25_doc_lengths
    global bm25_avg_doc_length
    global bm25_idf
    global bm25_postings

    cache_path = Path(cache_path)

    with cache_path.open("rb") as f:
        data = pickle.load(f)

    if data.get("signature") != BM25_SIGNATURE:
        raise ValueError("Cached BM25 index does not match current corpus/config.")

    bm25_doc_freqs = data["bm25_doc_freqs"]
    bm25_doc_lengths = data["bm25_doc_lengths"]
    bm25_avg_doc_length = data["bm25_avg_doc_length"]
    bm25_idf = data["bm25_idf"]
    bm25_postings = defaultdict(set, data["bm25_postings"])

    print(f"BM25 index loaded from: {cache_path}")


def fusion_load_or_build_bm25_index(cache_path=BM25_CACHE_PATH, force_rebuild=False):
    # Load BM25 cache if available, otherwise rebuild

    cache_path = Path(cache_path)

    if cache_path.exists() and not force_rebuild:
        try:
            fusion_load_bm25_index(cache_path)
            return
        except Exception as error:
            print(f"Could not load cached BM25 index: {error}")
            print("Rebuilding BM25 index...")

    fusion_build_bm25_index()
    fusion_save_bm25_index(cache_path)


# Backward-compatible aliases
def save_bm25_index(cache_path=BM25_CACHE_PATH):
    return fusion_save_bm25_index(cache_path)


def load_bm25_index(cache_path=BM25_CACHE_PATH):
    return fusion_load_bm25_index(cache_path)


def load_or_build_bm25_index(cache_path=BM25_CACHE_PATH, force_rebuild=False):
    return fusion_load_or_build_bm25_index(cache_path, force_rebuild=force_rebuild)


fusion_load_or_build_bm25_index()

BM25 index loaded from: /content/gdrive/MyDrive/NLP/PoliMillionaire/SemanticSearch/simplewiki_bm25_first_10_paragraphs.pkl


In [ ]:
# Query builders

def fusion_build_query(question, mode="question_options"):
    # question_only:
    # "What was the social structure of ancient Rome?"
    #
    # question_options:
    # "What was the social structure of ancient Rome? military society classless society patricians plebeians caste system"

    question_text = question.text.strip()

    if mode == "question_only":
        return question_text

    if mode == "question_options":
        options_text = ""

        if hasattr(question, "options") and question.options is not None:
            options_text = " ".join([opt.text for opt in question.options])

        return f"{question_text} {options_text}".strip()

    raise ValueError("Unknown mode. Use 'question_only' or 'question_options'.")


# Backward-compatible alias
def build_fusion_query(question, mode="question_options"):
    return fusion_build_query(question, mode=mode)


def fusion_get_queries(question):
    # Build semantic and BM25 queries separately

    semantic_query = fusion_build_query(
        question,
        mode=BASE_RAG_CONFIG.get(
            "fusion_semantic_query_mode",
            "question_options",
        ),
    )

    bm25_query = fusion_build_query(
        question,
        mode=BASE_RAG_CONFIG.get(
            "fusion_bm25_query_mode",
            "question_only",
        ),
    )

    return semantic_query, bm25_query

In [ ]:
# Retrieval methods

def fusion_semantic_faiss_search(query, top_k=20):
    # Semantic retrieval with SentenceTransformer + FAISS

    query_embedding = semb_model.encode(
        query,
        convert_to_tensor=False,
    ).astype("float32")

    query_embedding = np.expand_dims(query_embedding, axis=0)

    # Normalize so inner product works like cosine similarity
    faiss.normalize_L2(query_embedding)

    scores, indices = faiss_index.search(query_embedding, top_k)

    results = []

    for corpus_id, score in zip(indices[0], scores[0]):
        corpus_id = int(corpus_id)

        if corpus_id < 0:
            continue

        results.append({
            "corpus_id": corpus_id,
            "score": float(score),
        })

    return results


def fusion_bm25_score(query_tokens, corpus_id, k1=1.5, b=0.75):
    # Compute BM25 score for one passage

    corpus_id = int(corpus_id)

    score = 0.0
    doc_words = bm25_doc_freqs[corpus_id]
    doc_length = bm25_doc_lengths[corpus_id]

    for token in query_tokens:
        token_count = doc_words.get(token, 0)

        if token_count == 0:
            continue

        idf = bm25_idf.get(token, 0.0)

        denominator = token_count + k1 * (
            1 - b + b * doc_length / max(bm25_avg_doc_length, 1)
        )

        score += idf * (token_count * (k1 + 1)) / denominator

    return score


def fusion_bm25_search(query, top_k=20):
    # Lexical retrieval using local BM25 index

    query_tokens = fusion_tokenize_for_bm25(query)

    candidate_ids = set()

    for token in query_tokens:
        candidate_ids.update(bm25_postings.get(token, set()))

    results = []

    for corpus_id in candidate_ids:
        score = fusion_bm25_score(query_tokens, corpus_id)

        if score > 0:
            results.append({
                "corpus_id": int(corpus_id),
                "score": float(score),
            })

    return sorted(
        results,
        key=lambda x: x["score"],
        reverse=True,
    )[:top_k]


# Backward-compatible aliases
def semantic_faiss_search(query, top_k=20):
    return fusion_semantic_faiss_search(query, top_k=top_k)


def bm25_score(query_tokens, corpus_id, k1=1.5, b=0.75):
    return fusion_bm25_score(query_tokens, corpus_id, k1=k1, b=b)


def bm25_search(query, top_k=20):
    return fusion_bm25_search(query, top_k=top_k)

In [ ]:
# Fusion ranking

def fusion_reciprocal_rank_fusion(
    semantic_results,
    bm25_results,
    top_k=6,
    rrf_k=60,
):
    # Combine semantic and BM25 rankings using Reciprocal Rank Fusion

    fused_scores = defaultdict(float)
    details = defaultdict(dict)

    for rank, item in enumerate(semantic_results, start=1):
        corpus_id = int(item["corpus_id"])

        fused_scores[corpus_id] += 1.0 / (rrf_k + rank)

        details[corpus_id]["semantic_rank"] = rank
        details[corpus_id]["semantic_score"] = float(item["score"])

    for rank, item in enumerate(bm25_results, start=1):
        corpus_id = int(item["corpus_id"])

        fused_scores[corpus_id] += 1.0 / (rrf_k + rank)

        details[corpus_id]["bm25_rank"] = rank
        details[corpus_id]["bm25_score"] = float(item["score"])

    ranked_items = sorted(
        fused_scores.items(),
        key=lambda x: x[1],
        reverse=True,
    )

    if not ranked_items:
        return []

    max_score = ranked_items[0][1]
    fused_hits = []

    for corpus_id, raw_score in ranked_items[:top_k]:
        hit = {
            "corpus_id": int(corpus_id),
            "score": float(raw_score / max_score),
        }

        hit.update(details[corpus_id])
        fused_hits.append(hit)

    return fused_hits


def fusion_prepare_single_method_hits(results, method_name, top_k):
    # Normalize semantic-only or BM25-only results into the same hit format

    hits = []

    for rank, item in enumerate(results[:top_k], start=1):
        hit = dict(item)

        if method_name == "semantic":
            hit["semantic_rank"] = rank
            hit["semantic_score"] = float(item["score"])
            hit["bm25_rank"] = None
            hit["bm25_score"] = None

        elif method_name == "bm25":
            hit["bm25_rank"] = rank
            hit["bm25_score"] = float(item["score"])
            hit["semantic_rank"] = None
            hit["semantic_score"] = None

        hits.append(hit)

    return hits


# Backward-compatible alias
def reciprocal_rank_fusion(semantic_results, bm25_results, top_k=6, rrf_k=60):
    return fusion_reciprocal_rank_fusion(
        semantic_results,
        bm25_results,
        top_k=top_k,
        rrf_k=rrf_k,
    )

In [ ]:
# Context formatting

def fusion_add_metadata_to_hit(hit, rank):
    # Add source label and readable metadata for debugger

    corpus_id = int(hit["corpus_id"])

    enriched_hit = dict(hit)
    enriched_hit["source_label"] = chr(64 + rank)
    enriched_hit["title"] = passage_metadata[corpus_id]["title"]
    enriched_hit["paragraph_id"] = passage_metadata[corpus_id]["paragraph_id"]
    enriched_hit["text"] = passages[corpus_id]

    return enriched_hit


def fusion_format_context_from_hits(hits):
    # Format selected fusion hits as RAG context

    context_blocks = []

    for hit in hits:
        source_label = hit.get("source_label", "?")
        title = hit.get("title")
        paragraph_id = hit.get("paragraph_id")
        passage_text = hit.get("text", "")

        context_blocks.append(
            f"Context Source {source_label}\n"
            f"Title: {title} | Paragraph: {paragraph_id} | Fusion score: {float(hit['score']):.4f}\n"
            f"Semantic rank: {hit.get('semantic_rank')} | BM25 rank: {hit.get('bm25_rank')}\n"
            f"{passage_text}"
        )

    return "\n\n".join(context_blocks)


# Backward-compatible alias
def format_fusion_context_from_hits(hits):
    # Accept both old hits and enriched hits
    enriched_hits = []

    for rank, hit in enumerate(hits, start=1):
        if "title" in hit and "text" in hit:
            enriched_hits.append(hit)
        else:
            enriched_hits.append(fusion_add_metadata_to_hit(hit, rank))

    return fusion_format_context_from_hits(enriched_hits)

In [ ]:
# Selection/debug helpers

def fusion_select_hits_from_relevance(
    candidate_hits,
    relevant_source_labels,
    parsing_failed=False,
    max_final_sources=3,
):
    # If relevance parsing fails, keep first hits
    if parsing_failed:
        return candidate_hits[:max_final_sources]

    # If relevant sources exist, keep only those
    if relevant_source_labels:
        return [
            hit for hit in candidate_hits
            if hit.get("source_label") in relevant_source_labels
        ][:max_final_sources]

    # If no source is relevant, use no context
    return []


def fusion_compact_hits_for_debug(hits):
    # Compact readable hit representation for debugger

    compact = []

    for hit in hits or []:
        corpus_id = int(hit["corpus_id"])

        compact.append({
            "source_label": hit.get("source_label", "?"),
            "corpus_id": corpus_id,
            "title": hit.get("title", passage_metadata[corpus_id]["title"]),
            "paragraph_id": hit.get("paragraph_id", passage_metadata[corpus_id]["paragraph_id"]),
            "score": float(hit.get("score", 0.0)),

            "semantic_rank": hit.get("semantic_rank"),
            "semantic_score": hit.get("semantic_score"),
            "bm25_rank": hit.get("bm25_rank"),
            "bm25_score": hit.get("bm25_score"),

            "text": hit.get("text", passages[corpus_id]),
        })

    return compact


def fusion_build_chat_prompt(system_msg, user_msg, tokenizer):
    # Convert system/user messages to model chat format

    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )


def fusion_build_final_answer_prompt(question, context):
    # Use RAG prompt if context exists, otherwise fallback to base LLM prompt

    if context.strip():
        system_msg, user_msg = build_rag_prompt(
            question.text,
            question.options,
            context,
        )

        answer_parser = extract_option_id
        max_new_tokens = BASE_RAG_CONFIG.get("rag_max_new_tokens", 64)
        prompt_mode = "rag_with_fusion_context"

    else:
        system_msg, user_msg = build_prompt(
            question.text,
            question.options,
        )

        answer_parser = parse_answer
        max_new_tokens = CONFIG.get("max_new_tokens", 15)
        prompt_mode = "llm_fallback_no_context"

        if CONFIG.get("prompt_strategy") in {"zero_shot_cot", "cot"}:
            max_new_tokens = CONFIG.get("reasoning_max_new_tokens", 256)

    return system_msg, user_msg, answer_parser, max_new_tokens, prompt_mode


# Backward-compatible aliases
def select_hits_from_relevance_safe(candidate_hits, relevant_source_labels, parsing_failed=False, max_final_sources=3):
    return fusion_select_hits_from_relevance(
        candidate_hits,
        relevant_source_labels,
        parsing_failed=parsing_failed,
        max_final_sources=max_final_sources,
    )


def compact_fusion_hits_for_debug(hits):
    return fusion_compact_hits_for_debug(hits)

In [ ]:
# Fusion retriever

def fusion_run_retrieval(question, top_k=None):
    # Run semantic search, BM25 search, and optional RRF fusion

    if top_k is None:
        top_k = BASE_RAG_CONFIG.get("top_k_context", 8)

    use_semantic = BASE_RAG_CONFIG.get("fusion_enable_semantic", True)
    use_bm25 = BASE_RAG_CONFIG.get("fusion_enable_bm25", True)

    if not use_semantic and not use_bm25:
        raise ValueError(
            "At least one retrieval method must be enabled: "
            "fusion_enable_semantic or fusion_enable_bm25."
        )

    semantic_query, bm25_query = fusion_get_queries(question)

    semantic_results = []
    bm25_results = []

    if use_semantic:
        semantic_results = fusion_semantic_faiss_search(
            semantic_query,
            top_k=BASE_RAG_CONFIG.get("fusion_semantic_top_k", 20),
        )

    if use_bm25:
        bm25_results = fusion_bm25_search(
            bm25_query,
            top_k=BASE_RAG_CONFIG.get("fusion_bm25_top_k", 20),
        )

    if use_semantic and use_bm25:
        raw_hits = fusion_reciprocal_rank_fusion(
            semantic_results,
            bm25_results,
            top_k=top_k,
            rrf_k=BASE_RAG_CONFIG.get("fusion_rrf_k", 60),
        )

    elif use_semantic:
        raw_hits = fusion_prepare_single_method_hits(
            semantic_results,
            method_name="semantic",
            top_k=top_k,
        )

    else:
        raw_hits = fusion_prepare_single_method_hits(
            bm25_results,
            method_name="bm25",
            top_k=top_k,
        )

    hits = [
        fusion_add_metadata_to_hit(hit, rank)
        for rank, hit in enumerate(raw_hits, start=1)
    ]

    retrieval_details = {
        "fusion_semantic_query": semantic_query,
        "fusion_bm25_query": bm25_query,
        "fusion_semantic_results": semantic_results,
        "fusion_bm25_results": bm25_results,
        "fusion_enable_semantic": use_semantic,
        "fusion_enable_bm25": use_bm25,

        # Compatible with generic query printer
        "query_items": [
            {
                "query_type": "semantic_query",
                "query": semantic_query,
            },
            {
                "query_type": "bm25_query",
                "query": bm25_query,
            },
        ],
    }

    return hits, retrieval_details


def retrieve_local_wikipedia_context_fusion(question, top_k=None):
    # Public retriever used by non-logged code paths

    hits, _ = fusion_run_retrieval(question, top_k=top_k)
    context = fusion_format_context_from_hits(hits)

    return context, hits

In [ ]:
# Main answer function

def get_fusion_rag_answer_logged(question, tokenizer, model, max_retries=3):
    start_total = time.perf_counter()
    valid_ids = [opt.id for opt in question.options]

    fusion_use_rse = BASE_RAG_CONFIG.get("fusion_use_rse", False)
    top_k_context = BASE_RAG_CONFIG.get("top_k_context", 8)
    final_top_k_context = BASE_RAG_CONFIG.get("final_top_k_context", 3)

    # 1. Retrieval

    retrieval_start = time.perf_counter()

    candidate_hits, retrieval_details = fusion_run_retrieval(
        question,
        top_k=top_k_context,
    )

    retrieval_time_sec = time.perf_counter() - retrieval_start

    # --------------------------------------------------------
    # 2. Optional relevance filtering
    # --------------------------------------------------------

    relevance_start = time.perf_counter()

    if BASE_RAG_CONFIG.get("enable_llm_relevance_check", True):
        relevant_source_labels, relevance_parsing_failed = check_all_sources_relevance(
            question,
            candidate_hits,
            tokenizer,
            model,
        )

    else:
        max_initial_hits = (
            BASE_RAG_CONFIG.get("rse_max_seed_hits", final_top_k_context)
            if fusion_use_rse
            else final_top_k_context
        )

        relevant_source_labels = [
            hit.get("source_label")
            for hit in candidate_hits[:max_initial_hits]
            if hit.get("source_label")
        ]

        relevance_parsing_failed = False

    relevance_time_sec = time.perf_counter() - relevance_start

    max_selected_hits = (
        BASE_RAG_CONFIG.get("rse_max_seed_hits", final_top_k_context)
        if fusion_use_rse
        else final_top_k_context
    )

    selected_hits = fusion_select_hits_from_relevance(
        candidate_hits,
        relevant_source_labels,
        parsing_failed=relevance_parsing_failed,
        max_final_sources=max_selected_hits,
    )

    # --------------------------------------------------------
    # 3. Optional RSE enrichment
    # --------------------------------------------------------

    context_start = time.perf_counter()

    rse_segments = []

    if fusion_use_rse:
        rse_segments = rse_build_segments_from_hits(selected_hits)
        context = rse_format_context(rse_segments)
        context_status = f"fusion_rse_segments_{len(rse_segments)}"

    else:
        context = fusion_format_context_from_hits(selected_hits)
        context_status = f"fusion_context_{len(selected_hits)}_sources"

    context_time_sec = time.perf_counter() - context_start

    # --------------------------------------------------------
    # 4. Build answer prompt
    # --------------------------------------------------------

    system_msg, user_msg, answer_parser, max_new_tokens, prompt_mode = (
        fusion_build_final_answer_prompt(question, context)
    )

    prompt = fusion_build_chat_prompt(
        system_msg,
        user_msg,
        tokenizer,
    )

    # --------------------------------------------------------
    # 5. Generate answer with retries
    # --------------------------------------------------------

    generation_start = time.perf_counter()

    final_answer = None
    attempts = []
    last_output = ""

    for attempt_idx in range(1, max_retries + 1):
        output = generate_answer(
            prompt,
            tokenizer,
            model,
            do_sample=CONFIG.get("do_sample", False),
            max_new_tokens=max_new_tokens,
            temperature=CONFIG.get("temperature", 1.0),
        )

        last_output = output
        parsed_answer = answer_parser(output, valid_ids)

        attempts.append({
            "attempt": attempt_idx,
            "raw_output": output,
            "parsed_answer": parsed_answer,
        })

        if parsed_answer is not None:
            final_answer = parsed_answer
            break

    generation_time_sec = time.perf_counter() - generation_start

    # --------------------------------------------------------
    # 6. Fallback if parsing failed
    # --------------------------------------------------------

    parsing_failed = False
    fallback_used = False
    fallback_reason = None

    if final_answer is None:
        final_answer = valid_ids[0]
        parsing_failed = True
        fallback_used = True
        fallback_reason = "default_first_valid_option"

    total_answer_time_sec = time.perf_counter() - start_total

    # --------------------------------------------------------
    # 7. Metadata for Advanced Debugger
    # --------------------------------------------------------

    details = {
        "active_rag_variant": "fusion_rag",
        "rag_type": "fusion_local_rag",

        "prompt_mode": prompt_mode,
        "context_status": context_status,
        "selection_mode": "semantic_bm25_fusion_relevance_selection",

        "fusion_enable_semantic": retrieval_details["fusion_enable_semantic"],
        "fusion_enable_bm25": retrieval_details["fusion_enable_bm25"],
        "fusion_semantic_query": retrieval_details["fusion_semantic_query"],
        "fusion_bm25_query": retrieval_details["fusion_bm25_query"],
        "query_items": retrieval_details["query_items"],

        "relevant_source_labels": relevant_source_labels,
        "relevance_parsing_failed": relevance_parsing_failed,

        "hits": fusion_compact_hits_for_debug(candidate_hits),
        "candidate_hits": fusion_compact_hits_for_debug(candidate_hits),
        "selected_hits": fusion_compact_hits_for_debug(selected_hits),

        "rse_segments": rse_segments,
        "context": context,
        "context_used": bool(context.strip()),

        "attempts": attempts,
        "raw_output": last_output,

        "answer_id": final_answer,
        "parsing_failed": parsing_failed,
        "fallback_used": fallback_used,
        "fallback_reason": fallback_reason,

        "retrieval_time_sec": retrieval_time_sec,
        "relevance_time_sec": relevance_time_sec,
        "context_time_sec": context_time_sec,
        "generation_time_sec": generation_time_sec,
        "total_answer_time_sec": total_answer_time_sec,
    }

    return final_answer, details

In [ ]:
# Shared dispatcher

def get_fusion_rag_answer(question, tokenizer, model, max_retries=3):
    answer_id, _ = get_fusion_rag_answer_logged(
        question,
        tokenizer,
        model,
        max_retries=max_retries,
    )
    return answer_id

In [ ]:
# Activation

def activate_fusion_rag_variant(
    top_k_context=8,
    final_top_k_context=3,
    fusion_enable_semantic=True,
    fusion_enable_bm25=True,
    fusion_semantic_top_k=20,
    fusion_bm25_top_k=20,
    fusion_rrf_k=60,
    fusion_semantic_query_mode="question_options",
    fusion_bm25_query_mode="question_only",
    fusion_use_rse=False,
    rse_neighbor_window=1,
    rse_max_segment_length=3,
    rse_max_total_paragraphs=3,
    enable_llm_relevance_check=True,
):
    global retrieve_local_wikipedia_context
    global get_rag_answer_logged

    if not fusion_enable_semantic and not fusion_enable_bm25:
        raise ValueError(
            "At least one retrieval method must be enabled: "
            "fusion_enable_semantic or fusion_enable_bm25."
        )

    retrieve_local_wikipedia_context = retrieve_local_wikipedia_context_fusion
    get_rag_answer_logged = get_fusion_rag_answer_logged

    BASE_RAG_CONFIG.update({
        "active_rag_variant": "fusion_rag",

        "top_k_context": top_k_context,
        "final_top_k_context": final_top_k_context,

        "fusion_enable_semantic": fusion_enable_semantic,
        "fusion_enable_bm25": fusion_enable_bm25,

        "fusion_semantic_top_k": fusion_semantic_top_k,
        "fusion_bm25_top_k": fusion_bm25_top_k,
        "fusion_rrf_k": fusion_rrf_k,

        "fusion_semantic_query_mode": fusion_semantic_query_mode,
        "fusion_bm25_query_mode": fusion_bm25_query_mode,

        "fusion_use_rse": fusion_use_rse,

        "rse_neighbor_window": rse_neighbor_window,
        "rse_max_segment_length": rse_max_segment_length,
        "rse_max_total_paragraphs": rse_max_total_paragraphs,
        "rse_min_segment_score": 0.15,
        "rse_max_seed_hits": 5,

        "enable_llm_relevance_check": enable_llm_relevance_check,
        "relevance_max_new_tokens": 16,

        "rag_max_new_tokens": 64,
    })

    print("Fusion Retrieval RAG activated.")
    print(f"Semantic retrieval: {fusion_enable_semantic}")
    print(f"BM25 retrieval: {fusion_enable_bm25}")
    print(f"Semantic query mode: {fusion_semantic_query_mode}")
    print(f"BM25 query mode: {fusion_bm25_query_mode}")
    print(f"RSE enabled: {fusion_use_rse}")
    print(f"LLM relevance filtering: {enable_llm_relevance_check}")


print("BM25 + Semantic Fusion RAG cell loaded successfully.")

BM25 + Semantic Fusion RAG cell loaded successfully.


## 🟢 Cross-Encoder Reranking


In [ ]:
import time
from sentence_transformers import CrossEncoder

In [ ]:
# Cross-encoder setup

cross_encoder_reranker = None
cross_encoder_reranker_model_id = None


def cross_encoder_get_model_id(model_id=None):
    # Default reranker model
    if model_id is not None:
        return model_id

    return BASE_RAG_CONFIG.get(
        "rerank_model_id",
        "cross-encoder/ms-marco-MiniLM-L-6-v2",
    )


def load_cross_encoder_reranker(model_id=None):
    # Load the local cross-encoder only when needed
    global cross_encoder_reranker
    global cross_encoder_reranker_model_id

    model_id = cross_encoder_get_model_id(model_id)

    if (
        cross_encoder_reranker is None
        or cross_encoder_reranker_model_id != model_id
    ):
        print(f"Loading local cross-encoder reranker: {model_id}")
        cross_encoder_reranker = CrossEncoder(model_id)
        cross_encoder_reranker_model_id = model_id

    return cross_encoder_reranker


In [ ]:
# Helpers

def cross_encoder_get_config():
    # Read config values from BASE_RAG_CONFIG
    return {
        "top_k_context": BASE_RAG_CONFIG.get("top_k_context", 12),
        "final_top_k_context": BASE_RAG_CONFIG.get("final_top_k_context", 3),

        "use_cross_encoder_rerank": BASE_RAG_CONFIG.get("use_cross_encoder_rerank", True),
        "rerank_query_mode": BASE_RAG_CONFIG.get("rerank_query_mode", "question_options"),
        "rerank_batch_size": BASE_RAG_CONFIG.get("rerank_batch_size", 16),

        "use_rse": BASE_RAG_CONFIG.get("fusion_use_rse", False),
        "enable_llm_relevance_check": BASE_RAG_CONFIG.get("enable_llm_relevance_check", False),

        "rse_max_seed_hits": BASE_RAG_CONFIG.get("rse_max_seed_hits", 5),
        "rag_max_new_tokens": BASE_RAG_CONFIG.get("rag_max_new_tokens", 64),
    }


def cross_encoder_select_hits_from_relevance(
    candidate_hits,
    relevant_source_labels,
    parsing_failed=False,
    max_final_sources=3,
):
    # If relevance parsing fails, keep first hits
    if parsing_failed:
        return candidate_hits[:max_final_sources]

    # If relevant sources exist, keep only those
    if relevant_source_labels:
        return [
            hit for hit in candidate_hits
            if hit.get("source_label") in relevant_source_labels
        ][:max_final_sources]

    # If no source is relevant, use no context
    return []


def cross_encoder_compact_hits_for_debug(hits):
    # Compact readable hit representation for debugger
    compact = []

    for hit in hits or []:
        corpus_id = int(hit["corpus_id"])

        compact.append({
            "source_label": hit.get("source_label", "?"),
            "corpus_id": corpus_id,
            "title": hit.get("title", passage_metadata[corpus_id]["title"]),
            "paragraph_id": hit.get("paragraph_id", passage_metadata[corpus_id]["paragraph_id"]),
            "score": float(hit.get("score", 0.0)),

            "semantic_rank": hit.get("semantic_rank"),
            "semantic_score": hit.get("semantic_score"),
            "bm25_rank": hit.get("bm25_rank"),
            "bm25_score": hit.get("bm25_score"),

            "rerank_score": hit.get("rerank_score"),
            "text": hit.get("text", passages[corpus_id]),
        })

    return compact


def cross_encoder_build_chat_prompt(system_msg, user_msg, tokenizer):
    # Convert system/user messages to model chat format
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )


def cross_encoder_build_final_answer_prompt(question, context):
    # Use RAG prompt if context exists, otherwise fallback to base LLM prompt
    if context.strip():
        system_msg, user_msg = build_rag_prompt(
            question.text,
            question.options,
            context,
        )

        answer_parser = extract_option_id
        max_new_tokens = BASE_RAG_CONFIG.get("rag_max_new_tokens", 64)
        prompt_mode = "rag_with_cross_encoder_context"

    else:
        system_msg, user_msg = build_prompt(
            question.text,
            question.options,
        )

        answer_parser = parse_answer
        max_new_tokens = CONFIG.get("max_new_tokens", 15)
        prompt_mode = "llm_fallback_no_context"

        if CONFIG.get("prompt_strategy") in {"zero_shot_cot", "cot"}:
            max_new_tokens = CONFIG.get("reasoning_max_new_tokens", 256)

    return system_msg, user_msg, answer_parser, max_new_tokens, prompt_mode


# Backward-compatible aliases
def select_hits_from_relevance_safe(
    candidate_hits,
    relevant_source_labels,
    parsing_failed=False,
    max_final_sources=3,
):
    return cross_encoder_select_hits_from_relevance(
        candidate_hits,
        relevant_source_labels,
        parsing_failed=parsing_failed,
        max_final_sources=max_final_sources,
    )


def compact_cross_encoder_hits_for_debug(hits):
    return cross_encoder_compact_hits_for_debug(hits)

In [ ]:
# Reranking

def cross_encoder_build_rerank_query(question, mode=None):
    # Build query used by the cross-encoder
    if mode is None:
        mode = BASE_RAG_CONFIG.get("rerank_query_mode", "question_options")

    return build_fusion_query(question, mode=mode)


def cross_encoder_make_document_text(hit):
    # Convert one passage into the text paired with the query
    corpus_id = int(hit["corpus_id"])

    title = hit.get("title", passage_metadata[corpus_id]["title"])
    paragraph_id = hit.get("paragraph_id", passage_metadata[corpus_id]["paragraph_id"])
    passage_text = hit.get("text", passages[corpus_id])

    return f"Title: {title}\nParagraph: {paragraph_id}\n{passage_text}"


def cross_encoder_rerank_hits(question, hits, top_k=3):
    # Rerank candidate hits using a local cross-encoder
    if not hits:
        return [], {
            "rerank_query": None,
            "rerank_scores": [],
        }

    reranker = load_cross_encoder_reranker()
    rerank_query = cross_encoder_build_rerank_query(question)

    pairs = [
        [rerank_query, cross_encoder_make_document_text(hit)]
        for hit in hits
    ]

    scores = reranker.predict(
        pairs,
        batch_size=BASE_RAG_CONFIG.get("rerank_batch_size", 16),
        show_progress_bar=False,
    )

    scored_hits = []

    for hit, score in zip(hits, scores):
        item = dict(hit)
        item["rerank_score"] = float(score)
        scored_hits.append(item)

    scored_hits.sort(
        key=lambda x: x["rerank_score"],
        reverse=True,
    )

    rerank_details = {
        "rerank_query": rerank_query,
        "rerank_scores": [
            {
                "source_label": hit.get("source_label"),
                "corpus_id": int(hit["corpus_id"]),
                "rerank_score": float(hit.get("rerank_score", 0.0)),
            }
            for hit in scored_hits
        ],
    }

    return scored_hits[:top_k], rerank_details


# Backward-compatible aliases
def build_rerank_query(question, mode=None):
    return cross_encoder_build_rerank_query(question, mode=mode)


def make_cross_encoder_document_text(hit):
    return cross_encoder_make_document_text(hit)


def rerank_hits_with_cross_encoder(question, hits, top_k=3):
    selected_hits, _ = cross_encoder_rerank_hits(
        question,
        hits,
        top_k=top_k,
    )
    return selected_hits

In [ ]:
# Retrieval + selection

def cross_encoder_retrieve_candidates(question, top_k_context):
    # Uses Fusion retriever as first-stage retrieval
    _, candidate_hits = retrieve_local_wikipedia_context_fusion(
        question,
        top_k=top_k_context,
    )

    return candidate_hits


def cross_encoder_select_final_hits(question, candidate_hits, tokenizer, model):
    # Select final hits using cross-encoder, LLM relevance, or top retrieved hits

    config = cross_encoder_get_config()

    use_cross_encoder_rerank = config["use_cross_encoder_rerank"]
    enable_llm_relevance_check = config["enable_llm_relevance_check"]
    use_rse = config["use_rse"]

    max_selected_hits = (
        config["rse_max_seed_hits"]
        if use_rse
        else config["final_top_k_context"]
    )

    selection_start = time.perf_counter()

    rerank_time_sec = 0.0
    relevance_time_sec = 0.0
    relevance_parsing_failed = False
    relevant_source_labels = None
    rerank_details = {
        "rerank_query": None,
        "rerank_scores": [],
    }

    if use_cross_encoder_rerank:
        rerank_start = time.perf_counter()

        selected_hits, rerank_details = cross_encoder_rerank_hits(
            question,
            candidate_hits,
            top_k=max_selected_hits,
        )

        rerank_time_sec = time.perf_counter() - rerank_start
        selection_mode = "cross_encoder_rerank"

    elif enable_llm_relevance_check:
        relevance_start = time.perf_counter()

        relevant_source_labels, relevance_parsing_failed = check_all_sources_relevance(
            question,
            candidate_hits,
            tokenizer,
            model,
        )

        relevance_time_sec = time.perf_counter() - relevance_start

        selected_hits = cross_encoder_select_hits_from_relevance(
            candidate_hits,
            relevant_source_labels,
            parsing_failed=relevance_parsing_failed,
            max_final_sources=max_selected_hits,
        )

        selection_mode = "safe_relevance_selection"

    else:
        selected_hits = candidate_hits[:max_selected_hits]
        selection_mode = "top_retrieved_hits"

    selection_time_sec = time.perf_counter() - selection_start

    selection_details = {
        "selection_mode": selection_mode,
        "relevant_source_labels": relevant_source_labels,
        "relevance_parsing_failed": relevance_parsing_failed,
        "rerank_query": rerank_details.get("rerank_query"),
        "rerank_scores": rerank_details.get("rerank_scores", []),
        "selection_time_sec": selection_time_sec,
        "relevance_time_sec": relevance_time_sec,
        "rerank_time_sec": rerank_time_sec,
    }

    return selected_hits, selection_details

In [ ]:
# Context construction

def cross_encoder_build_context(selected_hits):
    # Build either normal context or RSE-enriched context

    use_rse = BASE_RAG_CONFIG.get("fusion_use_rse", False)

    context_start = time.perf_counter()
    rse_segments = []

    if use_rse:
        rse_segments = rse_build_segments_from_hits(selected_hits)
        context = rse_format_context(rse_segments)
        context_status = f"fusion_cross_encoder_rse_segments_{len(rse_segments)}"

    else:
        context = format_fusion_context_from_hits(selected_hits)
        context_status = f"fusion_cross_encoder_context_{len(selected_hits)}_sources"

    context_time_sec = time.perf_counter() - context_start

    return context, rse_segments, {
        "context_status": context_status,
        "context_time_sec": context_time_sec,
    }

In [ ]:

# ============================================================
# Main answer function
# ============================================================

def get_fusion_rerank_rag_answer_logged(question, tokenizer, model, max_retries=3):
    start_total = time.perf_counter()
    valid_ids = [opt.id for opt in question.options]

    config = cross_encoder_get_config()

    # --------------------------------------------------------
    # 1. First-stage retrieval
    # --------------------------------------------------------

    retrieval_start = time.perf_counter()

    candidate_hits = cross_encoder_retrieve_candidates(
        question,
        top_k_context=config["top_k_context"],
    )

    retrieval_time_sec = time.perf_counter() - retrieval_start

    # --------------------------------------------------------
    # 2. Final hit selection
    # --------------------------------------------------------

    selected_hits, selection_details = cross_encoder_select_final_hits(
        question,
        candidate_hits,
        tokenizer,
        model,
    )

    # --------------------------------------------------------
    # 3. Context construction
    # --------------------------------------------------------

    context, rse_segments, context_details = cross_encoder_build_context(
        selected_hits
    )

    # --------------------------------------------------------
    # 4. Build answer prompt
    # --------------------------------------------------------

    system_msg, user_msg, answer_parser, max_new_tokens, prompt_mode = (
        cross_encoder_build_final_answer_prompt(question, context)
    )

    prompt = cross_encoder_build_chat_prompt(
        system_msg,
        user_msg,
        tokenizer,
    )

    # --------------------------------------------------------
    # 5. Generate answer with retries
    # --------------------------------------------------------

    generation_start = time.perf_counter()

    final_answer = None
    attempts = []
    last_output = ""

    for attempt_idx in range(1, max_retries + 1):
        output = generate_answer(
            prompt,
            tokenizer,
            model,
            do_sample=CONFIG.get("do_sample", False),
            max_new_tokens=max_new_tokens,
            temperature=CONFIG.get("temperature", 1.0),
        )

        last_output = output
        parsed_answer = answer_parser(output, valid_ids)

        attempts.append({
            "attempt": attempt_idx,
            "raw_output": output,
            "parsed_answer": parsed_answer,
        })

        if parsed_answer is not None:
            final_answer = parsed_answer
            break

    generation_time_sec = time.perf_counter() - generation_start

    # --------------------------------------------------------
    # 6. Fallback if parsing failed
    # --------------------------------------------------------

    parsing_failed = False
    fallback_used = False
    fallback_reason = None

    if final_answer is None:
        final_answer = valid_ids[0]
        parsing_failed = True
        fallback_used = True
        fallback_reason = "default_first_valid_option"

    total_answer_time_sec = time.perf_counter() - start_total

    # --------------------------------------------------------
    # 7. Metadata for Advanced Debugger
    # --------------------------------------------------------

    details = {
        "active_rag_variant": "fusion_cross_encoder_rag",
        "rag_type": "fusion_cross_encoder_local_rag",

        "prompt_mode": prompt_mode,
        "context_status": context_details["context_status"],
        "selection_mode": selection_details["selection_mode"],

        "relevant_source_labels": selection_details["relevant_source_labels"],
        "relevance_parsing_failed": selection_details["relevance_parsing_failed"],

        "rerank_query": selection_details["rerank_query"],
        "rerank_scores": selection_details["rerank_scores"],

        "hits": cross_encoder_compact_hits_for_debug(candidate_hits),
        "candidate_hits": cross_encoder_compact_hits_for_debug(candidate_hits),
        "selected_hits": cross_encoder_compact_hits_for_debug(selected_hits),

        "rse_segments": rse_segments,
        "context": context,
        "context_used": bool(context.strip()),

        "attempts": attempts,
        "raw_output": last_output,

        "answer_id": final_answer,
        "parsing_failed": parsing_failed,
        "fallback_used": fallback_used,
        "fallback_reason": fallback_reason,

        "retrieval_time_sec": retrieval_time_sec,
        "selection_time_sec": selection_details["selection_time_sec"],
        "relevance_time_sec": selection_details["relevance_time_sec"],
        "rerank_time_sec": selection_details["rerank_time_sec"],
        "context_time_sec": context_details["context_time_sec"],
        "generation_time_sec": generation_time_sec,
        "total_answer_time_sec": total_answer_time_sec,
    }

    return final_answer, details

In [ ]:
# ============================================================
# Shared dispatcher
# ============================================================

def get_fusion_rerank_rag_answer(question, tokenizer, model, max_retries=3):
    answer_id, _ = get_fusion_rerank_rag_answer_logged(
        question,
        tokenizer,
        model,
        max_retries=max_retries,
    )
    return answer_id

In [ ]:
# ============================================================
# Activation
# ============================================================

def activate_fusion_rerank_rag_variant(
    top_k_context=12,
    final_top_k_context=3,
    use_semantic_retrieval=True,
    use_bm25_retrieval=True,
    fusion_semantic_top_k=30,
    fusion_bm25_top_k=30,
    fusion_rrf_k=60,
    fusion_semantic_query_mode="question_options",
    fusion_bm25_query_mode="question_only",
    use_cross_encoder_rerank=True,
    rerank_model_id="cross-encoder/ms-marco-MiniLM-L-6-v2",
    rerank_query_mode="question_options",
    rerank_batch_size=16,
    load_reranker_now=True,
    use_rse=False,
    rse_neighbor_window=1,
    rse_max_segment_length=3,
    rse_max_total_paragraphs=5,
    rse_max_seed_hits=5,
    enable_llm_relevance_check=False,
):
    global retrieve_local_wikipedia_context
    global get_rag_answer_logged

    if not use_semantic_retrieval and not use_bm25_retrieval:
        raise ValueError(
            "Enable at least one retrieval technology: semantic FAISS or BM25."
        )

    if use_cross_encoder_rerank and enable_llm_relevance_check:
        print(
            "Both cross-encoder reranking and LLM relevance filtering are enabled. "
            "Cross-encoder will be used for final selection."
        )

    retrieve_local_wikipedia_context = retrieve_local_wikipedia_context_fusion
    get_rag_answer_logged = get_fusion_rerank_rag_answer_logged

    BASE_RAG_CONFIG.update({
        "active_rag_variant": "fusion_cross_encoder_rag",

        "top_k_context": top_k_context,
        "final_top_k_context": final_top_k_context,

        "fusion_enable_semantic": use_semantic_retrieval,
        "fusion_enable_bm25": use_bm25_retrieval,

        "fusion_semantic_top_k": fusion_semantic_top_k,
        "fusion_bm25_top_k": fusion_bm25_top_k,
        "fusion_rrf_k": fusion_rrf_k,

        "fusion_semantic_query_mode": fusion_semantic_query_mode,
        "fusion_bm25_query_mode": fusion_bm25_query_mode,

        "use_cross_encoder_rerank": use_cross_encoder_rerank,
        "rerank_model_id": rerank_model_id,
        "rerank_query_mode": rerank_query_mode,
        "rerank_batch_size": rerank_batch_size,

        "fusion_use_rse": use_rse,
        "use_rse": use_rse,

        "rse_neighbor_window": rse_neighbor_window,
        "rse_max_segment_length": rse_max_segment_length,
        "rse_max_total_paragraphs": rse_max_total_paragraphs,
        "rse_min_segment_score": 0.15,
        "rse_max_seed_hits": rse_max_seed_hits,

        "enable_llm_relevance_check": enable_llm_relevance_check,
        "relevance_max_new_tokens": 16,

        "rag_max_new_tokens": 64,
    })

    if use_cross_encoder_rerank and load_reranker_now:
        load_cross_encoder_reranker(rerank_model_id)

    print("Fusion + Cross-Encoder RAG activated.")
    print(f"Semantic retrieval: {use_semantic_retrieval}")
    print(f"BM25 retrieval: {use_bm25_retrieval}")
    print(f"Cross-Encoder rerank: {use_cross_encoder_rerank}")
    print(f"Rerank model: {rerank_model_id}")
    print(f"Rerank query mode: {rerank_query_mode}")
    print(f"RSE enabled: {use_rse}")
    print(f"LLM relevance filtering: {enable_llm_relevance_check}")

# 🟡 Final evaluation


In [ ]:
# Load best Advanced Retrieval configs for Qwen 3B and Qwen 7B

import os
import json
import glob
import hashlib
import pandas as pd
from IPython.display import display

# Paths

BENCHMARK_DIR = "/content/gdrive/MyDrive/NLP/PoliMillionaire/ASSO_RAG/Benchmark"

SOLVED_BENCHMARK_PATH = os.path.join(
    BENCHMARK_DIR,
    "llm_zero_shot_wrong_questions_solved.json"
)

ADVANCED_RETRIEVAL_CACHE_ROOT = os.path.join(
    BENCHMARK_DIR,
    "benchmark_cache",
    "advanced_retrieval"
)

TARGET_MODEL_IDS = [
    "Qwen/Qwen2.5-3B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
]

# Helpers

def md5_file(path):
    h = hashlib.md5()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


def find_advanced_retrieval_summaries_for_models(target_model_ids):
    benchmark_hash = md5_file(SOLVED_BENCHMARK_PATH)

    search_pattern = os.path.join(
        ADVANCED_RETRIEVAL_CACHE_ROOT,
        f"benchmark_{benchmark_hash}",
        "model_*",
        "summary",
        "advanced_retrieval_benchmark_summary.json",
    )

    candidate_paths = glob.glob(search_pattern)

    matches = []

    for path in candidate_paths:
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)

            metadata = data.get("metadata", {})
            model_metadata = metadata.get("model_metadata", {})
            model_id = model_metadata.get("model_id")

            if model_id in target_model_ids:
                matches.append({
                    "model_id": model_id,
                    "path": path,
                    "data": data,
                    "modified_time": os.path.getmtime(path),
                    "model_metadata": model_metadata,
                })

        except Exception as e:
            print(f"Could not read summary: {path}")
            print(e)

    return matches, benchmark_hash, search_pattern


def get_best_configs_from_summary(summary_data, model_id, summary_path):
    summaries = summary_data.get("summaries", [])
    rows = []

    for s in summaries:
        total_questions = int(s.get("total_questions", 0))
        correct = int(s.get("correct", 0))
        accuracy = float(s.get("accuracy", 0.0))
        avg_time = float(s.get("avg_time_sec", 0.0))

        rows.append({
            "Model": model_id,
            "Configuration": s.get("pretty_name", s.get("config_name")),
            "Config name": s.get("config_name"),
            "Technology": s.get("technology"),
            "Method": s.get("method"),
            "Correct": f"{correct}/{total_questions}",
            "Accuracy": f"{accuracy * 100:.0f}%",
            "Avg. time": f"{avg_time:.2f}s",
            "Summary path": summary_path,
            "_accuracy_sort": accuracy,
            "_time_sort": avg_time,
        })

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    df = df.sort_values(
        by=["_accuracy_sort", "_time_sort"],
        ascending=[False, True],
    ).reset_index(drop=True)

    df["Rank"] = df.index + 1

    df = df[
        [
            "Rank",
            "Model",
            "Configuration",
            "Config name",
            "Technology",
            "Method",
            "Correct",
            "Accuracy",
            "Avg. time",
        ]
    ]

    return df

# Load summaries

matches, benchmark_hash, search_pattern = find_advanced_retrieval_summaries_for_models(
    TARGET_MODEL_IDS
)

if not matches:
    print("No cached Advanced Retrieval summaries found for target models.")
    print()
    print("Target models:")
    for model_id in TARGET_MODEL_IDS:
        print(f"- {model_id}")
    print()
    print("Benchmark hash:")
    print(benchmark_hash)
    print()
    print("Searched here:")
    print(search_pattern)

else:
    # If multiple summaries exist for the same model, keep the most recently modified one.
    latest_by_model = {}

    for match in matches:
        model_id = match["model_id"]

        if (
            model_id not in latest_by_model
            or match["modified_time"] > latest_by_model[model_id]["modified_time"]
        ):
            latest_by_model[model_id] = match

    all_best_tables = []

    print("Advanced Retrieval best configurations")
    print(f"Benchmark hash: {benchmark_hash}")
    print()

    for model_id in TARGET_MODEL_IDS:
        if model_id not in latest_by_model:
            print(f"No cached summary found for: {model_id}")
            print()
            continue

        match = latest_by_model[model_id]
        data = match["data"]
        metadata = data.get("metadata", {})
        summaries = data.get("summaries", [])

        best_df = get_best_configs_from_summary(
            summary_data=data,
            model_id=model_id,
            summary_path=match["path"],
        )

        print("=" * 100)
        print(f"Model: {model_id}")
        print(f"Benchmark questions: {metadata.get('num_questions')}")
        print(f"Configs loaded: {len(summaries)}")
        print(f"Summary JSON: {match['path']}")
        print()

        display(best_df.drop(columns=["Model"]))

        if not best_df.empty:
            all_best_tables.append(best_df)

    # Compact comparison: only best config per model

    if all_best_tables:
        combined_df = pd.concat(all_best_tables, ignore_index=True)

        best_per_model = (
            combined_df
            .sort_values(
                by=["Model", "Rank"],
                ascending=[True, True],
            )
            .groupby("Model", as_index=False)
            .first()
        )

        best_per_model = best_per_model[
            [
                "Model",
                "Configuration",
                "Config name",
                "Technology",
                "Method",
                "Correct",
                "Accuracy",
                "Avg. time",
            ]
        ]

        print("=" * 100)
        print("Best Advanced Retrieval config per model")
        print()

        display(best_per_model)

Advanced Retrieval best configurations
Benchmark hash: 208f033deb4f6166cd10ce46fc8eb4c2

Model: Qwen/Qwen2.5-3B-Instruct
Benchmark questions: 20
Configs loaded: 9
Summary JSON: /content/gdrive/MyDrive/NLP/PoliMillionaire/ASSO_RAG/Benchmark/benchmark_cache/advanced_retrieval/benchmark_208f033deb4f6166cd10ce46fc8eb4c2/model_dc7c68033a18f81696395420280d55a4/summary/advanced_retrieval_benchmark_summary.json



,Rank,Configuration,Config name,Technology,Method,Correct,Accuracy,Avg. time
0,1,"Fusion Retrieval, semantic q+opts + BM25 q-onl...",fusion_sem_qopts_bm25_qonly_top8,fusion_retrieval,fusion,16/20,80%,4.27s
1,2,"Fusion + RSE, top-8, total paragraphs 5",fusion_rse_top8_total5,fusion_rse_rag,fusion_rse,16/20,80%,4.58s
2,3,"BM25 only, question-only, top-6",bm25_only_qonly_top6,bm25_retrieval,bm25_only,15/20,75%,3.93s
3,4,"Fusion Retrieval, semantic q-only + BM25 q-onl...",fusion_sem_qonly_bm25_qonly_top8,fusion_retrieval,fusion,15/20,75%,4.21s
4,5,"Fusion + Cross-Encoder + RSE, top-12, total pa...",fusion_cross_encoder_rse_top12_total5,fusion_cross_encoder_rse_rag,fusion_cross_encoder_rse,15/20,75%,4.83s
5,6,"Fusion + Cross-Encoder, top-20 to top-3",fusion_cross_encoder_top20,fusion_cross_encoder_rag,fusion_cross_encoder,14/20,70%,4.28s
6,7,"Fusion + Cross-Encoder, top-12 to top-3",fusion_cross_encoder_top12,fusion_cross_encoder_rag,fusion_cross_encoder,14/20,70%,4.51s
7,8,"Fusion Retrieval, semantic q+opts + BM25 q+opt...",fusion_sem_qopts_bm25_qopts_top8,fusion_retrieval,fusion,14/20,70%,5.89s
8,9,"Semantic FAISS only, question + options, top-6",semantic_only_qopts_top6,semantic_retrieval,semantic_only,13/20,65%,1.52s


Model: Qwen/Qwen2.5-7B-Instruct
Benchmark questions: 20
Configs loaded: 9
Summary JSON: /content/gdrive/MyDrive/NLP/PoliMillionaire/ASSO_RAG/Benchmark/benchmark_cache/advanced_retrieval/benchmark_208f033deb4f6166cd10ce46fc8eb4c2/model_03a7ab1c6ed4d695f771ca1dc34a8927/summary/advanced_retrieval_benchmark_summary.json



,Rank,Configuration,Config name,Technology,Method,Correct,Accuracy,Avg. time
0,1,"Fusion + Cross-Encoder + RSE, top-12, total pa...",fusion_cross_encoder_rse_top12_total5,fusion_cross_encoder_rse_rag,fusion_cross_encoder_rse,16/20,80%,3.62s
1,2,"Fusion + RSE, top-8, total paragraphs 5",fusion_rse_top8_total5,fusion_rse_rag,fusion_rse,15/20,75%,3.64s
2,3,"Fusion + Cross-Encoder, top-20 to top-3",fusion_cross_encoder_top20,fusion_cross_encoder_rag,fusion_cross_encoder,15/20,75%,4.55s
3,4,"Fusion Retrieval, semantic q+opts + BM25 q+opt...",fusion_sem_qopts_bm25_qopts_top8,fusion_retrieval,fusion,15/20,75%,5.00s
4,5,"Semantic FAISS only, question + options, top-6",semantic_only_qopts_top6,semantic_retrieval,semantic_only,14/20,70%,0.99s
5,6,"Fusion Retrieval, semantic q+opts + BM25 q-onl...",fusion_sem_qopts_bm25_qonly_top8,fusion_retrieval,fusion,14/20,70%,3.53s
6,7,"Fusion + Cross-Encoder, top-12 to top-3",fusion_cross_encoder_top12,fusion_cross_encoder_rag,fusion_cross_encoder,14/20,70%,3.89s
7,8,"BM25 only, question-only, top-6",bm25_only_qonly_top6,bm25_retrieval,bm25_only,13/20,65%,3.36s
8,9,"Fusion Retrieval, semantic q-only + BM25 q-onl...",fusion_sem_qonly_bm25_qonly_top8,fusion_retrieval,fusion,13/20,65%,3.50s


Best Advanced Retrieval config per model



,Model,Configuration,Config name,Technology,Method,Correct,Accuracy,Avg. time
0,Qwen/Qwen2.5-3B-Instruct,"Fusion Retrieval, semantic q+opts + BM25 q-onl...",fusion_sem_qopts_bm25_qonly_top8,fusion_retrieval,fusion,16/20,80%,4.27s
1,Qwen/Qwen2.5-7B-Instruct,"Fusion + Cross-Encoder + RSE, top-12, total pa...",fusion_cross_encoder_rse_top12_total5,fusion_cross_encoder_rse_rag,fusion_cross_encoder_rse,16/20,80%,3.62s


In [ ]:
# ------------------------------------------------------------
# Load summaries
# ------------------------------------------------------------

matches, benchmark_hash, search_pattern = find_advanced_retrieval_summaries_for_models(
    TARGET_MODEL_IDS
)

if not matches:
    print("No cached Advanced Retrieval summaries found for target models.")
    print()
    print("Target models:")
    for model_id in TARGET_MODEL_IDS:
        print(f"- {model_id}")
    print()
    print("Benchmark hash:")
    print(benchmark_hash)
    print()
    print("Searched here:")
    print(search_pattern)

else:
    # If multiple summaries exist for the same model, keep the most recently modified one.
    latest_by_model = {}

    for match in matches:
        model_id = match["model_id"]

        if (
            model_id not in latest_by_model
            or match["modified_time"] > latest_by_model[model_id]["modified_time"]
        ):
            latest_by_model[model_id] = match

    all_best_tables = []

    print("Advanced Retrieval best configurations")
    print(f"Benchmark hash: {benchmark_hash}")
    print()

    for model_id in TARGET_MODEL_IDS:
        if model_id not in latest_by_model:
            print(f"No cached summary found for: {model_id}")
            print()
            continue

        match = latest_by_model[model_id]
        data = match["data"]
        metadata = data.get("metadata", {})
        summaries = data.get("summaries", [])

        best_df = get_best_configs_from_summary(
            summary_data=data,
            model_id=model_id,
            summary_path=match["path"],
        )

        print("=" * 100)
        print(f"Model: {model_id}")
        print(f"Benchmark questions: {metadata.get('num_questions')}")
        print(f"Configs loaded: {len(summaries)}")
        print(f"Summary JSON: {match['path']}")
        print()

        display(best_df.drop(columns=["Model"]))

        if not best_df.empty:
            all_best_tables.append(best_df)

    # --------------------------------------------------------
    # Compact comparison: only best config per model
    # --------------------------------------------------------

    if all_best_tables:
        combined_df = pd.concat(all_best_tables, ignore_index=True)

        best_per_model = (
            combined_df
            .sort_values(
                by=["Model", "Rank"],
                ascending=[True, True],
            )
            .groupby("Model", as_index=False)
            .first()
        )

        best_per_model = best_per_model[
            [
                "Model",
                "Configuration",
                "Config name",
                "Technology",
                "Method",
                "Correct",
                "Accuracy",
                "Avg. time",
            ]
        ]

        print("=" * 100)
        print("Best Advanced Retrieval config per model")
        print()

        display(best_per_model)

Advanced Retrieval best configurations
Benchmark hash: 208f033deb4f6166cd10ce46fc8eb4c2

Model: Qwen/Qwen2.5-3B-Instruct
Benchmark questions: 20
Configs loaded: 9
Summary JSON: /content/gdrive/MyDrive/NLP/PoliMillionaire/ASSO_RAG/Benchmark/benchmark_cache/advanced_retrieval/benchmark_208f033deb4f6166cd10ce46fc8eb4c2/model_dc7c68033a18f81696395420280d55a4/summary/advanced_retrieval_benchmark_summary.json



,Rank,Configuration,Config name,Technology,Method,Correct,Accuracy,Avg. time
0,1,"Fusion Retrieval, semantic q+opts + BM25 q-onl...",fusion_sem_qopts_bm25_qonly_top8,fusion_retrieval,fusion,16/20,80%,4.27s
1,2,"Fusion + RSE, top-8, total paragraphs 5",fusion_rse_top8_total5,fusion_rse_rag,fusion_rse,16/20,80%,4.58s
2,3,"BM25 only, question-only, top-6",bm25_only_qonly_top6,bm25_retrieval,bm25_only,15/20,75%,3.93s
3,4,"Fusion Retrieval, semantic q-only + BM25 q-onl...",fusion_sem_qonly_bm25_qonly_top8,fusion_retrieval,fusion,15/20,75%,4.21s
4,5,"Fusion + Cross-Encoder + RSE, top-12, total pa...",fusion_cross_encoder_rse_top12_total5,fusion_cross_encoder_rse_rag,fusion_cross_encoder_rse,15/20,75%,4.83s
5,6,"Fusion + Cross-Encoder, top-20 to top-3",fusion_cross_encoder_top20,fusion_cross_encoder_rag,fusion_cross_encoder,14/20,70%,4.28s
6,7,"Fusion + Cross-Encoder, top-12 to top-3",fusion_cross_encoder_top12,fusion_cross_encoder_rag,fusion_cross_encoder,14/20,70%,4.51s
7,8,"Fusion Retrieval, semantic q+opts + BM25 q+opt...",fusion_sem_qopts_bm25_qopts_top8,fusion_retrieval,fusion,14/20,70%,5.89s
8,9,"Semantic FAISS only, question + options, top-6",semantic_only_qopts_top6,semantic_retrieval,semantic_only,13/20,65%,1.52s


Model: Qwen/Qwen2.5-7B-Instruct
Benchmark questions: 20
Configs loaded: 9
Summary JSON: /content/gdrive/MyDrive/NLP/PoliMillionaire/ASSO_RAG/Benchmark/benchmark_cache/advanced_retrieval/benchmark_208f033deb4f6166cd10ce46fc8eb4c2/model_03a7ab1c6ed4d695f771ca1dc34a8927/summary/advanced_retrieval_benchmark_summary.json



,Rank,Configuration,Config name,Technology,Method,Correct,Accuracy,Avg. time
0,1,"Fusion + Cross-Encoder + RSE, top-12, total pa...",fusion_cross_encoder_rse_top12_total5,fusion_cross_encoder_rse_rag,fusion_cross_encoder_rse,16/20,80%,3.62s
1,2,"Fusion + RSE, top-8, total paragraphs 5",fusion_rse_top8_total5,fusion_rse_rag,fusion_rse,15/20,75%,3.64s
2,3,"Fusion + Cross-Encoder, top-20 to top-3",fusion_cross_encoder_top20,fusion_cross_encoder_rag,fusion_cross_encoder,15/20,75%,4.55s
3,4,"Fusion Retrieval, semantic q+opts + BM25 q+opt...",fusion_sem_qopts_bm25_qopts_top8,fusion_retrieval,fusion,15/20,75%,5.00s
4,5,"Semantic FAISS only, question + options, top-6",semantic_only_qopts_top6,semantic_retrieval,semantic_only,14/20,70%,0.99s
5,6,"Fusion Retrieval, semantic q+opts + BM25 q-onl...",fusion_sem_qopts_bm25_qonly_top8,fusion_retrieval,fusion,14/20,70%,3.53s
6,7,"Fusion + Cross-Encoder, top-12 to top-3",fusion_cross_encoder_top12,fusion_cross_encoder_rag,fusion_cross_encoder,14/20,70%,3.89s
7,8,"BM25 only, question-only, top-6",bm25_only_qonly_top6,bm25_retrieval,bm25_only,13/20,65%,3.36s
8,9,"Fusion Retrieval, semantic q-only + BM25 q-onl...",fusion_sem_qonly_bm25_qonly_top8,fusion_retrieval,fusion,13/20,65%,3.50s


Best Advanced Retrieval config per model



,Model,Configuration,Config name,Technology,Method,Correct,Accuracy,Avg. time
0,Qwen/Qwen2.5-3B-Instruct,"Fusion Retrieval, semantic q+opts + BM25 q-onl...",fusion_sem_qopts_bm25_qonly_top8,fusion_retrieval,fusion,16/20,80%,4.27s
1,Qwen/Qwen2.5-7B-Instruct,"Fusion + Cross-Encoder + RSE, top-12, total pa...",fusion_cross_encoder_rse_top12_total5,fusion_cross_encoder_rse_rag,fusion_cross_encoder_rse,16/20,80%,3.62s


**Query Transformation**  
- **Pro:** it reaches good accuracy, so rewriting the query can help retrieval when the original question is not well suited for search.  
- **Con:** it is often slow and not always stable. Sometimes the transformed query adds noise or changes the focus of the question.  
- **Conclusion:** useful in some cases, but not one of the most robust techniques.

**HyDE**  
- **Pro:** it works better than standard Query Transformation because it generates a richer hypothetical document and improves the match with relevant passages.  
- **Con:** it requires an extra generation step, so it increases latency.  
- **Conclusion:** effective for improving semantic retrieval, but more expensive than direct retrieval.

**Fusion Retrieval**  
- **Pro:** it is one of the most solid techniques because it combines semantic retrieval and BM25. Semantic retrieval captures the meaning of the question, while BM25 keeps exact keyword matching.  
- **Con:** it is slightly more complex than using a single retrieval method.  
- **Conclusion:** one of the best trade-offs between accuracy, robustness, and time.

**Cross-Encoder Reranking**  
- **Pro:** in theory, it should improve the order of the retrieved passages.  
- **Con:** in our results, it does not always improve over simple Fusion Retrieval and it adds complexity to the pipeline.  
- **Conclusion:** not always necessary.

**RSE**  
- **Pro:** it can be useful when the answer depends on the context around the retrieved passage.  
- **Con:** it often adds unnecessary text and does not always improve accuracy.  
- **Conclusion:** useful only in some cases; increasing the amount of context does not automatically improve the answer.

**Contextual Compression**  
- **Pro:** it helps reduce noise in the retrieved context by keeping only the most relevant parts. In several cases, it gives good results.  
- **Con:** it adds an extra step and may remove useful information if the compression is too aggressive.  
- **Conclusion:** a useful technique, especially when retrieval produces long or noisy context.

**Reliable-RAG / Relevance Filtering**  
- **Pro:** it makes the system more robust because it checks whether the retrieved context is actually useful before using it to answer.  
- **Con:** if the relevance check uses the LLM, it increases latency.  
- **Conclusion:** stable and interpretable, with a good balance between quality and cost.

**Overall conclusion**  
The most convincing techniques are **Fusion Retrieval**, **HyDE**, **Contextual Compression**, and **Reliable-RAG**. Query Transformation can work, but it is less stable. RSE and Cross-Encoder are more situational: they can help, but they do not give a consistent improvement over simpler techniques.

# 🔵 Advanced Architecture


## 🟢 Adaptive RAG

In [ ]:
import os
import glob
import json
import re
import time
from copy import deepcopy

import pandas as pd
from IPython.display import display

In [ ]:
# Static configuration

ADAPTIVE_ALLOWED_TECHNOLOGIES = {
    "llm",
    "rag_baseline_question_only",
    "rag_question_options",
    "reliable_rag",
    "query_transformation_rag",
    "hyde_rag",
    "rse_rag",
    "contextual_compression_rag",
    "rse_contextual_compression_rag",
    "fusion_rag",
    "fusion_cross_encoder_rag",
}

ADAPTIVE_EXCLUDED_TECHNOLOGIES = {
    "self_rag",
    "crag",
}

ADAPTIVE_PROFILE_TO_CANDIDATE_TECHNOLOGIES = {
    "calculation": [
        "llm",
        "rag_baseline_question_only",
        "rag_question_options",
    ],
    "exact_entity": [
        "fusion_cross_encoder_rag",
        "fusion_rag",
        "rag_question_options",
        "reliable_rag",
    ],
    "definition": [
        "contextual_compression_rag",
        "reliable_rag",
        "rag_question_options",
        "rse_contextual_compression_rag",
    ],
    "comparison": [
        "fusion_cross_encoder_rag",
        "rse_contextual_compression_rag",
        "query_transformation_rag",
        "contextual_compression_rag",
    ],
    "causal_broad": [
        "rse_contextual_compression_rag",
        "rse_rag",
        "contextual_compression_rag",
        "query_transformation_rag",
    ],
    "ambiguous_or_broad": [
        "hyde_rag",
        "query_transformation_rag",
        "fusion_cross_encoder_rag",
        "contextual_compression_rag",
    ],
    "balanced": [
        "contextual_compression_rag",
        "fusion_cross_encoder_rag",
        "reliable_rag",
        "fusion_rag",
    ],
}

ADAPTIVE_ALLOWED_PROFILES = set(ADAPTIVE_PROFILE_TO_CANDIDATE_TECHNOLOGIES.keys())

_ADAPTIVE_BASE_LOAD_BEST_CONFIGS_FOR_CURRENT_MODEL = globals().get(
    "load_best_configs_for_current_model"
)

BASE_RAG_CONFIG.update({
    "adaptive_enabled": True,
    "adaptive_default_profile": "balanced",
    "adaptive_use_best_config_cache": True,
    "adaptive_debug": True,
})

In [ ]:
# Classifier prompts

CLASSIFIER_SYSTEM_PROMPT = (
    "You are a retrieval strategy classifier for a multiple-choice quiz RAG system.\n"
    "Choose exactly one profile from this list:\n"
    "- calculation: numeric or formula-based science/math question.\n"
    "- exact_entity: asks for a specific person, place, event, date, object, title, or named fact.\n"
    "- definition: asks what term/concept/principle best describes something.\n"
    "- comparison: asks how two concepts differ, relate, compare, or contrast.\n"
    "- causal_broad: asks why something happened, what caused it, what resulted, or what happens in response.\n"
    "- ambiguous_or_broad: vague, underspecified, or broad question with weak retrieval clues.\n"
    "- balanced: none of the above.\n\n"
    "Return only the profile name. Do not explain."
)


def adaptive_build_classifier_user_prompt(question_text, options_text):
    return (
        f"Question:\n{question_text}\n\n"
        f"Options:\n{options_text}\n\n"
        "Profile:"
    )


# Backward-compatible alias
def _build_classifier_user_prompt(question_text: str, options_text: str) -> str:
    return adaptive_build_classifier_user_prompt(question_text, options_text)

In [ ]:
# Generic helpers

def adaptive_get_question_text(question):
    return str(getattr(question, "text", question)).strip()


def adaptive_get_question_options_text(question):
    return " ".join(
        str(getattr(opt, "text", opt)).strip()
        for opt in (getattr(question, "options", []) or [])
    )


def adaptive_make_fake_question(question_text, options=None):
    fake_options = []

    if options:
        fake_options = [
            type("Option", (), {"id": idx, "text": text})()
            for idx, text in enumerate(options)
        ]

    return type(
        "Question",
        (),
        {
            "text": question_text,
            "options": fake_options,
        },
    )()


def adaptive_restore_global(name, value):
    if value is None:
        globals().pop(name, None)
    else:
        globals()[name] = value


def adaptive_apply_decision_to_details(details, decision):
    details = details or {}

    details["adaptive_profile"] = decision.get("adaptive_profile")
    details["adaptive_reason"] = decision.get("adaptive_reason")
    details["adaptive_candidate_technologies"] = decision.get("candidate_technologies")
    details["adaptive_selected_technology"] = decision.get("selected_technology")
    details["adaptive_selected_config_name"] = decision.get("selected_config_name")
    details["adaptive_selected_pretty_name"] = decision.get("selected_pretty_name")
    details["adaptive_selection_accuracy"] = decision.get("selection_accuracy")
    details["adaptive_selection_avg_time_sec"] = decision.get("selection_avg_time_sec")
    details["adaptive_source_summary_path"] = decision.get("source_summary_path")
    details["adaptive_actually_activated_config"] = decision.get("actually_activated_config")
    details["adaptive_actually_activated_source"] = decision.get("actually_activated_source")

    profile = decision.get("adaptive_profile")
    selected = decision.get("selected_technology")

    details["prompt_mode"] = f"adaptive_{profile}_{selected}_{details.get('prompt_mode')}"
    details["context_status"] = f"adaptive_{profile}_{details.get('context_status')}"

    return details


# Backward-compatible aliases
def _get_question_text(question) -> str:
    return adaptive_get_question_text(question)


def _get_question_options_text(question) -> str:
    return adaptive_get_question_options_text(question)


def _make_fake_question(question_text: str, options=None):
    return adaptive_make_fake_question(question_text, options)


def _annotate_details_with_decision(details: dict, decision: dict) -> dict:
    return adaptive_apply_decision_to_details(details, decision)


def _restore_global(name: str, value):
    return adaptive_restore_global(name, value)


In [ ]:
# Technology normalization and config records

def adaptive_normalize_technology(row):
    if "get_family_from_summary_row" in globals():
        mapped = globals()["get_family_from_summary_row"](row)
        if mapped:
            return mapped

    technology = str(row.get("technology", "") or "").lower()
    method = str(row.get("method", "") or "").lower()
    config_name = str(row.get("config_name", "") or "").lower()

    text = " ".join([technology, method, config_name])

    for name in ADAPTIVE_ALLOWED_TECHNOLOGIES | ADAPTIVE_EXCLUDED_TECHNOLOGIES:
        if name in text:
            return name

    if technology == "baseline_rag":
        if "qonly" in config_name or "question_only" in config_name:
            return "rag_baseline_question_only"
        return "rag_question_options"

    if technology == "fusion_retrieval":
        return "fusion_rag"

    if technology in {"semantic_retrieval", "bm25_retrieval"}:
        return technology

    if "cross_encoder" in text:
        return "fusion_cross_encoder_rag"

    return technology or method or config_name


def adaptive_build_config_record(technology, row):
    known_keys = {
        "technology",
        "config_name",
        "pretty_name",
        "accuracy",
        "avg_time_sec",
        "model_id",
        "quantization_bits",
        "source_summary_path",
        "summary_path",
    }

    return {
        "technology": technology,
        "config_name": row.get("config_name"),
        "pretty_name": row.get("pretty_name", row.get("config_name")),
        "accuracy": pd.to_numeric(row.get("accuracy"), errors="coerce"),
        "avg_time_sec": pd.to_numeric(row.get("avg_time_sec"), errors="coerce"),
        "model_id": row.get("model_id", CONFIG.get("model_id")),
        "quantization_bits": row.get(
            "quantization_bits",
            CONFIG.get("quantization_bits"),
        ),
        "source_summary_path": row.get("source_summary_path") or row.get("summary_path"),
        **{k: v for k, v in row.items() if k not in known_keys},
    }


def adaptive_filter_and_rank_configs(records):
    empty_cols = [
        "technology",
        "config_name",
        "pretty_name",
        "accuracy",
        "avg_time_sec",
        "model_id",
        "quantization_bits",
        "source_summary_path",
    ]

    if not records:
        return pd.DataFrame(columns=empty_cols)

    df = pd.DataFrame(records)

    df = df[
        df["technology"].isin(ADAPTIVE_ALLOWED_TECHNOLOGIES)
        & ~df["technology"].isin(ADAPTIVE_EXCLUDED_TECHNOLOGIES)
    ].copy()

    if df.empty:
        return df

    if "model_id" in df.columns and CONFIG.get("model_id") is not None:
        df = df[
            df["model_id"].isna()
            | (df["model_id"] == CONFIG.get("model_id"))
        ].copy()

    if "quantization_bits" in df.columns:
        df = df[
            df["quantization_bits"].isna()
            | (
                df["quantization_bits"].astype(str)
                == str(CONFIG.get("quantization_bits"))
            )
        ].copy()

    if df.empty:
        return df

    df["accuracy"] = pd.to_numeric(df["accuracy"], errors="coerce")
    df["avg_time_sec"] = pd.to_numeric(df["avg_time_sec"], errors="coerce")

    df["_acc"] = df["accuracy"].fillna(-1.0)
    df["_time"] = df["avg_time_sec"].fillna(999_999.0)

    df = (
        df.sort_values(
            by=["technology", "_acc", "_time"],
            ascending=[True, False, True],
        )
        .groupby("technology", as_index=False)
        .first()
        .drop(columns=["_acc", "_time"], errors="ignore")
    )

    return df.sort_values(
        by=["accuracy", "avg_time_sec"],
        ascending=[False, True],
        na_position="last",
    ).reset_index(drop=True)


# Backward-compatible aliases
def _normalize_technology(row: dict) -> str:
    return adaptive_normalize_technology(row)


def _build_config_record(technology: str, row: dict) -> dict:
    return adaptive_build_config_record(technology, row)


def _filter_and_rank_configs(records: list) -> pd.DataFrame:
    return adaptive_filter_and_rank_configs(records)

In [ ]:
# Best-config loading

def adaptive_load_records_from_summary_files():
    benchmark_dir = globals().get(
        "BENCHMARK_DIR",
        "/content/gdrive/MyDrive/NLP/PoliMillionaire/ASSO_RAG/Benchmark",
    )

    cache_root = globals().get(
        "CACHE_ROOT",
        os.path.join(benchmark_dir, "benchmark_cache"),
    )

    summary_paths = glob.glob(
        os.path.join(cache_root, "**", "*summary*.json"),
        recursive=True,
    )

    skip_patterns = [
        "news_web_crag_eval",
        "wiki20231101_first3_best_config_eval",
        "wikipedia_comparison",
        "crashed",
    ]

    records = []

    for summary_path in summary_paths:
        if any(pattern in summary_path.lower() for pattern in skip_patterns):
            continue

        try:
            with open(summary_path, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception:
            continue

        metadata = data.get("metadata", {}) or {}
        model_metadata = data.get("model_metadata") or metadata.get("model_metadata") or {}

        model_id = metadata.get("model_id") or model_metadata.get("model_id")
        quant_bits = metadata.get("quantization_bits") or model_metadata.get("quantization_bits")

        for item in data.get("summaries", []) or []:
            row = dict(item)

            row.setdefault("model_id", model_id or CONFIG.get("model_id"))
            row.setdefault(
                "quantization_bits",
                quant_bits if quant_bits is not None else CONFIG.get("quantization_bits"),
            )

            row["source_summary_path"] = summary_path

            technology = adaptive_normalize_technology(row)
            records.append(
                adaptive_build_config_record(
                    technology,
                    row,
                )
            )

    return records


def adaptive_load_llm_baseline_record():
    paths_to_try = []

    if "GLOBAL_RUNS_INDEX_PATH" in globals():
        paths_to_try.append(str(globals()["GLOBAL_RUNS_INDEX_PATH"]))

    benchmark_dir = globals().get(
        "BENCHMARK_DIR",
        "/content/gdrive/MyDrive/NLP/PoliMillionaire/ASSO_RAG/Benchmark",
    )

    paths_to_try.extend(
        glob.glob(
            os.path.join(benchmark_dir, "**", "*runs*.jsonl"),
            recursive=True,
        )
    )

    placeholder = {
        "technology": "llm",
        "config_name": "llm",
        "pretty_name": "LLM-only baseline",
        "accuracy": pd.NA,
        "avg_time_sec": pd.NA,
        "model_id": CONFIG.get("model_id"),
        "quantization_bits": CONFIG.get("quantization_bits"),
        "source_summary_path": None,
    }

    rows = []

    for path in paths_to_try:
        if not os.path.exists(path):
            continue

        try:
            with open(path, "r", encoding="utf-8") as f:
                for line in f:
                    if not line.strip():
                        continue

                    obj = json.loads(line)

                    if (
                        obj.get("technique_name") == "base_llm"
                        and obj.get("model_id") == CONFIG.get("model_id")
                        and obj.get("quantization_bits") == CONFIG.get("quantization_bits")
                    ):
                        rows.append(obj)

        except Exception:
            continue

    if not rows:
        return placeholder

    df = pd.DataFrame(rows)

    df["accuracy"] = pd.to_numeric(
        df.get("accuracy"),
        errors="coerce",
    )

    df["avg_response_time_sec"] = pd.to_numeric(
        df.get("avg_response_time_sec"),
        errors="coerce",
    )

    group_cols = [
        c
        for c in [
            "benchmark_prompt_strategy",
            "prompt_strategy",
            "n_shots",
            "config_hash",
        ]
        if c in df.columns
    ]

    if group_cols:
        summary = (
            df.groupby(group_cols, dropna=False)
            .agg(
                accuracy=("accuracy", "mean"),
                avg_time_sec=("avg_response_time_sec", "mean"),
                runs=("accuracy", "count"),
            )
            .reset_index()
            .sort_values(["accuracy", "avg_time_sec"], ascending=[False, True])
        )

        best = summary.iloc[0]
        config_name = str(
            best.get(
                "benchmark_prompt_strategy",
                best.get("prompt_strategy", "llm"),
            )
        )

        return {
            **placeholder,
            "config_name": config_name,
            "pretty_name": f"LLM-only baseline | {config_name}",
            "accuracy": best["accuracy"],
            "avg_time_sec": best["avg_time_sec"],
            "source_summary_path": paths_to_try[0] if paths_to_try else None,
        }

    return {
        **placeholder,
        "accuracy": df["accuracy"].mean(),
        "avg_time_sec": df["avg_response_time_sec"].mean(),
        "source_summary_path": paths_to_try[0] if paths_to_try else None,
    }


def adaptive_load_records_via_existing_selector(verbose=False):
    base = globals().get("_ADAPTIVE_BASE_LOAD_BEST_CONFIGS_FOR_CURRENT_MODEL")

    if base is None:
        return []

    best_by_key = base(verbose=verbose, display_table=False)

    if not best_by_key:
        return []

    return [
        adaptive_build_config_record(tech, dict(row))
        for tech, row in best_by_key.items()
        if row is not None
    ]


def load_adaptive_best_configs_for_current_model(
    verbose=True,
    display_table=True,
    as_dataframe=True,
):
    records = []

    base_loader = globals().get("load_best_configs_for_current_model")

    if (
        base_loader is not None
        and base_loader is not load_adaptive_best_configs_for_current_model
    ):
        try:
            result = base_loader(
                verbose=verbose,
                display_table=display_table,
                as_dataframe=True,
            )

            if isinstance(result, pd.DataFrame):
                records = result.to_dict(orient="records")

        except TypeError:
            try:
                result = base_loader(
                    verbose=verbose,
                    display_table=display_table,
                )

                if isinstance(result, dict):
                    records = [
                        adaptive_build_config_record(tech, dict(row))
                        for tech, row in result.items()
                        if row is not None
                    ]

            except Exception:
                pass

        except Exception:
            pass

    if not records:
        records = adaptive_load_records_via_existing_selector(verbose=False)

    if not records:
        records = adaptive_load_records_from_summary_files()

    records.append(adaptive_load_llm_baseline_record())

    df = adaptive_filter_and_rank_configs(records)

    if display_table:
        print("Adaptive best configurations for current model")
        display(df)

    if as_dataframe:
        return df

    return {
        row["technology"]: row
        for row in df.to_dict(orient="records")
        if row.get("technology") != "llm"
    }


# Backward-compatible aliases
def _load_records_from_summary_files() -> list:
    return adaptive_load_records_from_summary_files()


def _load_llm_baseline_record() -> dict:
    return adaptive_load_llm_baseline_record()


def _load_records_via_existing_selector(verbose=False) -> list:
    return adaptive_load_records_via_existing_selector(verbose=verbose)

In [ ]:
# Question classification

def adaptive_classify_with_llm(question_text, options_text, tokenizer, model):
    messages = [
        {
            "role": "system",
            "content": CLASSIFIER_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": adaptive_build_classifier_user_prompt(
                question_text,
                options_text,
            ),
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    raw_output = generate_answer(
        prompt,
        tokenizer,
        model,
        do_sample=False,
        max_new_tokens=8,
        temperature=0.0,
    )

    parsed = raw_output.strip().lower().replace("`", "").replace(".", "")
    parsed = parsed.split()[0] if parsed else ""

    if parsed in ADAPTIVE_ALLOWED_PROFILES:
        return parsed, f"local LLM classified the question as '{parsed}'"

    return None, f"local LLM produced invalid profile: {raw_output!r}"


def adaptive_classify_with_heuristics(question_text, options_text):
    question_lower = question_text.lower()
    combined_text = f"{question_text} {options_text}".lower()
    question_tokens = re.findall(r"\w+", question_lower)

    ambiguous_patterns = [
        "which of the following",
        "what can be said",
        "what is true",
        "best answer",
        "most likely",
        "generally",
        "in general",
    ]

    if len(question_tokens) < 6 or any(pattern in question_lower for pattern in ambiguous_patterns):
        return "ambiguous_or_broad", "heuristic: broad or underspecified wording"

    has_number = bool(re.search(r"\d", combined_text))

    calculation_terms = {
        "calculate",
        "mass",
        "speed",
        "velocity",
        "momentum",
        "force",
        "energy",
        "distance",
        "time",
        "rate",
        "density",
        "volume",
        "area",
        "acceleration",
        "kg",
        "m/s",
        "meters per second",
        "newton",
        "joule",
        "watt",
    }

    if has_number and any(term in combined_text for term in calculation_terms):
        return "calculation", "heuristic: numeric/science wording suggests formula reasoning"

    comparison_patterns = [
        "difference",
        "differ",
        "compare",
        "relationship",
        "relate",
        "how does",
        "in terms of",
        "similar to",
        "unlike",
        "contrast",
        "between",
    ]

    if any(pattern in question_lower for pattern in comparison_patterns):
        return "comparison", "heuristic: comparison or relationship question"

    causal_patterns = [
        "why",
        "reason",
        "cause",
        "caused",
        "led to",
        "result",
        "in response",
        "what happens",
    ]

    if any(pattern in question_lower for pattern in causal_patterns):
        return "causal_broad", "heuristic: causal question"

    exact_entity_patterns = [
        "who was",
        "who is",
        "when did",
        "where did",
        "where is",
        "at which location",
        "which event",
        "which period",
        "which country",
        "which city",
        "which object",
        "what is the name",
        "what are the names",
        "primary language",
    ]

    if any(pattern in question_lower for pattern in exact_entity_patterns):
        return "exact_entity", "heuristic: specific named fact question"

    definition_patterns = [
        "what term",
        "which term",
        "refers to",
        "is called",
        "best describes",
        "describes the",
        "core principle",
        "fundamental principle",
        "primary principle",
    ]

    if any(pattern in question_lower for pattern in definition_patterns):
        return "definition", "heuristic: definition-style question"

    default_profile = BASE_RAG_CONFIG.get("adaptive_default_profile", "balanced")

    return default_profile, f"heuristic: default '{default_profile}' profile"


def classify_question_for_adaptive_rag(question, tokenizer=None, model=None):
    question_text = adaptive_get_question_text(question)
    options_text = adaptive_get_question_options_text(question)

    if tokenizer is not None and model is not None:
        profile, reason = adaptive_classify_with_llm(
            question_text,
            options_text,
            tokenizer,
            model,
        )

        if profile is not None:
            return profile, reason

    return adaptive_classify_with_heuristics(question_text, options_text)


In [ ]:
# Technology selection

def select_best_technology_for_profile(profile, best_configs_df):
    candidate_overrides = BASE_RAG_CONFIG.get(
        "adaptive_candidate_technology_overrides",
        {},
    )

    candidate_map = {
        **ADAPTIVE_PROFILE_TO_CANDIDATE_TECHNOLOGIES,
        **candidate_overrides,
    }

    default_profile = BASE_RAG_CONFIG.get(
        "adaptive_default_profile",
        "balanced",
    )

    candidates = candidate_map.get(
        profile,
        candidate_map.get(default_profile, []),
    )

    candidates = [
        technology
        for technology in candidates
        if (
            technology in ADAPTIVE_ALLOWED_TECHNOLOGIES
            and technology not in ADAPTIVE_EXCLUDED_TECHNOLOGIES
        )
    ]

    df = best_configs_df.copy()

    if df.empty:
        df = pd.DataFrame([adaptive_load_llm_baseline_record()])

    pool = df[df["technology"].isin(candidates)].copy()

    if pool.empty:
        pool = df[
            df["technology"].isin(ADAPTIVE_ALLOWED_TECHNOLOGIES)
            & ~df["technology"].isin(ADAPTIVE_EXCLUDED_TECHNOLOGIES)
        ].copy()

    if pool.empty:
        raise ValueError("No adaptive RAG technology is available for selection.")

    pool["_acc"] = pd.to_numeric(
        pool["accuracy"],
        errors="coerce",
    ).fillna(-1.0)

    pool["_time"] = pd.to_numeric(
        pool["avg_time_sec"],
        errors="coerce",
    ).fillna(999_999.0)

    selected = (
        pool.sort_values(
            by=["_acc", "_time"],
            ascending=[False, True],
        )
        .iloc[0]
        .to_dict()
    )

    selected.pop("_acc", None)
    selected.pop("_time", None)

    selected["candidate_technologies"] = candidates

    return selected


def build_adaptive_rag_decision(question):
    profile_name, reason = classify_question_for_adaptive_rag(
        question,
        tokenizer=globals().get("llm_tokenizer", globals().get("tokenizer")),
        model=globals().get("llm_model", globals().get("model")),
    )

    if profile_name not in ADAPTIVE_PROFILE_TO_CANDIDATE_TECHNOLOGIES:
        profile_name = BASE_RAG_CONFIG.get(
            "adaptive_default_profile",
            "balanced",
        )

        reason = f"classifier selected unknown profile; falling back to '{profile_name}'"

    best_configs_df = load_adaptive_best_configs_for_current_model(
        verbose=False,
        display_table=False,
        as_dataframe=True,
    )

    selected = select_best_technology_for_profile(
        profile_name,
        best_configs_df,
    )

    return {
        "adaptive_profile": profile_name,
        "adaptive_reason": reason,
        "candidate_technologies": selected.get("candidate_technologies", []),
        "selected_technology": selected.get("technology"),
        "selected_config_name": selected.get("config_name"),
        "selected_pretty_name": selected.get("pretty_name"),
        "selection_accuracy": selected.get("accuracy"),
        "selection_avg_time_sec": selected.get("avg_time_sec"),
        "source_summary_path": selected.get("source_summary_path"),
    }


In [ ]:
# Adaptive dispatch

def adaptive_answer_with_llm(question, tokenizer, model, max_retries=3):
    start_time = time.time()

    if "get_llm_answer_logged" in globals():
        info = globals()["get_llm_answer_logged"](
            question,
            tokenizer,
            model,
            max_retries=max_retries,
        )

        if isinstance(info, tuple):
            answer_id, details = info
        else:
            answer_id = info.get("answer_id")
            details = dict(info)

    else:
        answer_id = globals()["get_llm_answer"](
            question,
            tokenizer,
            model,
            max_retries=max_retries,
        )

        details = {
            "answer_id": answer_id,
            "logging_note": "LLM-only answer without get_llm_answer_logged metadata.",
        }

    details["active_rag_variant"] = "llm"
    details.setdefault("prompt_mode", "llm_only")
    details.setdefault("context_status", "no_retrieval")
    details["total_answer_time_sec"] = time.time() - start_time

    return answer_id, details


def adaptive_answer_with_baseline_rag(question, tokenizer, model, max_retries=3):
    answer_id = globals()["get_rag_answer"](
        question,
        tokenizer,
        model,
        max_retries=max_retries,
    )

    details = {
        "answer_id": answer_id,
        "active_rag_variant": BASE_RAG_CONFIG.get("active_rag_variant"),
        "context_status": "retrieved_context",
        "prompt_mode": "baseline_rag",
        "logging_note": "Baseline RAG does not expose get_rag_answer_logged details.",
    }

    return answer_id, details


def adaptive_answer_with_selected_rag(question, tokenizer, model, max_retries=3):
    return globals()["get_rag_answer_logged"](
        question,
        tokenizer,
        model,
        max_retries=max_retries,
    )


def get_adaptive_rag_answer_logged(question, tokenizer, model, max_retries=3):
    decision = build_adaptive_rag_decision(question)
    selected_technology = decision["selected_technology"]

    # LLM-only path
    if selected_technology == "llm":
        answer_id, details = adaptive_answer_with_llm(
            question,
            tokenizer,
            model,
            max_retries=max_retries,
        )

        return answer_id, adaptive_apply_decision_to_details(
            details,
            decision,
        )

    old_base_config = deepcopy(BASE_RAG_CONFIG)
    old_get_rag_answer_logged = globals().get("get_rag_answer_logged")
    old_get_rag_answer = globals().get("get_rag_answer")
    old_retrieve_local_wikipedia = globals().get("retrieve_local_wikipedia_context")

    try:
        activate_selected_configuration(
            selected_technology,
            use_best_if_available=True,
        )

        decision["actually_activated_config"] = BASE_RAG_CONFIG.get("active_configuration")
        decision["actually_activated_source"] = BASE_RAG_CONFIG.get("active_configuration_source")

        if selected_technology in {
            "rag_baseline_question_only",
            "rag_question_options",
        }:
            answer_id, details = adaptive_answer_with_baseline_rag(
                question,
                tokenizer,
                model,
                max_retries=max_retries,
            )

        else:
            answer_id, details = adaptive_answer_with_selected_rag(
                question,
                tokenizer,
                model,
                max_retries=max_retries,
            )

    finally:
        BASE_RAG_CONFIG.clear()
        BASE_RAG_CONFIG.update(old_base_config)

        adaptive_restore_global(
            "get_rag_answer_logged",
            old_get_rag_answer_logged,
        )

        adaptive_restore_global(
            "get_rag_answer",
            old_get_rag_answer,
        )

        adaptive_restore_global(
            "retrieve_local_wikipedia_context",
            old_retrieve_local_wikipedia,
        )

    return answer_id, adaptive_apply_decision_to_details(
        details,
        decision,
    )

In [ ]:
# Activation and debug utilities

def activate_adaptive_rag_variant(
    default_profile="balanced",
    refresh_best_configs=True,
    debug=True,
    **_unused_legacy_kwargs,
):
    global get_rag_answer_logged

    get_rag_answer_logged = get_adaptive_rag_answer_logged

    BASE_RAG_CONFIG.update({
        "adaptive_enabled": True,
        "adaptive_default_profile": default_profile,
        "adaptive_use_best_config_cache": True,
        "active_rag_variant": "adaptive_best_config_rag",
        "debug": debug,
    })

    if refresh_best_configs:
        load_adaptive_best_configs_for_current_model(
            verbose=False,
            display_table=False,
            as_dataframe=True,
        )

    print("Adaptive RAG activated.")
    print(f"default_profile: {default_profile}")
    print(f"debug: {debug}")


def show_adaptive_rag_decision(question_text, options=None):
    decision = build_adaptive_rag_decision(
        adaptive_make_fake_question(
            question_text,
            options,
        )
    )

    print("QUESTION:")
    print(question_text)
    print()
    print(f"PROFILE:                {decision['adaptive_profile']}")
    print(f"REASON:                 {decision['adaptive_reason']}")
    print(f"CANDIDATE TECHNOLOGIES: {decision['candidate_technologies']}")
    print(f"SELECTED TECHNOLOGY:    {decision['selected_technology']}")
    print(f"SELECTED BEST CONFIG:   {decision['selected_config_name']}")
    print(f"SELECTED PRETTY NAME:   {decision['selected_pretty_name']}")
    print(f"EXPECTED ACCURACY:      {decision['selection_accuracy']}")
    print(f"EXPECTED AVG TIME:      {decision['selection_avg_time_sec']}")
    print(f"SOURCE SUMMARY:         {decision['source_summary_path']}")

    return decision

In [ ]:
# Backward-compatible aliases

get_adaptive_best_config_answer_logged = get_adaptive_rag_answer_logged


def build_adaptive_rag_profile(question):
    decision = build_adaptive_rag_decision(question)

    return (
        decision["adaptive_profile"],
        decision["adaptive_reason"],
        {
            "candidate_technologies": decision["candidate_technologies"],
            "selected_technology": decision["selected_technology"],
            "selected_config_name": decision["selected_config_name"],
        },
    )


print("Adaptive RAG cell loaded successfully.")

Adaptive RAG cell loaded successfully.


## 🟢 Self RAG

In [ ]:
# Configuration

import re
import json
import time

SELF_RAG_CONFIG = {
    # Retrieval
    "top_k_context": 4,

    # To stay closer to the 30s limit, start with 2.
    # You can raise it to 3 for analysis runs.
    "max_final_contexts": 2,

    # Generation limits for the small critic calls
    "retrieval_decision_max_new_tokens": 8,
    "relevance_max_new_tokens": 12,
    "selfrag_generation_max_new_tokens": 80,
    "support_max_new_tokens": 16,
    "utility_max_new_tokens": 8,

    # Fallback
    "direct_answer_max_retries": 2,

    # Debug
    "debug": True,
}

In [ ]:
# Helper functions

def selfrag_get_config():
    """
    Merge SELF_RAG_CONFIG with BASE_RAG_CONFIG.
    BASE_RAG_CONFIG can override Self-RAG parameters when needed.
    """
    config = dict(SELF_RAG_CONFIG)

    if "BASE_RAG_CONFIG" in globals():
        for key in SELF_RAG_CONFIG:
            if key in BASE_RAG_CONFIG:
                config[key] = BASE_RAG_CONFIG[key]

    return config


def selfrag_build_prompt(system_msg, user_msg, tokenizer):
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def selfrag_generate_text(system_msg, user_msg, tokenizer, model, max_new_tokens):
    prompt = selfrag_build_prompt(system_msg, user_msg, tokenizer)

    return generate_answer(
        prompt,
        tokenizer,
        model,
        do_sample=False,
        max_new_tokens=max_new_tokens,
        temperature=1.0,
    )


def selfrag_get_option_text_by_id(question, answer_id):
    for opt in question.options:
        if opt.id == answer_id:
            return opt.text
    return ""


def selfrag_enrich_hits(hits):
    """
    Converts raw FAISS hits into readable debug objects.
    This avoids depending on CRAG helper functions.
    """
    enriched = []

    for hit in hits or []:
        corpus_id = int(hit["corpus_id"])

        enriched.append({
            "source_label": hit.get("source_label", "?"),
            "corpus_id": corpus_id,
            "score": float(hit.get("score", 0.0)),
            "query": hit.get("query", ""),
            "used_options": hit.get("used_options", False),
            "title": passage_metadata[corpus_id]["title"],
            "paragraph_id": passage_metadata[corpus_id]["paragraph_id"],
            "text": passages[corpus_id],
        })

    return enriched


def selfrag_context_from_hit(hit):
    corpus_id = int(hit["corpus_id"])
    source_label = hit.get("source_label", "?")
    score = float(hit.get("score", 0.0))

    title = passage_metadata[corpus_id]["title"]
    paragraph_id = passage_metadata[corpus_id]["paragraph_id"]
    passage_text = passages[corpus_id]

    return (
        f"Context Source {source_label}\n"
        f"Title: {title} | Paragraph: {paragraph_id} | Retrieval score: {score:.4f}\n"
        f"{passage_text}"
    )


def selfrag_parse_yes_no(raw_output):
    raw = str(raw_output or "").strip().lower()

    if re.search(r"\byes\b", raw):
        return "yes", False

    if re.search(r"\bno\b", raw):
        return "no", False

    # Safe default: retrieve rather than skipping evidence.
    return "yes", True


def selfrag_parse_relevance(raw_output):
    raw = str(raw_output or "").strip().lower()

    if re.search(r"\birrelevant\b", raw):
        return False, False

    if re.search(r"\brelevant\b", raw):
        return True, False

    yes_no, failed = selfrag_parse_yes_no(raw)
    if not failed:
        return yes_no == "yes", False

    return None, True


def selfrag_parse_support(raw_output):
    raw = str(raw_output or "").strip().lower()
    raw = raw.replace("-", " ").replace("_", " ")

    if re.search(r"\bfully\s+supported\b", raw):
        return "fully_supported", False

    if re.search(r"\bpartially\s+supported\b", raw):
        return "partially_supported", False

    if re.search(r"\bno\s+support\b", raw) or re.search(r"\bunsupported\b", raw):
        return "no_support", False

    return "no_support", True


def selfrag_parse_utility(raw_output):
    raw = str(raw_output or "").strip()
    match = re.search(r"\b([1-5])\b", raw)

    if match:
        return int(match.group(1)), False

    return 1, True


def selfrag_select_best_candidate(candidates):
    valid_candidates = [
        candidate for candidate in candidates
        if candidate.get("answer_id") is not None
    ]

    if not valid_candidates:
        return None

    support_rank = {
        "fully_supported": 2,
        "partially_supported": 1,
        "no_support": 0,
    }

    return max(
        valid_candidates,
        key=lambda candidate: (
            support_rank.get(candidate.get("support_label"), 0),
            int(candidate.get("utility_score", 1)),
            float(candidate.get("retrieval_score", 0.0)),
        ),
    )

In [ ]:
# Critic steps

def selfrag_decide_retrieval(question, tokenizer, model):
    options_text = format_options(question.options)

    system_msg = (
        "You are a routing module for a multiple-choice quiz system.\n"
        "Your task is to decide whether retrieval should be used before answering.\n"
        "Return only YES or NO.\n\n"
        "Return YES if retrieved evidence could make the answer more reliable.\n"
        "Return NO if the question can be answered reliably without retrieved evidence.\n"
        "When uncertain, return YES."
    )

    user_msg = (
        f"Question:\n{question.text}\n\n"
        f"Options:\n{options_text}\n\n"
        "Is retrieval necessary?"
    )

    raw_output = selfrag_generate_text(
        system_msg,
        user_msg,
        tokenizer,
        model,
        max_new_tokens=selfrag_get_config().get("retrieval_decision_max_new_tokens", 8),
    )

    decision, parsing_failed = selfrag_parse_yes_no(raw_output)
    return decision, raw_output, parsing_failed


def selfrag_grade_context_relevance(question, hit, tokenizer, model):
    context = selfrag_context_from_hit(hit)
    options_text = format_options(question.options)

    system_msg = (
        "You are a Self-RAG relevance critic.\n"
        "Judge whether the retrieved context is useful for answering the quiz question.\n\n"
        "Return only Relevant or Irrelevant.\n"
        "Return Relevant if the context mentions the key entity, relation, date, event, concept, or one answer option.\n"
        "Return Irrelevant only if the context is clearly unrelated."
    )

    user_msg = (
        f"Retrieved context:\n{context}\n\n"
        f"Question:\n{question.text}\n\n"
        f"Options:\n{options_text}\n\n"
        "Relevance:"
    )

    raw_output = selfrag_generate_text(
        system_msg,
        user_msg,
        tokenizer,
        model,
        max_new_tokens=selfrag_get_config().get("relevance_max_new_tokens", 12),
    )

    is_relevant, parsing_failed = selfrag_parse_relevance(raw_output)
    return is_relevant, raw_output, parsing_failed


def selfrag_generate_answer_for_context(question, context, tokenizer, model):
    options_text = format_options(question.options)

    system_msg = (
        "You are playing a multiple-choice quiz game using Self-RAG.\n"
        "Use the provided context as evidence.\n"
        "If the context is not decisive, choose the most plausible option.\n\n"
        "Output format:\n"
        "Reasoning: <short evidence-based reasoning>\n"
        "Final answer: <option_id>\n\n"
        "The final answer must be one of the provided option IDs."
    )

    user_msg = (
        f"Context:\n{context}\n\n"
        f"Question:\n{question.text}\n\n"
        f"Options:\n{options_text}\n\n"
        "Answer:"
    )

    raw_output = selfrag_generate_text(
        system_msg,
        user_msg,
        tokenizer,
        model,
        max_new_tokens=selfrag_get_config().get("selfrag_generation_max_new_tokens", 80),
    )

    valid_ids = [opt.id for opt in question.options]
    answer_id = extract_option_id(raw_output, valid_ids)

    return answer_id, raw_output


def selfrag_assess_support(question, context, answer_id, raw_output, tokenizer, model):
    answer_text = selfrag_get_option_text_by_id(question, answer_id)
    options_text = format_options(question.options)

    system_msg = (
        "You are a Self-RAG support critic.\n"
        "Judge whether the selected answer is supported by the retrieved context.\n\n"
        "Return only one label:\n"
        "Fully supported\n"
        "Partially supported\n"
        "No support"
    )

    user_msg = (
        f"Context:\n{context}\n\n"
        f"Question:\n{question.text}\n\n"
        f"Options:\n{options_text}\n\n"
        f"Selected answer id: {answer_id}\n"
        f"Selected answer text: {answer_text}\n"
        f"Generated response:\n{raw_output}\n\n"
        "Support label:"
    )

    raw_support = selfrag_generate_text(
        system_msg,
        user_msg,
        tokenizer,
        model,
        max_new_tokens=selfrag_get_config().get("support_max_new_tokens", 16),
    )

    support_label, parsing_failed = selfrag_parse_support(raw_support)
    return support_label, raw_support, parsing_failed


def selfrag_evaluate_utility(question, answer_id, raw_output, tokenizer, model):
    answer_text = selfrag_get_option_text_by_id(question, answer_id)

    system_msg = (
        "You are a Self-RAG utility critic.\n"
        "Rate how useful the generated response is for answering the multiple-choice question.\n\n"
        "Return only one integer from 1 to 5.\n"
        "1 = not useful, 3 = somewhat useful, 5 = directly useful and decisive."
    )

    user_msg = (
        f"Question:\n{question.text}\n\n"
        f"Options:\n{format_options(question.options)}\n\n"
        f"Selected answer id: {answer_id}\n"
        f"Selected answer text: {answer_text}\n"
        f"Generated response:\n{raw_output}\n\n"
        "Utility score:"
    )

    raw_utility = selfrag_generate_text(
        system_msg,
        user_msg,
        tokenizer,
        model,
        max_new_tokens=selfrag_get_config().get("utility_max_new_tokens", 8),
    )

    utility_score, parsing_failed = selfrag_parse_utility(raw_utility)
    return utility_score, raw_utility, parsing_failed

In [ ]:
# Self-RAG answer flow

def selfrag_generate_direct_answer(question, tokenizer, model, max_retries=None):
    valid_ids = [opt.id for opt in question.options]

    system_msg, user_msg = build_prompt(question.text, question.options)
    prompt = selfrag_build_prompt(system_msg, user_msg, tokenizer)

    max_new_tokens = CONFIG.get("max_new_tokens", 15)
    if CONFIG.get("prompt_strategy") in {"zero_shot_cot", "cot"}:
        max_new_tokens = CONFIG.get("reasoning_max_new_tokens", 256)

    if max_retries is None:
        max_retries = selfrag_get_config().get("direct_answer_max_retries", 2)

    attempts = []

    for attempt_idx in range(1, max_retries + 1):
        raw_output = generate_answer(
            prompt,
            tokenizer,
            model,
            do_sample=CONFIG.get("do_sample", False),
            max_new_tokens=max_new_tokens,
            temperature=CONFIG.get("temperature", 1.0),
        )

        parsed_answer = parse_answer(raw_output, valid_ids)

        attempts.append({
            "attempt": attempt_idx,
            "raw_output": raw_output,
            "parsed_answer": parsed_answer,
        })

        if parsed_answer is not None:
            return parsed_answer, raw_output, attempts, False

    fallback_answer = valid_ids[0]
    fallback_output = attempts[-1]["raw_output"] if attempts else ""

    return fallback_answer, fallback_output, attempts, True


def get_self_rag_answer_logged(question, tokenizer, model, max_retries=3):
    valid_ids = [opt.id for opt in question.options]
    config = selfrag_get_config()

    max_contexts = int(config.get("max_final_contexts", 2))

    retrieval_decision_time_sec = 0.0
    retrieval_time_sec = 0.0
    relevance_time_sec = 0.0
    generation_time_sec = 0.0
    support_time_sec = 0.0
    utility_time_sec = 0.0

    candidate_hits = []
    selected_hits = []
    relevance_outputs = []
    selfrag_candidates = []
    attempts = []

    fallback_used = False
    fallback_reason = None

    context = ""
    context_status = "selfrag_no_retrieval"
    prompt_mode = "selfrag_direct_generation"

    # --------------------------------------------------------
    # 1. Decide whether retrieval is needed
    # --------------------------------------------------------
    start = time.perf_counter()

    retrieval_decision, retrieval_decision_raw_output, retrieval_decision_parsing_failed = (
        selfrag_decide_retrieval(question, tokenizer, model)
    )

    retrieval_decision_time_sec = time.perf_counter() - start

    # --------------------------------------------------------
    # 2. Direct answer branch
    # --------------------------------------------------------
    if retrieval_decision == "no":
        start = time.perf_counter()

        final_answer, final_raw_output, attempts, fallback_used = selfrag_generate_direct_answer(
            question,
            tokenizer,
            model,
            max_retries=max_retries,
        )

        generation_time_sec = time.perf_counter() - start

        if fallback_used:
            fallback_reason = "direct_generation_parse_fallback"

    # --------------------------------------------------------
    # 3. Retrieval branch
    # --------------------------------------------------------
    else:
        start = time.perf_counter()

        candidate_hits = retrieve_local_wikipedia_context_with_options_candidates(
            question,
            top_k=int(config.get("top_k_context", 4)),
        )

        retrieval_time_sec = time.perf_counter() - start

        # ----------------------------------------------------
        # 3a. Relevance grading
        # ----------------------------------------------------
        relevant_hits = []
        relevance_parsing_failed = False

        start = time.perf_counter()

        for hit in candidate_hits:
            is_relevant, raw_relevance, parsing_failed = selfrag_grade_context_relevance(
                question,
                hit,
                tokenizer,
                model,
            )

            relevance_outputs.append({
                "source_label": hit.get("source_label"),
                "raw_output": raw_relevance,
                "parsed_relevant": is_relevant,
                "parsing_failed": parsing_failed,
            })

            if parsing_failed:
                relevance_parsing_failed = True

            if is_relevant is True:
                relevant_hits.append(hit)

        relevance_time_sec = time.perf_counter() - start

        # If critic parsing fails, keep top-1 instead of losing all context.
        if relevance_parsing_failed:
            selected_hits = candidate_hits[:1]
            context_status = "selfrag_top1_after_relevance_parse_failure"

        elif relevant_hits:
            selected_hits = relevant_hits[:max_contexts]
            context_status = f"selfrag_relevant_contexts_{len(selected_hits)}"

        else:
            selected_hits = []
            context_status = "selfrag_empty_after_relevance_filter"

        # ----------------------------------------------------
        # 3b. Fallback if no context remains
        # ----------------------------------------------------
        if not selected_hits:
            prompt_mode = f"selfrag_strategy_fallback_{CONFIG.get('prompt_strategy', 'zero_shot')}"

            start = time.perf_counter()

            final_answer, final_raw_output, attempts, fallback_used = selfrag_generate_direct_answer(
                question,
                tokenizer,
                model,
                max_retries=max_retries,
            )

            generation_time_sec = time.perf_counter() - start

            if fallback_used:
                fallback_reason = "empty_context_direct_generation_parse_fallback"

        # ----------------------------------------------------
        # 3c. Generate one candidate answer per selected context
        # ----------------------------------------------------
        else:
            prompt_mode = "selfrag_context_candidate_generation"
            context = "\n\n".join(selfrag_context_from_hit(hit) for hit in selected_hits)

            for hit in selected_hits:
                single_context = selfrag_context_from_hit(hit)

                start = time.perf_counter()

                answer_id, raw_answer = selfrag_generate_answer_for_context(
                    question,
                    single_context,
                    tokenizer,
                    model,
                )

                generation_time_sec += time.perf_counter() - start

                attempts.append({
                    "attempt": len(attempts) + 1,
                    "source_label": hit.get("source_label"),
                    "raw_output": raw_answer,
                    "parsed_answer": answer_id,
                })

                if answer_id is None:
                    selfrag_candidates.append({
                        "source_label": hit.get("source_label"),
                        "answer_id": None,
                        "raw_output": raw_answer,
                        "support_label": "no_support",
                        "support_raw_output": "SKIPPED_UNPARSED_ANSWER",
                        "support_parsing_failed": True,
                        "utility_score": 1,
                        "utility_raw_output": "SKIPPED_UNPARSED_ANSWER",
                        "utility_parsing_failed": True,
                        "retrieval_score": float(hit.get("score", 0.0)),
                    })
                    continue

                # Support check
                start = time.perf_counter()

                support_label, support_raw_output, support_parsing_failed = selfrag_assess_support(
                    question,
                    single_context,
                    answer_id,
                    raw_answer,
                    tokenizer,
                    model,
                )

                support_time_sec += time.perf_counter() - start

                # Utility score
                start = time.perf_counter()

                utility_score, utility_raw_output, utility_parsing_failed = selfrag_evaluate_utility(
                    question,
                    answer_id,
                    raw_answer,
                    tokenizer,
                    model,
                )

                utility_time_sec += time.perf_counter() - start

                selfrag_candidates.append({
                    "source_label": hit.get("source_label"),
                    "answer_id": answer_id,
                    "raw_output": raw_answer,
                    "support_label": support_label,
                    "support_raw_output": support_raw_output,
                    "support_parsing_failed": support_parsing_failed,
                    "utility_score": utility_score,
                    "utility_raw_output": utility_raw_output,
                    "utility_parsing_failed": utility_parsing_failed,
                    "retrieval_score": float(hit.get("score", 0.0)),
                })

            best_candidate = selfrag_select_best_candidate(selfrag_candidates)

            if best_candidate is not None:
                final_answer = best_candidate["answer_id"]
                final_raw_output = best_candidate["raw_output"]

            else:
                parsed_attempts = [
                    attempt for attempt in attempts
                    if attempt.get("parsed_answer") in valid_ids
                ]

                if parsed_attempts:
                    final_answer = parsed_attempts[0]["parsed_answer"]
                    final_raw_output = parsed_attempts[0]["raw_output"]
                    fallback_used = True
                    fallback_reason = "first_valid_candidate_after_selection_failure"

                else:
                    final_answer = valid_ids[0]
                    final_raw_output = attempts[-1]["raw_output"] if attempts else ""
                    fallback_used = True
                    fallback_reason = "default_first_valid_id"

    total_answer_time_sec = (
        retrieval_decision_time_sec
        + retrieval_time_sec
        + relevance_time_sec
        + generation_time_sec
        + support_time_sec
        + utility_time_sec
    )

    relevant_source_labels = [
        hit.get("source_label")
        for hit in selected_hits
        if hit.get("source_label")
    ]

    details = {
        "answer_id": final_answer,
        "fallback_used": fallback_used,
        "fallback_reason": fallback_reason,

        "prompt_mode": prompt_mode,
        "context_status": context_status,
        "selection_mode": "selfrag_support_utility_retrieval_score",

        "context": context,
        "hits": selfrag_enrich_hits(candidate_hits),
        "filtered_hits": selfrag_enrich_hits(selected_hits),
        "relevant_source_labels": relevant_source_labels,

        "attempts": attempts,
        "selfrag_candidates": selfrag_candidates,

        "retrieval_decision": retrieval_decision,
        "retrieval_decision_raw_output": retrieval_decision_raw_output,
        "retrieval_decision_parsing_failed": retrieval_decision_parsing_failed,
        "relevance_check_raw_output": json.dumps(relevance_outputs, ensure_ascii=False),

        "retrieval_decision_time_sec": retrieval_decision_time_sec,
        "retrieval_time_sec": retrieval_time_sec,
        "relevance_time_sec": relevance_time_sec,
        "generation_time_sec": generation_time_sec,
        "support_time_sec": support_time_sec,
        "utility_time_sec": utility_time_sec,
        "total_answer_time_sec": total_answer_time_sec,

        "grounded": "selfrag_support_assessed",
        "active_rag_variant": "self_rag",
        "selfrag_config": dict(config),
    }

    if config.get("debug", True):
        print("\n=== Self-RAG Flow ===")

        print(f"\n1. Retrieval decision\nDecision: {retrieval_decision.upper()}")

        print("\n2. Retrieved contexts")
        for hit in candidate_hits:
            corpus_id = int(hit["corpus_id"])
            title = passage_metadata[corpus_id]["title"]
            paragraph_id = passage_metadata[corpus_id]["paragraph_id"]
            score = float(hit.get("score", 0.0))
            label = hit.get("source_label", "?")
            print(f"[{label}] {title} | paragraph {paragraph_id} | score={score:.4f}")

        print("\n3. Relevance filtering")
        for item in relevance_outputs:
            label = item.get("source_label", "?")
            parsed = item.get("parsed_relevant")
            status = "Relevant" if parsed is True else "Irrelevant" if parsed is False else "Unclear"
            print(f"[{label}] {status}")

        print("\n4. Candidate generation")
        for attempt in attempts:
            label = attempt.get("source_label", f"attempt {attempt.get('attempt', '?')}")
            raw = str(attempt.get("raw_output", "")).replace("\n", " ")
            raw = raw[:250] + "..." if len(raw) > 250 else raw
            print(f"Candidate from {label}:")
            print(f"Answer: {attempt.get('parsed_answer')}")
            print(f"Reasoning: {raw}")

        print("\n5. Support + utility")
        for candidate in selfrag_candidates:
            label = candidate.get("source_label", "?")
            support = candidate.get("support_label", "unknown")
            utility = candidate.get("utility_score", "?")
            print(f"Candidate {label}: Support={support} | Utility={utility}/5")

        print("\n6. Final selection")
        print(f"Selected answer: {final_answer}")

        if fallback_used:
            print(f"Selection reason: fallback used ({fallback_reason})")
        elif len([c for c in selfrag_candidates if c.get('answer_id') is not None]) == 1:
            print("Selection reason: only valid candidate available")
        else:
            print("Selection reason: best candidate by support, utility and retrieval score")

        for candidate in selfrag_candidates:
            if candidate.get("answer_id") == final_answer:
                if candidate.get("support_label") != "fully_supported":
                    print("Warning: selected answer is weakly supported")
                if int(candidate.get("utility_score", 1)) <= 2:
                    print("Warning: selected answer has low utility")

        if context_status == "selfrag_empty_after_relevance_filter":
            print("Warning: no retrieved context survived relevance filtering")

        print("\nTiming:")
        print(
            f"decision={retrieval_decision_time_sec:.2f}s | "
            f"retrieval={retrieval_time_sec:.2f}s | "
            f"relevance={relevance_time_sec:.2f}s | "
            f"generation={generation_time_sec:.2f}s | "
            f"support={support_time_sec:.2f}s | "
            f"utility={utility_time_sec:.2f}s | "
            f"total={total_answer_time_sec:.2f}s"
        )

        print("=== End Self-RAG Flow ===\n")

    return final_answer, details

In [ ]:
# Activate Self-RAG

def activate_self_rag_variant(
    top_k_context=4,
    max_final_contexts=2,
    retrieval_query_mode="question_only",
    debug=True,
):
    global retrieve_local_wikipedia_context
    global get_rag_answer_logged

    retrieve_local_wikipedia_context = retrieve_local_wikipedia_context_with_options_candidates
    get_rag_answer_logged = get_self_rag_answer_logged

    SELF_RAG_CONFIG.update({
        "top_k_context": top_k_context,
        "max_final_contexts": max_final_contexts,
        "debug": debug,
    })

    BASE_RAG_CONFIG.update({
        "active_rag_variant": "self_rag",
        "top_k_context": top_k_context,
        "max_final_contexts": max_final_contexts,
        "retrieval_query_mode": retrieval_query_mode,
        "retrieval_decision_max_new_tokens": SELF_RAG_CONFIG["retrieval_decision_max_new_tokens"],
        "relevance_max_new_tokens": SELF_RAG_CONFIG["relevance_max_new_tokens"],
        "selfrag_generation_max_new_tokens": SELF_RAG_CONFIG["selfrag_generation_max_new_tokens"],
        "support_max_new_tokens": SELF_RAG_CONFIG["support_max_new_tokens"],
        "utility_max_new_tokens": SELF_RAG_CONFIG["utility_max_new_tokens"],
        "debug": debug,
    })

    print("Activated Self-RAG variant.")
    print(f"top_k_context={top_k_context}")
    print(f"max_final_contexts={max_final_contexts}")
    print(f"retrieval_query_mode={retrieval_query_mode}")

## 🟢 Corrective RAG (CRAG)

In [ ]:
# CRAG dependencies and config

!pip install -q -U langchain-community duckduckgo-search ddgs requests beautifulsoup4

import time
import json
import re
import math
import numpy as np
import faiss
from collections import defaultdict
from urllib.parse import urlparse


CRAG_TRUSTED_DOMAINS = {
    "wikipedia.org": 0.30,
    "britannica.com": 0.25,
    "imdb.com": 0.20,
    "netflix.com": 0.20,
    "bbc.com": 0.15,
}

CRAG_LOW_QUALITY_DOMAINS = {
    "tiktok.com": -0.40,
    "youtube.com": -0.20,
    "quora.com": -0.30,
    "reddit.com": -0.20,
}

CRAG_CONFIG = {
    # Local retrieval
    "use_semantic_retrieval": True,
    "use_bm25_retrieval": True,
    "use_cross_encoder_rerank": False,

    # Corrective routing
    "local_answerability_min_confidence": 0.65,
    "local_min_context_chars": 250,
    "router_max_local_context_chars": 3500,

    # News routing
    "force_web_for_news": True,
    "news_comp_id": 5,

    # Web fallback
    "use_web_fallback": True,
    "web_fallback_on_incorrect": True,
    "web_search_top_k": 4,
    "web_search_fetch_k": 4,
    "web_min_relevance_score": 0.0,
    "web_use_llm_query_rewrite": False,
    "web_use_option_wise_search": False,
    "web_optionwise_per_option_k": 1,
    "web_optionwise_general_k": 1,
    "web_use_domain_quality_prior": True,

    # Local budgets
    "top_k_context": 12,
    "final_top_k_context": 3,
    "fusion_semantic_top_k": 30,
    "fusion_bm25_top_k": 30,
    "fusion_rrf_k": 60,
    "fusion_semantic_query_mode": "question_only",
    "fusion_bm25_query_mode": "question_only",

    # Generation
    "rag_max_new_tokens": 64,
    "crag_answer_max_retries": 1,
    "answer_repair_max_new_tokens": 12,
    "enable_answer_repair": True,
    "enable_direct_option_match": True,
    "answer_fallback_mode": "option_text_context_overlap",

    # Debug
    "debug": True,
    "debug_show_local_context": True,
    "debug_show_router_raw_output": True,
    "debug_show_router_prompt": False,
    "debug_show_final_context": True,
    "debug_show_final_user_prompt": False,
    "debug_context_max_chars": 2200,
    "debug_prompt_max_chars": 3000,
}

print("CRAG_CONFIG initialized.")
print(f"  Web fallback:      {CRAG_CONFIG['use_web_fallback']}")
print(f"  Option-wise web:   {CRAG_CONFIG['web_use_option_wise_search']}")
print(f"  News comp_id:      {CRAG_CONFIG['news_comp_id']}")
print(f"  Debug:             {CRAG_CONFIG['debug']}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 121.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
CRAG_CONFIG initialized.
  Web fallback:      True
  Option-wise web:   False
  News comp_id:      5
  Debug:             True


In [ ]:
# Constants

CRAG_FALLBACK_STOPWORDS = {
    "the", "a", "an", "and", "or", "of", "in", "on", "to", "for", "by",
    "with", "from", "as", "at", "is", "are", "was", "were", "be", "been",
    "being", "which", "what", "who", "where", "when", "why", "how",
    "term", "describes", "describe",
}

In [ ]:
# Helpers

# --- Text processing ---

# "Red Planet: Mars!" -> "red planet mars"
# Lowercase text, remove punctuation/symbols, and normalize spaces.
def crag_normalize_text_for_match(text):
    text = str(text or "").lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


# "Mars is the red planet" -> {"mars", "red", "planet"}
# Keep only informative tokens: length >= 3 and not in the stopword list.
def crag_content_tokens(text):
    return {
        token
        for token in crag_normalize_text_for_match(text).split()
        if len(token) >= 3 and token not in CRAG_FALLBACK_STOPWORDS
    }

def crag_score_option_against_text(option_text, text):
    option_norm = crag_normalize_text_for_match(option_text)
    text_norm = crag_normalize_text_for_match(text)
    if not option_norm or not text_norm:
        return 0.0

    # Count exact occurrences of the full normalized option in the text.
    # option_norm = "red planet"
    # text_norm = "mars is known as the red planet"
    # exact_count = 1
    exact_count = len(re.findall(rf"\b{re.escape(option_norm)}\b", text_norm))

    # Extract informative words from both option and text.
    # option_tokens = {"red", "planet"}
    # text_tokens   = {"mars", "known", "red", "planet"}
    option_tokens = crag_content_tokens(option_norm)
    text_tokens = crag_content_tokens(text_norm)

    # Compute how many option tokens appear in the text.
    # {"red", "planet"} ∩ {"mars", "known", "red", "planet"} = {"red", "planet"}
    # overlap = 2 / 2 = 1.0
    overlap = len(option_tokens & text_tokens) / max(len(option_tokens), 1) if option_tokens else 0.0

    # Final score:
    # - exact phrase match is weighted by 3.0
    # - token overlap adds a smaller partial-match signal
    return float(3.0 * exact_count + overlap)

# --- Question / routing helpers ---

def crag_get_question_comp_id(question):
    for name in [
        "comp_id", "competition_id", "competitionId",
        "category_id", "categoryId", "topic_id", "topicId",
        "competition", "category", "topic",
    ]:
        if hasattr(question, name):
            value = getattr(question, name)
            try:
                return int(value)
            except Exception:
                pass
            if isinstance(value, dict):
                for key in ["id", "comp_id", "competition_id", "category_id"]:
                    if key in value:
                        try:
                            return int(value[key])
                        except Exception:
                            pass
            for nested_name in ["id", "comp_id", "competition_id", "category_id"]:
                if hasattr(value, nested_name):
                    try:
                        return int(getattr(value, nested_name))
                    except Exception:
                        pass
    for name in ["comp_id", "COMP_ID", "competition_id", "COMPETITION_ID", "category_id", "CATEGORY_ID"]:
        if name in globals():
            try:
                return int(globals()[name])
            except Exception:
                pass
    return None

# Force web fallback for News questions.
def crag_should_force_web(question, config):
    return (
        config.get("force_web_for_news", True)
        and crag_get_question_comp_id(question) == config.get("news_comp_id", 5)
    )

# Build the query
# "Which planet is known as the Red Planet?"
def crag_build_local_query(question, mode="question_only"):
    question_text = str(getattr(question, "text", question)).strip()

    # "Which planet is known as the Red Planet? Earth Mars Jupiter Venus"
    if mode == "question_options":
        options_text = " ".join(str(opt.text).strip() for opt in question.options)
        return f"{question_text} {options_text}".strip()
    return question_text


# --- Output parsing ---

# It returns a clean dictionary:
# {
#   "answerable": True/False,
#   "answer": option_id or None,
#   "confidence": value between 0.0 and 1.0
# }
def crag_parse_answerability_router_output(raw_output, valid_ids):
    text = str(raw_output or "").strip()
    text = text.replace("```json", "").replace("```", "").strip()

    # {{"answerable": true, ...} -> {"answerable": true, ...}
    while text.startswith("{{"):
        text = text[1:].strip()

    candidates = []

    # "Here is the result: {"answerable": true, "answer": 1}" -> {"answerable": true, "answer": 1}
    first = text.find("{")
    last = text.rfind("}")

    if first != -1 and last != -1 and last > first:
        candidates.append(text[first:last + 1])
    if first != -1:
        partial = text[first:]
        for pos in [m.start() for m in re.finditer(r"\}", partial)]:
            candidates.append(partial[:pos + 1])

    for candidate in candidates:
        candidate = candidate.strip()
        while candidate.startswith("{{"): # fix duplicated opening braces.
            candidate = candidate[1:].strip()
        try:
            data = json.loads(candidate)
            answer = data.get("answer") # "1" -> 1
            try:
                answer = int(answer) if answer is not None else None
            except Exception:
                answer = None
            if answer not in valid_ids:
                answer = None
            confidence = data.get("confidence", 0.0) # Parse confidence as float.
            try:
                confidence = float(confidence)
            except Exception:
                confidence = 0.0
            return {
                "answerable": bool(data.get("answerable", False)),
                "answer": answer,
                "confidence": max(0.0, min(1.0, confidence)), # Confidence is clipped between 0.0 and 1.0.
            }
        except Exception:
            pass

    # "The answer is 2" -> answer = 2
    answer = extract_option_id(text, valid_ids)

    # Try to recover confidence with regex.
    conf_match = re.search(r'"?confidence"?\s*:\s*([01](?:\.\d+)?)', text, flags=re.I)
    confidence = 0.50 if answer is not None else 0.0
    if conf_match:
        try:
            confidence = float(conf_match.group(1))
        except Exception:
            pass

    answerable = False
    if re.search(r'"?answerable"?\s*:\s*true', text, flags=re.I):
        answerable = True
    elif answer is not None and confidence >= 0.65:
        answerable = True

    return {
        "answerable": answerable,
        "answer": answer,
        "confidence": max(0.0, min(1.0, confidence)),
    }


# Parse web search results into a standard list of dictionaries.
#
# Expected output format:
# [
#   {
#     "title": "...",
#     "link": "...",
#     "snippet": "..."
#   },
#   ...
# ]
#
def crag_parse_web_results(raw_results):

    # already a Python list
    if isinstance(raw_results, list):
        items = raw_results

    # results are a JSON string
    else:
        try:
            items = json.loads(raw_results)
        except Exception:
            items = []

    parsed = []
    if isinstance(items, list):
        for item in items:
            if not isinstance(item, dict):
                continue
            parsed.append({
                "title": item.get("title", "Untitled"),
                "link": item.get("link") or item.get("href") or item.get("url") or "",
                "snippet": item.get("snippet") or item.get("body") or item.get("description") or "",
            })

    if parsed:
        return parsed

    # Fallback for messy string outputs.
    # This regex extracts snippet, title, and link manually.
    pattern = re.compile(
        r"snippet:\s*(.*?),\s*title:\s*(.*?),\s*link:\s*(https?://\S+?)(?=,\s*snippet:|$)",
        re.DOTALL,
    )
    for snippet, title, link in pattern.findall(str(raw_results)):
        parsed.append({
            "title": " ".join(title.split()),
            "link": link.strip().rstrip(","),
            "snippet": " ".join(snippet.split()),
        })

    return parsed


# --- Web query building ---

# Clean text before using it in a search query.
def crag_clean_search_text(text):
    text = str(text or "").replace("\n", " ")
    text = re.sub(r"\s*\[\d+\]\s*", " ", text) # Remove citation-like markers such as [1], [2], [10].
    text = re.sub(r"\s*\(\d+\)\s*", " ", text) # Remove numeric markers such as (1), (2), (10).
    return re.sub(r"\s+", " ", text).strip() # Normalize multiple spaces into a single space.


# "Who discovered “Mars — the Red Planet”?" -> 'Who discovered "Mars - the Red Planet"?'
def crag_normalize_for_query(text):
    text = crag_clean_search_text(text)
    for old, new in {"'": "'", "'": "'", "\u201c": '"', "\u201d": '"', "\u2013": "-", "\u2014": "-"}.items():
        text = text.replace(old, new)
    return text.strip()

# Build the basic web query from the question text only.
def crag_build_plain_web_query(question):
    return crag_normalize_for_query(getattr(question, "text", question))

# Prepare an answer option before adding it to a web search query.
def crag_quote_option_for_query(option_text):
    option_text = crag_normalize_for_query(option_text)
    if not option_text:
        return ""

    # "Red Planet" -> "\"Red Planet\""
    # This helps the search engine look for the exact phrase instead of searching the words independently.
    if len(option_text) > 2 and len(option_text.split()) >= 2:
        return f'"{option_text}"'
    return option_text

# Build a web query using the question plus one candidate answer option.
# "Which planet is known as the Red Planet? Mars"
def crag_build_option_web_query(question, option):
    base = crag_build_plain_web_query(question)
    opt = crag_quote_option_for_query(option.text)
    return f"{base} {opt}".strip() if opt else base


# --- Web result scoring / selection ---


# result = {
#   "title": "Mars - Wikipedia",
#   "snippet": "Mars is often called the Red Planet."
# }
#
# -> "Mars - Wikipedia\nMars is often called the Red Planet."
def crag_result_text(result):
    return f"{result.get('title', '')}\n{result.get('snippet', '')}"


def crag_get_domain(url):
    try:
        domain = urlparse(str(url or "")).netloc.lower()
        return domain[4:] if domain.startswith("www.") else domain
    except Exception:
        return ""


def crag_domain_quality_bonus(url, config):
    if not config.get("web_use_domain_quality_prior", True):
        return 0.0

    domain = crag_get_domain(url)
    if not domain:
        return 0.0

    for trusted_domain, bonus in CRAG_TRUSTED_DOMAINS.items():
        if domain.endswith(trusted_domain):
            return float(bonus)

    for low_quality_domain, penalty in CRAG_LOW_QUALITY_DOMAINS.items():
        if domain.endswith(low_quality_domain):
            return float(penalty)

    return 0.0


# Score how relevant a web result is to the original question.
def crag_score_web_result(result, question, config):
    question_tokens = crag_content_tokens(question.text)
    result_tokens = crag_content_tokens(crag_result_text(result))
    if not question_tokens or not result_tokens:
        return 0.0
    return len(question_tokens & result_tokens) / max(len(question_tokens), 1)

# Build a summary of how much evidence each option received from web results.
# [
#   {"option_id": 1, "option_text": "Mars", "score": 4.0, "num_results": 2},
#   {"option_id": 0, "option_text": "Earth", "score": 0.5, "num_results": 1},
#   ...
# ]
def crag_build_option_evidence_summary(question, option_results):
    summary = []
    for opt in question.options:
        results = option_results.get(int(opt.id), [])
        combined_text = "\n".join(crag_result_text(r) for r in results)
        summary.append({
            "option_id": int(opt.id),
            "option_text": opt.text,
            "score": crag_score_option_against_text(opt.text, combined_text),
            "num_results": len(results),
        })
    return sorted(summary, key=lambda x: x["score"], reverse=True)

# It combines:
# 1. a few general web results for the question;
# 2. a few option-wise results, one per answer option;
# 3. remaining high-scoring results if there is still space.
def crag_select_balanced_optionwise_results(question, general_results, option_results, config):
    final_k = int(config.get("web_search_top_k", 4)) # Final number of web sources to keep.
    per_option_k = int(config.get("web_optionwise_per_option_k", 1))
    general_k = int(config.get("web_optionwise_general_k", 1))

    selected = []
    seen_urls = set()

    # Step 1: add top general results first.
    for result in general_results[:general_k]:
        url = result.get("link", "")
        if url and url not in seen_urls:
            selected.append(result)
            seen_urls.add(url)

    # Step 2: add option-wise results.
    # for option "Mars", add the best result retrieved with question + "Mars".
    for opt in question.options:
        added = 0
        for result in option_results.get(int(opt.id), []):
            url = result.get("link", "")
            if not url or url in seen_urls:
                continue
            selected.append(result)
            seen_urls.add(url)
            added += 1
            if added >= per_option_k:
                break

    # Step 3: collect all remaining results as backup candidates.
    remaining = general_results[:]
    for results in option_results.values():
        remaining.extend(results)
    remaining.sort(
        key=lambda x: (
            x.get("option_evidence_score", 0.0),
            x.get("final_web_score", x.get("web_score", 0.0)),
        ),
        reverse=True,
    )

    # Step 4: fill remaining slots with best non-duplicate results.
    for result in remaining:
        if len(selected) >= final_k:
            break
        url = result.get("link", "")
        if url and url not in seen_urls:
            selected.append(result)
            seen_urls.add(url)

    # Keep at most final_k results.
    return selected[:final_k]

# Format selected web results as final context for the LLM.

# WEB CONTEXT
# Web query: Which planet is known as the Red Planet? Mars
#
# Option evidence summary:
# [1] Mars | score=4.00 | results=2
#
# Context Source W1
# Title: Mars - Wikipedia
# URL: ...
# Snippet: Mars is often called the Red Planet.
def crag_format_web_context(selected, heading, query, option_evidence=None):
    if not selected:
        return ""

    blocks = []
    for idx, item in enumerate(selected, start=1):
        label = f"W{idx}"
        block = (
            f"Context Source {label}\n"
            f"Title: {item.get('title', 'Untitled')}\n"
            f"URL: {item.get('link', '')}\n"
            f"Snippet: {item.get('snippet', '')}"
        )
        if item.get("query_type") == "option_wise":
            block += (
                f"\nQuery type: option_wise"
                f"\nOption id: {item.get('option_id')}"
                f"\nOption text: {item.get('option_text')}"
            )
        blocks.append(block)

    evidence_text = ""
    if option_evidence:
        evidence_text = (
            "\nOption evidence summary:\n"
            + "\n".join(
                f"[{e['option_id']}] {e['option_text']} | score={e['score']:.2f} | results={e['num_results']}"
                for e in option_evidence
            )
            + "\n"
        )

    return (
        f"{heading}\n"
        f"Web query: {query}\n"
        f"{evidence_text}\n"
        + "\n\n".join(blocks)
    )


# --- Answer fallback ---

# Fallback used when the model output does not contain a clean option id.

# 1.
# raw_output = "The correct answer is Mars."
# option "Mars" appears exactly -> return its option id.

# 2.
# context = "Mars is often called the Red Planet."
# option "Mars" is strongly supported by the context -> return its option id.

def crag_option_text_fallback(question, context, raw_output):
    options = list(getattr(question, "options", []) or [])
    if not options:
        return None

    output_scores = [
        (opt.id, crag_score_option_against_text(opt.text, raw_output), opt.text)
        for opt in options
    ]
    positive = [(opt_id, score, text) for opt_id, score, text in output_scores if score >= 3.0]
    if len(positive) == 1:
        return positive[0][0]

    context_scores = [
        (opt.id, crag_score_option_against_text(opt.text, context), opt.text)
        for opt in options
    ]
    context_scores.sort(key=lambda item: item[1], reverse=True)

    best_id, best_score, _ = context_scores[0]
    second_score = context_scores[1][1] if len(context_scores) > 1 else 0.0

    if best_score >= 3.0 and best_score >= second_score + 1.0:
        return best_id

    return None


In [ ]:
# Prompts

def crag_prompt_answerability_router(question, local_context, config):
    options_text = format_options(question.options)
    max_context_chars = int(config.get("router_max_local_context_chars", 3500))
    compact_context = str(local_context or "")[:max_context_chars]

    system_msg = (
        "You are a routing judge for a multiple-choice RAG system.\n"
        "Your task is NOT to answer from general knowledge.\n"
        "Use only the provided local context.\n"
        "Decide whether the local context clearly supports exactly one option.\n"
        "Return JSON only."
    )

    user_msg = (
        f"Question:\n{question.text}\n\n"
        f"Options:\n{options_text}\n\n"
        f"Local context analyzed by router:\n{compact_context}\n\n"
        "Return exactly this JSON format:\n"
        "{\n"
        "  \"answerable\": true or false,\n"
        "  \"answer\": option_id_or_null,\n"
        "  \"confidence\": number_between_0_and_1,\n"
        "  \"reason\": \"short reason\"\n"
        "}\n\n"
        "Rules:\n"
        "- answerable=true only if the context clearly supports one option.\n"
        "- answerable=false if the context is generic, incomplete, contradictory, or supports multiple options.\n"
        "- Do not use outside knowledge."
    )

    return system_msg, user_msg


def crag_prompt_query_rewrite(question):
    system_msg = (
        "You rewrite quiz questions into concise web search queries.\n"
        "Do not answer the question.\n"
        "Do not include all answer options.\n"
        "Return only one search query."
    )
    user_msg = f"Question:\n{question.text}\n\nRewrite as a concise web search query:"

    return system_msg, user_msg


def crag_prompt_repair_answer(question, raw_output):
    options_text = format_options(question.options)
    previous = str(raw_output or "").strip()[:1200]

    system_msg = "You fix malformed multiple-choice answers. Return only one valid option ID. Do not explain."

    user_msg = (
        f"Question:\n{question.text}\n\n"
        f"Options:\n{options_text}\n\n"
        f"Previous model output:\n{previous}\n\n"
        "Return only the option ID."
    )

    return system_msg, user_msg


In [ ]:
# Local retrieval

# Run semantic search on the local FAISS index.
# Example: "Red Planet" -> passages semantically related to Mars.
def crag_semantic_search(query, top_k):
    query_emb = semb_model.encode(query, convert_to_tensor=False).astype("float32")
    query_emb = np.expand_dims(query_emb, axis=0)

    # Normalize embeddings so FAISS inner product works like cosine similarity.
    faiss.normalize_L2(query_emb)

    scores, indices = faiss_index.search(query_emb, top_k)

    hits = []
    for rank, (corpus_id, score) in enumerate(zip(indices[0], scores[0]), start=1):
        if int(corpus_id) < 0:
            continue

        hits.append({
            "corpus_id": int(corpus_id),
            "score": float(score),
            "semantic_rank": rank,
            "semantic_score": float(score),
        })

    return hits


# Fuse semantic and BM25 rankings with Reciprocal Rank Fusion.
# Example: if "Mars" is high in both rankings, it becomes even stronger.
def crag_reciprocal_rank_fusion(semantic_hits, bm25_hits, top_k=12, rrf_k=60):
    fused_scores = defaultdict(float)
    details = defaultdict(dict)

    # Add contribution from semantic ranking.
    for rank, hit in enumerate(semantic_hits, start=1):
        corpus_id = int(hit["corpus_id"])
        fused_scores[corpus_id] += 1.0 / (rrf_k + rank)
        details[corpus_id].update({
            "semantic_rank": rank,
            "semantic_score": float(hit.get("score", 0.0)),
        })

    # Add contribution from BM25 ranking.
    for rank, hit in enumerate(bm25_hits, start=1):
        corpus_id = int(hit["corpus_id"])
        fused_scores[corpus_id] += 1.0 / (rrf_k + rank)
        details[corpus_id].update({
            "bm25_rank": rank,
            "bm25_score": float(hit.get("score", 0.0)),
        })

    ranked = sorted(fused_scores.items(), key=lambda item: item[1], reverse=True)
    if not ranked:
        return []

    # Normalize scores so the best fused result has score 1.0.
    max_score = ranked[0][1]

    hits = []
    for rank, (corpus_id, raw_score) in enumerate(ranked[:top_k], start=1):
        hit = {
            "corpus_id": int(corpus_id),
            "score": float(raw_score / max_score),
            "source_label": chr(64 + rank) if rank <= 26 else f"S{rank}",
        }
        hit.update(details[corpus_id])
        hits.append(hit)

    return hits


# Add readable passage information to a retrieved hit.
# Example: corpus_id=42 -> title, paragraph_id, passage text.
def crag_enrich_hit(hit):
    corpus_id = int(hit["corpus_id"])
    meta = passage_metadata[corpus_id]

    enriched = dict(hit)
    enriched.update({
        "title": meta.get("title", ""),
        "paragraph_id": meta.get("paragraph_id", ""),
        "passage_text": passages[corpus_id],
    })

    return enriched


# Convert retrieved hits into the context string given to the LLM.
# Example: hit about Mars -> "Context Source A ... Mars is often called..."
def crag_format_context_from_hits(hits):
    blocks = []

    for idx, hit in enumerate(hits, start=1):
        enriched = crag_enrich_hit(hit)

        label = enriched.get(
            "source_label",
            chr(64 + idx) if idx <= 26 else f"S{idx}"
        )

        blocks.append(
            f"Context Source {label}\n"
            f"Title: {enriched['title']} | "
            f"Paragraph: {enriched['paragraph_id']} | "
            f"Score: {float(enriched.get('score', 0.0)):.4f}\n"
            f"{enriched['passage_text']}"
        )

    return "\n\n".join(blocks)


# Try local retrieval using semantic search, BM25, or both.
# Example: question about "Red Planet" -> retrieve local passages about Mars.
def crag_try_local_retrieval(question, config):
    top_k_context = int(config.get("top_k_context", 12))
    final_top_k = int(config.get("final_top_k_context", 3))

    semantic_hits = []
    bm25_hits = []

    # Semantic retrieval: good for meaning-based matches.
    if config.get("use_semantic_retrieval", True):
        semantic_query = crag_build_local_query(
            question,
            mode=config.get("fusion_semantic_query_mode", "question_only"),
        )

        semantic_hits = crag_semantic_search(
            semantic_query,
            top_k=int(config.get("fusion_semantic_top_k", top_k_context)),
        )

    # BM25 retrieval: good for exact keyword matches.
    if config.get("use_bm25_retrieval", True) and "bm25_search" in globals():
        bm25_query = crag_build_local_query(
            question,
            mode=config.get("fusion_bm25_query_mode", "question_only"),
        )

        bm25_hits = bm25_search(
            bm25_query,
            top_k=int(config.get("fusion_bm25_top_k", top_k_context)),
        )

    # If both retrievers worked, combine their rankings.
    if semantic_hits and bm25_hits:
        hits = crag_reciprocal_rank_fusion(
            semantic_hits,
            bm25_hits,
            top_k=top_k_context,
            rrf_k=int(config.get("fusion_rrf_k", 60)),
        )

    # If only semantic search worked, use semantic hits.
    elif semantic_hits:
        hits = semantic_hits[:top_k_context]
        for idx, hit in enumerate(hits, start=1):
            hit["source_label"] = chr(64 + idx) if idx <= 26 else f"S{idx}"

    # If only BM25 worked, use BM25 hits.
    elif bm25_hits:
        hits = bm25_hits[:top_k_context]
        for idx, hit in enumerate(hits, start=1):
            hit["source_label"] = chr(64 + idx) if idx <= 26 else f"S{idx}"

    # If no local evidence is found, return empty context.
    else:
        return ""

    # Keep only the final best passages for the LLM.
    return crag_format_context_from_hits(hits[:final_top_k])


In [ ]:
# Answerability router

def crag_run_local_answerability_router(question, local_context, tokenizer, model, config):
    valid_ids = [opt.id for opt in question.options]

    # Build the router prompt
    system_msg, user_msg = crag_prompt_answerability_router(question, local_context, config)

    prompt = tokenizer.apply_chat_template(
        [{"role": "system", "content": system_msg}, {"role": "user", "content": user_msg}],
        tokenize=False,
        add_generation_prompt=True,
    )

    # {"answerable": true, "answer": 1, "confidence": 0.85, "reason": "..."}
    raw_output = generate_answer(
        prompt, tokenizer, model,
        do_sample=False, max_new_tokens=96, temperature=1.0,
    )

    return crag_parse_answerability_router_output(raw_output, valid_ids)


In [ ]:
# Web search

def crag_load_duckduckgo_search_tool(num_results):
    try:
        # First choice: LangChain DuckDuckGo wrapper.
        # Example: query "Red Planet Mars" -> JSON search results from DuckDuckGo.
        from langchain_community.tools import DuckDuckGoSearchResults
        return DuckDuckGoSearchResults(num_results=num_results, output_format="json")
    except Exception as exc_community:
        community_error = exc_community

    try:
        # Second choice: ddgs package.
        # Example: if LangChain is unavailable, use DDGS directly
        from ddgs import DDGS
        class DDGSSearchWrapper:
            def __init__(self, max_results): self.max_results = max_results

            # Run a DuckDuckGo text search and return JSON-like results.
            # Example: "Red Planet Mars" -> [{"title": ..., "link": ..., "snippet": ...}]
            def run(self, query):
                results = []
                with DDGS() as ddgs:
                    for item in ddgs.text(query, max_results=self.max_results):
                        results.append({"title": item.get("title", ""), "link": item.get("href", item.get("url", "")), "snippet": item.get("body", "")})
                return json.dumps(results, ensure_ascii=False)
        return DDGSSearchWrapper(num_results)
    except Exception as exc_ddgs:
        ddgs_error = exc_ddgs

    try:
        # Third choice: older duckduckgo_search package.
        # Example: use this only if the first two backends fail.
        from duckduckgo_search import DDGS
        class DuckDuckGoSearchWrapper:
            def __init__(self, max_results): self.max_results = max_results
            def run(self, query):
                results = []
                with DDGS() as ddgs:
                    for item in ddgs.text(query, max_results=self.max_results):
                        results.append({"title": item.get("title", ""), "link": item.get("href", item.get("url", "")), "snippet": item.get("body", "")})
                return json.dumps(results, ensure_ascii=False)
        return DuckDuckGoSearchWrapper(num_results)
    except Exception as exc_old:
        old_error = exc_old

    raise ImportError(
        "Could not load any DuckDuckGo search backend.\n"
        f"langchain_community error: {community_error}\n"
        f"ddgs error: {ddgs_error}\n"
        f"duckduckgo_search error: {old_error}"
    )


# Example: "Which planet is known as the Red Planet?" -> "Red Planet planet known as Mars"
def crag_rewrite_query_for_web(question, tokenizer, model, config):
    plain_query = crag_build_plain_web_query(question)

    if not config.get("web_use_llm_query_rewrite", False):
        return plain_query

    system_msg, user_msg = crag_prompt_query_rewrite(question)

    try:
        prompt = tokenizer.apply_chat_template(
            [{"role": "system", "content": system_msg}, {"role": "user", "content": user_msg}],
            tokenize=False, add_generation_prompt=True,
        )
        raw_output = generate_answer(
            prompt, tokenizer, model,
            do_sample=False,
            max_new_tokens=int(config.get("web_query_rewrite_max_new_tokens", 32)),
            temperature=1.0,
        )

        # Example: "Query: Mars Red Planet" -> "Mars Red Planet"
        rewritten = crag_normalize_for_query(raw_output)
        rewritten = re.sub(r"^\s*(query|search query|rewritten query)\s*:\s*", "", rewritten, flags=re.I)
        if not rewritten or len(rewritten.split()) < 3:
            return plain_query
        return rewritten
    except Exception:
        return plain_query


def crag_web_search_once(query, question, config, query_type="general", option=None):
    search_top_k = int(config.get("web_search_top_k", 4))
    fetch_k = int(config.get("web_search_fetch_k", search_top_k))

    search = crag_load_duckduckgo_search_tool(max(search_top_k, fetch_k))
    raw_results = search.run(query)
    parsed_results = crag_parse_web_results(raw_results)

    results = []
    seen_urls = set()

    for result in parsed_results:
        url = result.get("link", "")
        if not url or url in seen_urls:
            continue
        seen_urls.add(url)
        item = dict(result)
        item["query_type"] = query_type

        if option is not None:
            item["option_id"] = int(option.id)
            item["option_text"] = option.text
            item["option_evidence_score"] = crag_score_option_against_text(option.text, crag_result_text(item))
        else:
            item["option_id"] = None
            item["option_text"] = None
            item["option_evidence_score"] = 0.0

        item["web_score"] = crag_score_web_result(item, question, config)
        item["domain"] = crag_get_domain(item.get("link", ""))
        item["domain_bonus"] = crag_domain_quality_bonus(item.get("link", ""), config)
        item["final_web_score"] = item["web_score"] + item["domain_bonus"]
        results.append(item)

    if option is not None:
        results.sort(
            key=lambda x: (
                x["option_evidence_score"],
                x.get("final_web_score", x.get("web_score", 0.0)),
            ),
            reverse=True,
        )
    else:
        results.sort(key=lambda x: x.get("final_web_score", x.get("web_score", 0.0)), reverse=True)

    return results[:search_top_k]


def crag_perform_classic_web_search(question, tokenizer, model, config):
    query = crag_rewrite_query_for_web(question, tokenizer, model, config)
    results = crag_web_search_once(query, question, config, query_type="general")

    min_score = float(config.get("web_min_relevance_score", 0.0))
    selected = [r for r in results if r.get("final_web_score", r.get("web_score", 0.0)) >= min_score]
    if not selected and results:
        selected = results[:1]

    return crag_format_web_context(
        selected, heading="Corrective RAG fast web evidence", query=query,
    )


#   "Which planet is known as the Red Planet? Earth"
#   "Which planet is known as the Red Planet? Mars"
#   "Which planet is known as the Red Planet? Jupiter"
#   "Which planet is known as the Red Planet? Venus"
def crag_perform_option_wise_web_search(question, tokenizer, model, config):
    general_query = crag_rewrite_query_for_web(question, tokenizer, model, config)
    general_results = crag_web_search_once(general_query, question, config, query_type="general")

    option_results = {}
    for opt in question.options:
        option_query = crag_build_option_web_query(question, opt)
        option_results[int(opt.id)] = crag_web_search_once(
            option_query, question, config, query_type="option_wise", option=opt,
        )

    option_evidence = crag_build_option_evidence_summary(question, option_results)
    selected = crag_select_balanced_optionwise_results(question, general_results, option_results, config)

    return crag_format_web_context(
        selected,
        heading="Corrective RAG balanced option-wise web evidence",
        query=general_query,
        option_evidence=option_evidence,
    )


def crag_perform_web_search(question, tokenizer, model, config):
    if config.get(
      "web_use_option_wise_search", False):
        return crag_perform_option_wise_web_search(question, tokenizer, model, config)
    return crag_perform_classic_web_search(question, tokenizer, model, config)


In [ ]:
# Answer generation

def crag_answer_from_context(question, context, tokenizer, model, config):
    valid_ids = [opt.id for opt in question.options]
    system_msg, user_msg = build_rag_prompt(question.text, question.options, context)
    prompt = tokenizer.apply_chat_template(
        [{"role": "system", "content": system_msg}, {"role": "user", "content": user_msg}],
        tokenize=False, add_generation_prompt=True,
    )
    raw_output = generate_answer(
        prompt, tokenizer, model,
        do_sample=globals().get("CONFIG", {}).get("do_sample", False),
        max_new_tokens=int(config.get("rag_max_new_tokens", 64)),
        temperature=globals().get("CONFIG", {}).get("temperature", 1.0),
    )
    return extract_option_id(raw_output, valid_ids), raw_output


def crag_answer_without_context(question, tokenizer, model):
    valid_ids = [opt.id for opt in question.options]
    system_msg, user_msg = build_prompt(question.text, question.options)
    prompt = tokenizer.apply_chat_template(
        [{"role": "system", "content": system_msg}, {"role": "user", "content": user_msg}],
        tokenize=False, add_generation_prompt=True,
    )
    base_config = globals().get("CONFIG", {})
    raw_output = generate_answer(
        prompt, tokenizer, model,
        do_sample=base_config.get("do_sample", False),
        max_new_tokens=int(base_config.get("max_new_tokens", 15)),
        temperature=base_config.get("temperature", 1.0),
    )
    return parse_answer(raw_output, valid_ids), raw_output


def crag_repair_answer_with_local_llm(question, raw_output, tokenizer, model, config):
    system_msg, user_msg = crag_prompt_repair_answer(question, raw_output)

    prompt = tokenizer.apply_chat_template(
        [{"role": "system", "content": system_msg}, {"role": "user", "content": user_msg}],
        tokenize=False, add_generation_prompt=True,
    )
    repaired_output = generate_answer(
        prompt, tokenizer, model,
        do_sample=False,
        max_new_tokens=int(config.get("answer_repair_max_new_tokens", 12)),
        temperature=1.0,
    )
    return extract_option_id(repaired_output, [opt.id for opt in question.options])


In [ ]:
# Main function

def get_crag_rag_answer_logged(question, tokenizer, model, max_retries=1):
    config = dict(CRAG_CONFIG)
    valid_ids = [opt.id for opt in question.options]
    forced_news_web = crag_should_force_web(question, config)

    if forced_news_web:
        try:
            context = crag_perform_web_search(question, tokenizer, model, config)
        except Exception:
            context = ""
    else:
        local_context = crag_try_local_retrieval(question, config)

        if len(local_context.strip()) < int(config.get("local_min_context_chars", 250)):
            route = "incorrect"
        else:
            router_payload = crag_run_local_answerability_router(
                question, local_context, tokenizer, model, config,
            )
            confidence_score = float(router_payload.get("confidence", 0.0))
            local_router_answer = router_payload.get("answer")
            min_conf = float(config.get("local_answerability_min_confidence", 0.65))

            if (
                router_payload.get("answerable", False)
                and local_router_answer in valid_ids
                and confidence_score >= min_conf
            ):
                route = "correct"
            else:
                route = "incorrect"

        if route == "correct":
            context = local_context
        else:
            try:
                context = crag_perform_web_search(question, tokenizer, model, config)
            except Exception:
                context = ""

    if context.strip():
        answer_id, raw_output = crag_answer_from_context(
            question, context, tokenizer, model, config,
        )
    else:
        answer_id, raw_output = crag_answer_without_context(question, tokenizer, model)

    if answer_id is None:
        if config.get("enable_answer_repair", True):
            repaired = crag_repair_answer_with_local_llm(
                question, raw_output, tokenizer, model, config,
            )
            if repaired in valid_ids:
                answer_id = repaired

        if answer_id is None and config.get("enable_direct_option_match", True):
            fallback = crag_option_text_fallback(question, context, raw_output)
            if fallback in valid_ids:
                answer_id = fallback

        if answer_id is None:
            answer_id = valid_ids[0]

    return answer_id, {}


In [ ]:
# Activation

def get_crag_rag_answer(question, tokenizer, model, max_retries=1):
    answer_id, _ = get_crag_rag_answer_logged(
        question, tokenizer, model, max_retries=max_retries,
    )
    return answer_id


def activate_local_first_crag(
    debug=True,       # kept for API compatibility
    min_confidence=0.65,
    **kwargs,
):
    global get_rag_answer_logged
    global get_rag_answer

    CRAG_CONFIG.update(kwargs)
    CRAG_CONFIG["local_answerability_min_confidence"] = min_confidence

    if "BASE_RAG_CONFIG" in globals():
        BASE_RAG_CONFIG["active_rag_variant"] = "local_first_crag"

    get_rag_answer_logged = get_crag_rag_answer_logged
    get_rag_answer = get_crag_rag_answer


def activate_crag_rag_variant(**kwargs):
    debug = kwargs.pop("debug", CRAG_CONFIG.get("debug", True))
    min_confidence = kwargs.pop(
        "min_confidence",
        CRAG_CONFIG.get("local_answerability_min_confidence", 0.65),
    )
    activate_local_first_crag(debug=debug, min_confidence=min_confidence, **kwargs)


Final setup


In [ ]:
# RAG configurations registry

RAG_CONFIGURATIONS = {
    "llm": {
        "description": "LLM-only baseline"
    },
    "rag_baseline_question_only": {
        "description": "Baseline RAG with question-only retrieval"
    },
    "rag_question_options": {
        "description": "Baseline RAG with question + options retrieval"
    },
    "reliable_rag": {
        "description": "Reliable-RAG with relevance filtering"
    },
    "query_transformation_rag": {
        "description": "Query Transformation RAG"
    },
    "hyde_rag": {
        "description": "HyDE-RAG"
    },
    "rse_rag": {
        "description": "Relevant Segment Extraction RAG"
    },
    "contextual_compression_rag": {
        "description": "Contextual Compression RAG"
    },
    "rse_contextual_compression_rag": {
        "description": "RSE + Contextual Compression RAG"
    },
    "fusion_rag": {
        "description": "Fusion Retrieval RAG"
    },
    "fusion_cross_encoder_rag": {
        "description": "Fusion Retrieval + Cross-Encoder Reranking RAG"
    },
    "adaptive_best_config_rag": {
        "description": "Adaptive Best-Config Router"
    },
    "self_rag": {
        "description": "Self-RAG"
    },
    "crag": {
        "description": "Corrective RAG"
    },
}


# ============================================================
# Model-aware best RAG configuration selector
# Loads best cached benchmark configs for the current model
# and activates the best available setup for the selected RAG type
# ============================================================

import os
import json
import hashlib
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

BENCHMARK_DIR = "/content/gdrive/MyDrive/NLP/PoliMillionaire/ASSO_RAG/Benchmark"

SOLVED_BENCHMARK_PATH = os.path.join(
    BENCHMARK_DIR,
    "llm_zero_shot_wrong_questions_solved.json"
)

CACHE_ROOT = os.path.join(
    BENCHMARK_DIR,
    "benchmark_cache"
)

BEST_CONFIG_ROOT = os.path.join(
    BENCHMARK_DIR,
    "best_configs"
)

CACHE_GROUPS = {
    "rag_baseline_reliable": "rag_benchmark_summary.json",
    "query_enhancement": "query_enhancement_benchmark_summary.json",
    "context_enrichment": "context_enrichment_benchmark_summary.json",
    "advanced_retrieval": "advanced_retrieval_benchmark_summary.json",
    "advanced_architecture": "advanced_architecture_benchmark_summary.json",
}


# ------------------------------------------------------------
# Hash helpers
# ------------------------------------------------------------

def md5_json(obj):
    text = json.dumps(obj, sort_keys=True, ensure_ascii=False)
    return hashlib.md5(text.encode("utf-8")).hexdigest()


def md5_file(path):
    h = hashlib.md5()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


def get_model_metadata_for_best_config():
    return {
        "model_id": CONFIG.get("model_id"),
        "quantization_bits": CONFIG.get("quantization_bits"),
        "prompt_strategy": CONFIG.get("prompt_strategy"),
        "temperature": CONFIG.get("temperature"),
        "do_sample": CONFIG.get("do_sample"),
        "max_new_tokens": CONFIG.get("max_new_tokens"),
        "reasoning_max_new_tokens": CONFIG.get("reasoning_max_new_tokens"),
        "model_kwargs": CONFIG.get("model_kwargs", {}),
    }


def get_current_benchmark_and_model_hashes():
    benchmark_hash = md5_file(SOLVED_BENCHMARK_PATH)
    model_metadata = get_model_metadata_for_best_config()
    model_hash = md5_json(model_metadata)

    return benchmark_hash, model_hash, model_metadata


# ------------------------------------------------------------
# Family / playable key mapping
# ------------------------------------------------------------

def get_family_from_summary_row(row):
    technology = row.get("technology", "")
    method = row.get("method", "")
    config_name = row.get("config_name", "")

    if technology == "baseline_rag":
        if "qonly" in config_name or "question_only" in config_name:
            return "rag_baseline_question_only"
        return "rag_question_options"

    if technology == "reliable_rag":
        return "reliable_rag"

    if technology == "query_transformation_rag" or method == "query_transformation":
        return "query_transformation_rag"

    if technology == "hyde_rag" or method == "hyde":
        return "hyde_rag"

    if technology == "rse_rag" or method == "rse":
        return "rse_rag"

    if technology == "contextual_compression_rag" or method == "contextual_compression":
        return "contextual_compression_rag"

    if technology == "rse_contextual_compression_rag" or method == "rse_contextual_compression":
        return "rse_contextual_compression_rag"

    if technology in {"semantic_retrieval", "bm25_retrieval", "fusion_retrieval"}:
        return "fusion_rag"

    if technology == "fusion_rse_rag" or method == "fusion_rse":
        return "fusion_rag"

    if technology == "fusion_cross_encoder_rag" or method == "fusion_cross_encoder":
        return "fusion_cross_encoder_rag"

    if technology == "fusion_cross_encoder_rse_rag" or method == "fusion_cross_encoder_rse":
        return "fusion_cross_encoder_rag"

    if technology == "adaptive_best_config_rag" or method == "adaptive_best_config_rag":
        return "adaptive_best_config_rag"

    if technology == "self_rag" or method == "self_rag":
        return "self_rag"

    if technology == "crag" or method == "crag":
        return "crag"

    return technology or method or config_name


def pretty_family_name(key):
    mapping = {
        "rag_baseline_question_only": "Baseline RAG question-only",
        "rag_question_options": "Baseline RAG question + options",
        "reliable_rag": "Reliable-RAG",
        "query_transformation_rag": "Query Transformation RAG",
        "hyde_rag": "HyDE-RAG",
        "rse_rag": "RSE-RAG",
        "contextual_compression_rag": "Contextual Compression RAG",
        "rse_contextual_compression_rag": "RSE + Contextual Compression RAG",
        "fusion_rag": "Fusion / Advanced Retrieval RAG",
        "fusion_cross_encoder_rag": "Fusion + Cross-Encoder RAG",
        "adaptive_best_config_rag": "Adaptive Best-Config Router",
        "self_rag": "Self-RAG",
        "crag": "CRAG",
    }

    return mapping.get(key, key)


# ------------------------------------------------------------
# Load cached summaries for current model
# ------------------------------------------------------------

def load_cached_benchmark_summaries_for_current_model(verbose=True):
    """
    Loads cached benchmark summaries.

    First it tries the current model_hash.
    If nothing is found, it automatically searches existing model_* folders
    for the same benchmark_hash and uses the available cached summaries.

    This avoids losing cached benchmark results when CONFIG changes slightly
    after the benchmark, for example max_new_tokens or prompt_strategy.
    """
    import glob

    benchmark_hash, current_model_hash, model_metadata = get_current_benchmark_and_model_hashes()

    all_summaries = []
    missing_groups = []
    used_paths = []
    discovered_model_hashes = set()

    if verbose:
        print("Current model:")
        print(model_metadata.get("model_id"))
        print()
        print("Benchmark hash:")
        print(benchmark_hash)
        print()
        print("Current model hash:")
        print(current_model_hash)

    for group_name, summary_filename in CACHE_GROUPS.items():
        expected_path = os.path.join(
            CACHE_ROOT,
            group_name,
            f"benchmark_{benchmark_hash}",
            f"model_{current_model_hash}",
            "summary",
            summary_filename,
        )

        summary_path = None

        # 1. Try exact current model hash first
        if os.path.exists(expected_path):
            summary_path = expected_path

        # 2. Fallback: search any model_* folder for this benchmark/group
        else:
            candidate_paths = glob.glob(
                os.path.join(
                    CACHE_ROOT,
                    group_name,
                    f"benchmark_{benchmark_hash}",
                    "model_*",
                    "summary",
                    summary_filename,
                )
            )

            if candidate_paths:
                # If multiple exist, use the most recently modified summary.
                candidate_paths = sorted(
                    candidate_paths,
                    key=lambda p: os.path.getmtime(p),
                    reverse=True,
                )
                summary_path = candidate_paths[0]

        if summary_path is None:
            missing_groups.append(group_name)

            if verbose:
                print(f"\nMissing summary for: {group_name}")
                print("Expected:")
                print(expected_path)

            continue

        used_paths.append(summary_path)

        # Extract discovered model hash from path
        path_parts = Path(summary_path).parts
        for part in path_parts:
            if part.startswith("model_"):
                discovered_model_hashes.add(part.replace("model_", ""))

        with open(summary_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        summaries = data.get("summaries", [])

        for item in summaries:
            row = dict(item)
            row["benchmark_group"] = group_name
            row["summary_path"] = summary_path
            row["play_key"] = get_family_from_summary_row(row)
            all_summaries.append(row)

        if verbose:
            exact = summary_path == expected_path
            print(f"\nLoaded {len(summaries)} configs from {group_name}")
            print(f"Exact current hash match: {exact}")
            print(f"Summary path: {summary_path}")

    if verbose:
        print("\nLoaded summary rows:")
        print(len(all_summaries))

        print("\nModel hashes used from cache:")
        if discovered_model_hashes:
            for h in sorted(discovered_model_hashes):
                print(f"- {h}")
        else:
            print("- none")

        if missing_groups:
            print("\nMissing groups:")
            for group in missing_groups:
                print(f"- {group}")

    return {
        "benchmark_hash": benchmark_hash,
        "model_hash": current_model_hash,
        "model_metadata": model_metadata,
        "summaries": all_summaries,
        "missing_groups": missing_groups,
        "used_summary_paths": used_paths,
        "discovered_model_hashes": sorted(discovered_model_hashes),
    }

# ------------------------------------------------------------
# Select best configs
# ------------------------------------------------------------

def is_forbidden_best_config(row):
    """
    Returns True for benchmark configs that must never be selected
    as best playable configurations.
    """

    play_key = row.get("play_key", "")
    technology = row.get("technology", "")
    method = row.get("method", "")
    config_name = row.get("config_name", "")
    pretty_name = row.get("pretty_name", "")

    text = " ".join([
        str(play_key),
        str(technology),
        str(method),
        str(config_name),
        str(pretty_name),
    ]).lower()

    # CRAG local-only must never be selected.
    # It is not a real final CRAG configuration for gameplay,
    # because CRAG should include corrective/web fallback behavior.
    if play_key == "crag":
        local_only_markers = [
            "local_only",
            "local-only",
            "local only",
            "no_web",
            "no-web",
            "without_web",
            "without web",
            "web_fallback_false",
            "use_web_fallback=false",
            "use_web_fallback_false",
        ]

        if any(marker in text for marker in local_only_markers):
            return True

        # Extra safety: if the benchmark row explicitly says web fallback is disabled
        if row.get("use_web_fallback") is False:
            return True

        if str(row.get("use_web_fallback", "")).lower() == "false":
            return True

    return False


def select_best_configs_by_play_key(summaries):
    """
    Selects best config for each playable RAG key.

    Sorting rule:
    1. highest accuracy
    2. lowest average time

    Forbidden configs are removed before sorting.
    Example: CRAG local-only is skipped, so if it was first,
    the selector automatically takes the second-best valid CRAG config.
    """
    best_by_key = {}

    play_keys = sorted(set(row["play_key"] for row in summaries))

    for play_key in play_keys:
        rows = [
            row for row in summaries
            if row["play_key"] == play_key
        ]

        valid_rows = [
            row for row in rows
            if not is_forbidden_best_config(row)
        ]

        if not valid_rows:
            print(f"Warning: no valid configs found for {play_key}.")
            print("All configs were forbidden or unavailable.")
            continue

        best = sorted(
            valid_rows,
            key=lambda x: (
                float(x.get("accuracy", 0.0)),
                -float(x.get("avg_time_sec", 999999.0)),
            ),
            reverse=True,
        )[0]

        best_by_key[play_key] = best

    return best_by_key

def build_best_config_table(best_by_key):
    table_rows = []

    for play_key, row in best_by_key.items():
        total = int(row.get("total_questions", 0))
        correct = int(row.get("correct", 0))
        accuracy = float(row.get("accuracy", 0.0))
        avg_time = float(row.get("avg_time_sec", 0.0))

        table_rows.append({
            "RAG key": play_key,
            "Technology": pretty_family_name(play_key),
            "Best configuration": row.get("pretty_name", row.get("config_name")),
            "Config name": row.get("config_name"),
            "Correct": f"{correct}/{total}",
            "Accuracy": f"{accuracy * 100:.0f}%",
            "Avg. time": f"{avg_time:.2f}s",
            "_accuracy_sort": accuracy,
            "_time_sort": avg_time,
        })

    if not table_rows:
        return pd.DataFrame()

    df = pd.DataFrame(table_rows)

    df = df.sort_values(
        by=["_accuracy_sort", "_time_sort"],
        ascending=[False, True],
    ).drop(columns=["_accuracy_sort", "_time_sort"])

    return df


def save_best_configs_for_current_model(best_by_key):
    benchmark_hash, model_hash, model_metadata = get_current_benchmark_and_model_hashes()

    best_config_dir = os.path.join(
        BEST_CONFIG_ROOT,
        f"benchmark_{benchmark_hash}",
        f"model_{model_hash}",
    )

    os.makedirs(best_config_dir, exist_ok=True)

    best_json_path = os.path.join(best_config_dir, "best_configs_by_play_key.json")
    best_csv_path = os.path.join(best_config_dir, "best_configs_by_play_key.csv")

    best_rows = list(best_by_key.values())
    best_df = build_best_config_table(best_by_key)

    with open(best_json_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "benchmark_hash": benchmark_hash,
                "model_hash": model_hash,
                "model_metadata": model_metadata,
                "best_configs_by_play_key": best_by_key,
                "best_configs": best_rows,
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    best_df.to_csv(best_csv_path, index=False)

    return best_json_path, best_csv_path


def load_best_configs_for_current_model(
    verbose=True,
    display_table=True,
    as_dataframe=False,
):
    """
    Loads best cached benchmark configs for the current model.

    Standard behavior:
        as_dataframe=False -> returns dict {play_key: best_row}

    Adaptive-compatible behavior:
        as_dataframe=True -> returns a DataFrame with one row per best technology,
        including technology, config_name, pretty_name, accuracy, avg_time_sec.
    """
    loaded = load_cached_benchmark_summaries_for_current_model(verbose=verbose)
    summaries = loaded["summaries"]

    if not summaries:
        if verbose:
            print("\nNo cached benchmark summaries found for current model.")

        if as_dataframe:
            return pd.DataFrame(
                columns=[
                    "technology",
                    "config_name",
                    "pretty_name",
                    "accuracy",
                    "avg_time_sec",
                    "model_id",
                    "quantization_bits",
                    "source_summary_path",
                ]
            )

        return {}

    best_by_key = select_best_configs_by_play_key(summaries)

    if display_table:
        best_df = build_best_config_table(best_by_key)
        print("\nBest configurations available for current model")
        display(best_df)

    best_json_path, best_csv_path = save_best_configs_for_current_model(best_by_key)

    if verbose:
        print("\nSaved best configs JSON:")
        print(best_json_path)
        print("\nSaved best configs CSV:")
        print(best_csv_path)

    if not as_dataframe:
        return best_by_key

    rows = []

    for play_key, row in best_by_key.items():
        item = dict(row)

        rows.append({
            **item,
            "technology": play_key,
            "config_name": item.get("config_name"),
            "pretty_name": item.get("pretty_name", item.get("config_name")),
            "accuracy": pd.to_numeric(item.get("accuracy"), errors="coerce"),
            "avg_time_sec": pd.to_numeric(item.get("avg_time_sec"), errors="coerce"),
            "model_id": CONFIG.get("model_id"),
            "quantization_bits": CONFIG.get("quantization_bits"),
            "source_summary_path": item.get("summary_path"),
        })

    df = pd.DataFrame(rows)

    if df.empty:
        return df

    if "ADAPTIVE_ALLOWED_TECHNOLOGIES" in globals():
        df = df[df["technology"].isin(ADAPTIVE_ALLOWED_TECHNOLOGIES)].copy()

    if "ADAPTIVE_EXCLUDED_TECHNOLOGIES" in globals():
        df = df[~df["technology"].isin(ADAPTIVE_EXCLUDED_TECHNOLOGIES)].copy()

    df["accuracy"] = pd.to_numeric(df["accuracy"], errors="coerce")
    df["avg_time_sec"] = pd.to_numeric(df["avg_time_sec"], errors="coerce")

    df = df.sort_values(
        ["accuracy", "avg_time_sec"],
        ascending=[False, True],
        na_position="last",
    ).reset_index(drop=True)

    return df


# ------------------------------------------------------------
# Debug print helper
# ------------------------------------------------------------

def _print_logged_rag_details(answer_id, details):
    """
    Print a compact debug summary for logged RAG variants.
    """
    if not BASE_RAG_CONFIG.get("debug", True):
        return

    details = details or {}

    print("\n=== RAG Debug ===")
    print(f"Final answer id: {answer_id}")

    for key in [
        "prompt_mode",
        "context_status",
        "retrieval_mode",
        "selection_mode",
        "selection_status",
        "relevant_source_labels",
        "active_rag_variant",
    ]:
        if key in details:
            print(f"{key}: {details.get(key)}")

    timing_keys = [
        "retrieval_decision_time_sec",
        "retrieval_time_sec",
        "relevance_time_sec",
        "rerank_time_sec",
        "selection_time_sec",
        "rse_time_sec",
        "compression_time_sec",
        "generation_time_sec",
        "support_time_sec",
        "utility_time_sec",
        "total_answer_time_sec",
    ]

    available_timings = [key for key in timing_keys if key in details]

    if available_timings:
        print("\nTiming:")
        for key in available_timings:
            value = details.get(key)
            if isinstance(value, (int, float)):
                print(f"{key}: {value:.2f}s")

    hits = details.get("hits") or details.get("candidate_hits") or []

    if hits:
        print("\nRetrieved sources:")

        for i, hit in enumerate(hits, start=1):
            label = hit.get("source_label", chr(64 + i))
            title = hit.get("title")
            paragraph_id = hit.get("paragraph_id")
            score = hit.get("score")

            if isinstance(score, (int, float)):
                print(f"[{label}] {title} | paragraph {paragraph_id} | score={score:.4f}")
            else:
                print(f"[{label}] {title} | paragraph {paragraph_id}")

    context = details.get("context")

    if context:
        print("\nContext passed to answer prompt:")
        print("-" * 80)
        print(context)

    attempts = details.get("attempts", [])

    if attempts:
        print("\nAnswer attempts:")
        for attempt in attempts:
            print(f"Attempt {attempt.get('attempt')}:")
            print(f"Raw output: {attempt.get('raw_output')}")
            print(f"Parsed answer: {attempt.get('parsed_answer')}")

    print("=== End RAG Debug ===\n")


def _logged_rag_answer_dispatcher(question, tokenizer, model, max_retries=3):
    """
    Dispatch play_game_rag to the currently active logged RAG variant.
    """
    answer_id, details = get_rag_answer_logged(
        question,
        tokenizer,
        model,
        max_retries=max_retries,
    )

    _print_logged_rag_details(answer_id, details)
    return answer_id


def _activate_logged_variant(activation_fn, **kwargs):
    global get_rag_answer

    activation_fn(**kwargs)
    get_rag_answer = _logged_rag_answer_dispatcher


# ------------------------------------------------------------
# Basic baseline activation helpers
# ------------------------------------------------------------

def _activate_baseline_question_only_with_params(
    top_k_context=4,
    rag_max_new_tokens=64,
    debug=True,
):
    global retrieve_local_wikipedia_context
    global get_rag_answer

    retrieve_local_wikipedia_context = BASELINE_RETRIEVE_LOCAL_WIKIPEDIA_CONTEXT
    get_rag_answer = BASELINE_GET_RAG_ANSWER

    BASE_RAG_CONFIG.update({
        "active_rag_variant": "baseline_question_only",
        "retrieval_method": "wikipedia_local",
        "retrieval_query_mode": "question_only",
        "top_k_context": top_k_context,
        "rag_max_new_tokens": rag_max_new_tokens,
        "debug": debug,
    })


def _activate_question_options_with_params(
    top_k_context=4,
    rag_max_new_tokens=64,
    debug=True,
):
    global retrieve_local_wikipedia_context
    global get_rag_answer

    retrieve_local_wikipedia_context = retrieve_local_wikipedia_context_with_options
    get_rag_answer = BASELINE_GET_RAG_ANSWER

    BASE_RAG_CONFIG.update({
        "active_rag_variant": "baseline_question_options",
        "retrieval_method": "wikipedia_local_question_options",
        "retrieval_query_mode": "question_plus_options",
        "top_k_context": top_k_context,
        "rag_max_new_tokens": rag_max_new_tokens,
        "debug": debug,
    })


# ------------------------------------------------------------
# Default fallback activations
# Used when no cached best config exists for current model
# ------------------------------------------------------------

def activate_default_configuration(configuration_name):
    """
    Fallback activations used when no cached benchmark result exists.
    """
    if configuration_name == "llm":
        return None

    if configuration_name == "rag_baseline_question_only":
        return _activate_baseline_question_only_with_params(
            top_k_context=4,
            rag_max_new_tokens=64,
            debug=True,
        )

    if configuration_name == "rag_question_options":
        return _activate_question_options_with_params(
            top_k_context=4,
            rag_max_new_tokens=64,
            debug=True,
        )

    if configuration_name == "reliable_rag":
        return _activate_logged_variant(
            activate_reliable_rag_variant,
            top_k_context=4,
            retrieval_query_mode="question_plus_options",
        )

    if configuration_name == "query_transformation_rag":
        return _activate_logged_variant(
            activate_query_transformation_rag_variant,
            top_k_context=6,
            top_k_per_query=2,
            max_subqueries=3,
            max_final_sources=3,
            query_transform_original_mode="question_plus_options",
            enable_llm_relevance_check=True,
        )

    if configuration_name == "hyde_rag":
        return _activate_logged_variant(
            activate_hyde_rag_variant,
            top_k_context=4,
            max_final_sources=3,
            hyde_include_options=False,
            hyde_fallback_query_mode="question_only",
            enable_llm_relevance_check=False,
        )

    if configuration_name == "rse_rag":
        return _activate_logged_variant(
            activate_rse_rag_variant,
            top_k_context=6,
            rse_neighbor_window=1,
            rse_max_segment_length=3,
            rse_max_total_paragraphs=3,
            enable_llm_relevance_check=False,
        )

    if configuration_name == "contextual_compression_rag":
        return _activate_logged_variant(
            activate_rse_compressed_rag_variant,
            top_k_context=6,
            max_final_sources=3,
            enable_rse=False,
            enable_llm_relevance_check=False,
            enable_contextual_compression=True,
        )

    if configuration_name == "rse_contextual_compression_rag":
        return _activate_logged_variant(
            activate_rse_compressed_rag_variant,
            top_k_context=6,
            max_final_sources=3,
            enable_rse=True,
            rse_neighbor_window=1,
            rse_max_segment_length=3,
            rse_max_total_paragraphs=5,
            enable_llm_relevance_check=False,
            enable_contextual_compression=True,
        )

    if configuration_name == "fusion_rag":
        return _activate_logged_variant(
            activate_fusion_rag_variant,
            top_k_context=8,
            final_top_k_context=3,
            fusion_enable_semantic=True,
            fusion_enable_bm25=True,
            fusion_semantic_top_k=20,
            fusion_bm25_top_k=20,
            fusion_rrf_k=60,
            fusion_semantic_query_mode="question_options",
            fusion_bm25_query_mode="question_only",
            fusion_use_rse=False,
            enable_llm_relevance_check=False,
        )

    if configuration_name == "fusion_cross_encoder_rag":
        return _activate_logged_variant(
            activate_fusion_rerank_rag_variant,
            top_k_context=12,
            final_top_k_context=3,
            use_semantic_retrieval=True,
            use_bm25_retrieval=True,
            fusion_semantic_top_k=30,
            fusion_bm25_top_k=30,
            fusion_rrf_k=60,
            fusion_semantic_query_mode="question_options",
            fusion_bm25_query_mode="question_only",
            use_cross_encoder_rerank=True,
            rerank_model_id="cross-encoder/ms-marco-MiniLM-L-6-v2",
            rerank_query_mode="question_options",
            rerank_batch_size=16,
            use_rse=False,
            enable_llm_relevance_check=False,
            load_reranker_now=True,
        )

    if configuration_name == "adaptive_best_config_rag":
        if "activate_adaptive_rag_variant" not in globals():
            raise NameError(
                "activate_adaptive_rag_variant is not defined. "
                "Run the Adaptive Retrieval Router cell after the Final setup cell."
            )

        return _activate_logged_variant(
            activate_adaptive_rag_variant,
            default_profile="balanced",
            refresh_best_configs=True,
            debug=True,
        )

    if configuration_name == "self_rag":
        return _activate_logged_variant(
            activate_self_rag_variant,
            top_k_context=4,
            max_final_contexts=2,
            retrieval_query_mode="question_only",
            debug=True,
        )

    if configuration_name == "crag":
        return activate_crag_rag_variant(
            force_web_for_news=True,
            news_comp_id=5,
            use_semantic_retrieval=True,
            use_bm25_retrieval=True,
            use_cross_encoder_rerank=False,
            local_answerability_min_confidence=0.65,
            local_min_context_chars=250,
            router_max_local_context_chars=3500,
            use_web_fallback=True,
            web_fallback_on_incorrect=True,
            web_search_top_k=4,
            web_search_fetch_k=4,
            web_min_relevance_score=0.0,
            web_use_llm_query_rewrite=False,
            web_use_option_wise_search=False,
            fusion_semantic_query_mode="question_only",
            fusion_bm25_query_mode="question_only",
            top_k_context=12,
            final_top_k_context=3,
            rag_max_new_tokens=64,
            crag_answer_max_retries=1,
            debug=True,
        )

    raise ValueError(f"No fallback activation defined for: {configuration_name}")


# ------------------------------------------------------------
# Apply cached best config
# ------------------------------------------------------------

def activate_best_cached_configuration(configuration_name, best_row):
    """
    Applies the benchmark-derived best configuration for the selected RAG type.
    """
    config_name = best_row.get("config_name")
    pretty_name = best_row.get("pretty_name", config_name)

    print("Using cached best configuration:")
    print(f"- RAG key: {configuration_name}")
    print(f"- Config: {config_name}")
    print(f"- Label: {pretty_name}")
    print(f"- Accuracy: {float(best_row.get('accuracy', 0.0)) * 100:.0f}%")
    print(f"- Avg time: {float(best_row.get('avg_time_sec', 0.0)):.2f}s")
    print()

    if configuration_name == "rag_baseline_question_only":
        return _activate_baseline_question_only_with_params(
            top_k_context=int(best_row.get("top_k_context", 4)),
            rag_max_new_tokens=64,
            debug=True,
        )

    if configuration_name == "rag_question_options":
        return _activate_question_options_with_params(
            top_k_context=int(best_row.get("top_k_context", 4)),
            rag_max_new_tokens=64,
            debug=True,
        )

    if configuration_name == "reliable_rag":
        return _activate_logged_variant(
            activate_reliable_rag_variant,
            top_k_context=int(best_row.get("top_k_context", 4)),
            retrieval_query_mode=best_row.get(
                "retrieval_query_mode",
                "question_plus_options",
            ),
        )

    if configuration_name == "query_transformation_rag":
        return _activate_logged_variant(
            activate_query_transformation_rag_variant,
            top_k_context=int(best_row.get("top_k_context", 6)),
            top_k_per_query=int(best_row.get("top_k_per_query", 2)),
            max_subqueries=int(best_row.get("max_subqueries", 3)),
            max_final_sources=int(best_row.get("max_final_sources", 3)),
            query_transform_original_mode=best_row.get(
                "query_transform_original_mode",
                "question_plus_options",
            ),
            enable_llm_relevance_check=bool(
                best_row.get("enable_llm_relevance_check", True)
            ),
        )

    if configuration_name == "hyde_rag":
        return _activate_logged_variant(
            activate_hyde_rag_variant,
            top_k_context=int(best_row.get("top_k_context", 4)),
            max_final_sources=int(best_row.get("max_final_sources", 3)),
            hyde_include_options=bool(best_row.get("hyde_include_options", False)),
            hyde_fallback_query_mode=best_row.get(
                "hyde_fallback_query_mode",
                "question_only",
            ),
            enable_llm_relevance_check=bool(
                best_row.get("enable_llm_relevance_check", False)
            ),
        )

    if configuration_name == "rse_rag":
        return _activate_logged_variant(
            activate_rse_rag_variant,
            top_k_context=int(best_row.get("top_k_context", 6)),
            rse_neighbor_window=int(best_row.get("rse_neighbor_window", 1)),
            rse_max_segment_length=int(best_row.get("rse_max_segment_length", 3)),
            rse_max_total_paragraphs=int(best_row.get("rse_max_total_paragraphs", 3)),
            enable_llm_relevance_check=bool(
                best_row.get("enable_llm_relevance_check", False)
            ),
        )

    if configuration_name == "contextual_compression_rag":
        return _activate_logged_variant(
            activate_rse_compressed_rag_variant,
            top_k_context=int(best_row.get("top_k_context", 6)),
            max_final_sources=int(best_row.get("max_final_sources", 3)),
            enable_rse=False,
            enable_llm_relevance_check=bool(
                best_row.get("enable_llm_relevance_check", False)
            ),
            enable_contextual_compression=True,
        )

    if configuration_name == "rse_contextual_compression_rag":
        return _activate_logged_variant(
            activate_rse_compressed_rag_variant,
            top_k_context=int(best_row.get("top_k_context", 6)),
            max_final_sources=int(best_row.get("max_final_sources", 3)),
            enable_rse=True,
            rse_neighbor_window=int(best_row.get("rse_neighbor_window", 1)),
            rse_max_segment_length=int(best_row.get("rse_max_segment_length", 3)),
            rse_max_total_paragraphs=int(best_row.get("rse_max_total_paragraphs", 5)),
            enable_llm_relevance_check=bool(
                best_row.get("enable_llm_relevance_check", False)
            ),
            enable_contextual_compression=True,
        )

    if configuration_name == "fusion_rag":
        return _activate_logged_variant(
            activate_fusion_rag_variant,
            top_k_context=int(best_row.get("top_k_context", 8)),
            final_top_k_context=int(best_row.get("final_top_k_context", 3)),
            fusion_enable_semantic=bool(best_row.get("fusion_enable_semantic", True)),
            fusion_enable_bm25=bool(best_row.get("fusion_enable_bm25", True)),
            fusion_semantic_top_k=int(best_row.get("fusion_semantic_top_k", 20)),
            fusion_bm25_top_k=int(best_row.get("fusion_bm25_top_k", 20)),
            fusion_rrf_k=int(best_row.get("fusion_rrf_k", 60)),
            fusion_semantic_query_mode=best_row.get(
                "fusion_semantic_query_mode",
                "question_options",
            ),
            fusion_bm25_query_mode=best_row.get(
                "fusion_bm25_query_mode",
                "question_only",
            ),
            fusion_use_rse=bool(best_row.get("fusion_use_rse", False)),
            enable_llm_relevance_check=bool(
                best_row.get("enable_llm_relevance_check", False)
            ),
        )

    if configuration_name == "fusion_cross_encoder_rag":
        return _activate_logged_variant(
            activate_fusion_rerank_rag_variant,
            top_k_context=int(best_row.get("top_k_context", 12)),
            final_top_k_context=int(best_row.get("final_top_k_context", 3)),
            use_semantic_retrieval=bool(
                best_row.get("fusion_enable_semantic", True)
            ),
            use_bm25_retrieval=bool(
                best_row.get("fusion_enable_bm25", True)
            ),
            fusion_semantic_top_k=int(best_row.get("fusion_semantic_top_k", 30)),
            fusion_bm25_top_k=int(best_row.get("fusion_bm25_top_k", 30)),
            fusion_rrf_k=int(best_row.get("fusion_rrf_k", 60)),
            fusion_semantic_query_mode=best_row.get(
                "fusion_semantic_query_mode",
                "question_options",
            ),
            fusion_bm25_query_mode=best_row.get(
                "fusion_bm25_query_mode",
                "question_only",
            ),
            use_cross_encoder_rerank=bool(
                best_row.get("use_cross_encoder_rerank", True)
            ),
            rerank_model_id=best_row.get(
                "rerank_model_id",
                "cross-encoder/ms-marco-MiniLM-L-6-v2",
            ),
            rerank_query_mode=best_row.get("rerank_query_mode", "question_options"),
            rerank_batch_size=int(best_row.get("rerank_batch_size", 16)),
            use_rse=bool(best_row.get("fusion_use_rse", False)),
            enable_llm_relevance_check=bool(
                best_row.get("enable_llm_relevance_check", False)
            ),
            load_reranker_now=True,
        )

    if configuration_name == "adaptive_best_config_rag":
        if "activate_adaptive_rag_variant" not in globals():
            raise NameError(
                "activate_adaptive_rag_variant is not defined. "
                "Run the Adaptive Retrieval Router cell after the Final setup cell."
            )

        return _activate_logged_variant(
            activate_adaptive_rag_variant,
            default_profile="balanced",
            refresh_best_configs=True,
            debug=True,
        )

    if configuration_name == "self_rag":
        return _activate_logged_variant(
            activate_self_rag_variant,
            top_k_context=int(best_row.get("top_k_context", 4)),
            max_final_contexts=int(best_row.get("max_final_contexts", 2)),
            retrieval_query_mode=best_row.get(
                "retrieval_query_mode",
                "question_only",
            ),
            debug=True,
        )

    if configuration_name == "crag":
        return activate_crag_rag_variant(
            force_web_for_news=bool(best_row.get("force_web_for_news", True)),
            news_comp_id=int(best_row.get("news_comp_id", 5)),

            use_semantic_retrieval=bool(
                best_row.get("use_semantic_retrieval", True)
            ),
            use_bm25_retrieval=bool(best_row.get("use_bm25_retrieval", True)),
            use_cross_encoder_rerank=bool(
                best_row.get("use_cross_encoder_rerank", False)
            ),

            local_answerability_min_confidence=float(
                best_row.get("local_answerability_min_confidence", 0.65)
            ),
            local_min_context_chars=int(best_row.get("local_min_context_chars", 250)),
            router_max_local_context_chars=int(
                best_row.get("router_max_local_context_chars", 3500)
            ),

            use_web_fallback=bool(best_row.get("use_web_fallback", True)),
            web_fallback_on_incorrect=bool(
                best_row.get("web_fallback_on_incorrect", True)
            ),
            web_search_top_k=int(best_row.get("web_search_top_k", 4)),
            web_search_fetch_k=int(best_row.get("web_search_fetch_k", 4)),
            web_min_relevance_score=float(
                best_row.get("web_min_relevance_score", 0.0)
            ),
            web_use_llm_query_rewrite=bool(
                best_row.get("web_use_llm_query_rewrite", False)
            ),
            web_use_option_wise_search=bool(
                best_row.get("web_use_option_wise_search", False)
            ),

            fusion_semantic_query_mode=best_row.get(
                "fusion_semantic_query_mode",
                "question_only",
            ),
            fusion_bm25_query_mode=best_row.get(
                "fusion_bm25_query_mode",
                "question_only",
            ),

            top_k_context=int(best_row.get("top_k_context", 12)),
            final_top_k_context=int(best_row.get("final_top_k_context", 3)),

            rag_max_new_tokens=int(best_row.get("rag_max_new_tokens", 64)),
            crag_answer_max_retries=int(best_row.get("crag_answer_max_retries", 1)),
            debug=True,
        )

    return activate_default_configuration(configuration_name)


# ------------------------------------------------------------
# Main public API
# ------------------------------------------------------------

BEST_CONFIGS_FOR_CURRENT_MODEL = None


def refresh_best_configs_for_current_model(verbose=True, display_table=True):
    global BEST_CONFIGS_FOR_CURRENT_MODEL

    BEST_CONFIGS_FOR_CURRENT_MODEL = load_best_configs_for_current_model(
        verbose=verbose,
        display_table=display_table,
    )

    return BEST_CONFIGS_FOR_CURRENT_MODEL


def list_available_configurations():
    print("Available configurations:")

    for name, spec in RAG_CONFIGURATIONS.items():
        print(f"- {name}: {spec['description']}")


def activate_selected_configuration(configuration_name, use_best_if_available=True):
    global BEST_CONFIGS_FOR_CURRENT_MODEL

    valid_names = set(RAG_CONFIGURATIONS.keys())

    if configuration_name not in valid_names:
        available = ", ".join(sorted(valid_names))
        raise ValueError(
            f"Unknown configuration '{configuration_name}'. Available: {available}"
        )

    if BEST_CONFIGS_FOR_CURRENT_MODEL is None:
        BEST_CONFIGS_FOR_CURRENT_MODEL = load_best_configs_for_current_model(
            verbose=False,
            display_table=False,
        )

    best_row = None

    if use_best_if_available:
        best_row = BEST_CONFIGS_FOR_CURRENT_MODEL.get(configuration_name)

    if best_row is not None:
        activate_best_cached_configuration(configuration_name, best_row)
        source = "cached_best"
    else:
        print(f"No cached best config found for '{configuration_name}'.")
        print("Using fallback/default activation.\n")
        activate_default_configuration(configuration_name)
        source = "fallback_default"

    if "BASE_RAG_CONFIG" in globals():
        BASE_RAG_CONFIG["active_configuration"] = configuration_name
        BASE_RAG_CONFIG["active_configuration_source"] = source

    print(f"Selected configuration: {configuration_name}")
    print(f"Configuration source: {source}")

    return {
        "name": configuration_name,
        "source": source,
        "best_row": best_row,
    }


# ============================================================
# Single-question RAG API for Speech mode
# ============================================================

def answer_question_with_selected_configuration(
    question,
    selected_configuration="crag",
    tokenizer=None,
    model=None,
    use_best_if_available=True,
):
    """
    Answer one already available question using the selected configuration.

    This function is mainly used by the speech/audio pipeline.

    It does NOT:
    - start a new game
    - call game.answer(...)
    - run a full game loop

    It only returns the predicted answer_id.
    """

    if tokenizer is None:
        tokenizer = llm_tokenizer

    if model is None:
        model = llm_model

    # 1. Activate the selected configuration
    activate_selected_configuration(
        selected_configuration,
        use_best_if_available=use_best_if_available,
    )

    # 2. LLM-only case
    if selected_configuration == "llm":
        return get_llm_answer(
            question,
            tokenizer,
            model,
        )

    # 3. RAG case
    return get_rag_answer(
        question,
        tokenizer,
        model,
    )


def play_selected_configuration(configuration_name, use_best_if_available=True):
    selected = activate_selected_configuration(
        configuration_name,
        use_best_if_available=use_best_if_available,
    )

    game = client.game.start(competition_id=comp_id)

    if configuration_name == "llm":
        return play_game_llm(game, llm_tokenizer, llm_model)

    return play_game_rag(game, llm_tokenizer, llm_model)

# ============================================================
# FINAL SETUP PATCH - AUTOMATIC NEWS-SPECIFIC BEST CONFIG
#
# Paste this cell AFTER the existing Final setup cell.
#
# What it does:
#   - keeps the original automatic best selector for all RAG configs;
#   - adds an automatic News-specific selector for CRAG;
#   - reads News-specific benchmark summaries from cache;
#   - selects the best News config by:
#       1. highest accuracy
#       2. lowest average time
#     while respecting max_time_sec <= 30 seconds when possible;
#   - uses the News-specific best config only when:
#       selected_configuration == "crag"
#       and competition_id == 5
# ============================================================

import os
import glob
import json
import hashlib
import pandas as pd
from pathlib import Path
from IPython.display import display


# ============================================================
# 1. Save original public functions
# ============================================================

if "activate_selected_configuration_original" not in globals():
    activate_selected_configuration_original = activate_selected_configuration

if "answer_question_with_selected_configuration_original" not in globals():
    answer_question_with_selected_configuration_original = answer_question_with_selected_configuration

if "play_selected_configuration_original" not in globals():
    play_selected_configuration_original = play_selected_configuration

if "refresh_best_configs_for_current_model_original" not in globals():
    refresh_best_configs_for_current_model_original = refresh_best_configs_for_current_model


# ============================================================
# 2. News-specific benchmark paths
# ============================================================

NEWS_SOLVED_BENCHMARK_PATH = os.path.join(
    BENCHMARK_DIR,
    "news_crag_wrong_questions_solved.json"
)

NEWS_CACHE_GROUP = "news_web_crag_eval"

NEWS_SUMMARY_FILENAME = "news_web_crag_all_configs_with_true_chunking_summary.json"

NEWS_MAX_ALLOWED_TIME_SEC = 30.0


# ============================================================
# 3. Generic helpers
# ============================================================

def md5_json_safe(obj):
    text = json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.md5(text.encode("utf-8")).hexdigest()


def md5_file_safe(path):
    h = hashlib.md5()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


def get_news_benchmark_and_model_hashes():
    """
    Uses the News-specific benchmark file and the current model metadata.
    This mirrors the existing model-aware selector logic.
    """

    if not os.path.exists(NEWS_SOLVED_BENCHMARK_PATH):
        return None, None, None

    news_benchmark_hash = md5_file_safe(NEWS_SOLVED_BENCHMARK_PATH)

    model_metadata = get_model_metadata_for_best_config()
    model_hash = md5_json_safe(model_metadata)

    return news_benchmark_hash, model_hash, model_metadata


def is_news_competition(competition_id=None, category_name=None):
    """
    News category has competition_id = 5 in our setup.
    """

    if competition_id is not None:
        try:
            return int(competition_id) == 5
        except Exception:
            pass

    if category_name is not None:
        return str(category_name).strip().lower() == "news"

    if "comp_id" in globals():
        try:
            return int(comp_id) == 5
        except Exception:
            pass

    return False


def to_bool(value, default=False):
    if value is None:
        return default

    if isinstance(value, bool):
        return value

    if isinstance(value, str):
        return value.strip().lower() in {"true", "1", "yes", "y"}

    return bool(value)


def to_int(value, default):
    try:
        if pd.isna(value):
            return default
    except Exception:
        pass

    try:
        return int(value)
    except Exception:
        return default


def to_float(value, default):
    try:
        if pd.isna(value):
            return default
    except Exception:
        pass

    try:
        return float(value)
    except Exception:
        return default


# ============================================================
# 4. Load News-specific cached benchmark summaries
# ============================================================

def load_cached_news_benchmark_summaries_for_current_model(verbose=True):
    """
    Loads News-specific benchmark summaries.

    First tries the exact current model hash.
    If not found, searches any available model_* folder for the same
    News benchmark hash and uses the most recent summary.

    This mirrors the fallback behavior used by the general selector.
    """

    news_benchmark_hash, current_model_hash, model_metadata = get_news_benchmark_and_model_hashes()

    result = {
        "news_benchmark_hash": news_benchmark_hash,
        "model_hash": current_model_hash,
        "model_metadata": model_metadata,
        "summaries": [],
        "summary_path": None,
        "exact_current_hash_match": False,
    }

    if news_benchmark_hash is None:
        if verbose:
            print("News-specific benchmark file not found:")
            print(NEWS_SOLVED_BENCHMARK_PATH)

        return result

    expected_path = os.path.join(
        CACHE_ROOT,
        NEWS_CACHE_GROUP,
        f"benchmark_{news_benchmark_hash}",
        f"model_{current_model_hash}",
        "summary",
        NEWS_SUMMARY_FILENAME,
    )

    summary_path = None
    exact_match = False

    if os.path.exists(expected_path):
        summary_path = expected_path
        exact_match = True
    else:
        candidate_paths = glob.glob(
            os.path.join(
                CACHE_ROOT,
                NEWS_CACHE_GROUP,
                f"benchmark_{news_benchmark_hash}",
                "model_*",
                "summary",
                NEWS_SUMMARY_FILENAME,
            )
        )

        if candidate_paths:
            candidate_paths = sorted(
                candidate_paths,
                key=lambda p: os.path.getmtime(p),
                reverse=True,
            )
            summary_path = candidate_paths[0]

    if summary_path is None:
        if verbose:
            print("\nNo News-specific benchmark summary found.")
            print("Expected:")
            print(expected_path)

        return result

    with open(summary_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    summaries = data.get("summaries", [])

    for row in summaries:
        row["play_key"] = "crag_news"
        row["benchmark_group"] = NEWS_CACHE_GROUP
        row["summary_path"] = summary_path
        row["category_specific"] = "News"

    result.update({
        "summaries": summaries,
        "summary_path": summary_path,
        "exact_current_hash_match": exact_match,
    })

    if verbose:
        print("\nLoaded News-specific benchmark summaries")
        print(f"Rows: {len(summaries)}")
        print(f"Exact current hash match: {exact_match}")
        print("Summary path:")
        print(summary_path)

    return result


# ============================================================
# 5. Select best News config automatically
# ============================================================

def select_best_news_crag_config(news_summaries, max_allowed_time_sec=NEWS_MAX_ALLOWED_TIME_SEC):
    """
    Selects the best News-specific CRAG config.

    Rule:
    1. Prefer configs with max_time_sec <= 30 seconds.
    2. Among eligible configs:
       - highest accuracy
       - lowest avg_time_sec
    3. If no config satisfies the time constraint, select best accuracy anyway.
    """

    if not news_summaries:
        return None

    rows = []

    for row in news_summaries:
        total_questions = to_int(row.get("total_questions"), 0)

        if total_questions <= 0:
            continue

        error_count = to_int(row.get("error_count"), 0)

        # Keep rows with errors, but they will naturally rank worse if accuracy is lower.
        candidate = dict(row)
        candidate["_accuracy"] = to_float(row.get("accuracy"), 0.0)
        candidate["_avg_time"] = to_float(row.get("avg_time_sec"), 999999.0)
        candidate["_max_time"] = to_float(row.get("max_time_sec"), 999999.0)
        candidate["_error_count"] = error_count

        rows.append(candidate)

    if not rows:
        return None

    eligible = [
        row for row in rows
        if row["_max_time"] <= max_allowed_time_sec
    ]

    if eligible:
        pool = eligible
        selection_rule = f"accuracy first, avg time tie-breaker, max_time <= {max_allowed_time_sec:.0f}s"
    else:
        pool = rows
        selection_rule = "accuracy first, avg time tie-breaker, no config under max_time limit"

    best = sorted(
        pool,
        key=lambda x: (
            x["_accuracy"],
            -x["_avg_time"],
        ),
        reverse=True,
    )[0]

    best["selection_rule"] = selection_rule
    best["news_max_allowed_time_sec"] = max_allowed_time_sec
    best["selected_under_time_limit"] = bool(eligible)

    return best


def build_news_best_config_table(best_news_row):
    if best_news_row is None:
        return pd.DataFrame()

    return pd.DataFrame([
        {
            "Category": "News",
            "RAG key": "crag",
            "Best configuration": best_news_row.get("pretty_name", best_news_row.get("config_name")),
            "Config name": best_news_row.get("config_name"),
            "Correct": f"{to_int(best_news_row.get('correct'), 0)}/{to_int(best_news_row.get('total_questions'), 0)}",
            "Accuracy": f"{to_float(best_news_row.get('accuracy'), 0.0) * 100:.0f}%",
            "Avg. time": f"{to_float(best_news_row.get('avg_time_sec'), 0.0):.2f}s",
            "Max time": f"{to_float(best_news_row.get('max_time_sec'), 0.0):.2f}s",
            "Selection rule": best_news_row.get("selection_rule"),
        }
    ])


# ============================================================
# 6. Global best config holders
# ============================================================

NEWS_BEST_CONFIG_FOR_CURRENT_MODEL = None
NEWS_BEST_CONFIG_LOAD_INFO = None


def refresh_news_best_config_for_current_model(verbose=True, display_table=True):
    global NEWS_BEST_CONFIG_FOR_CURRENT_MODEL
    global NEWS_BEST_CONFIG_LOAD_INFO

    loaded = load_cached_news_benchmark_summaries_for_current_model(verbose=verbose)

    NEWS_BEST_CONFIG_LOAD_INFO = loaded

    NEWS_BEST_CONFIG_FOR_CURRENT_MODEL = select_best_news_crag_config(
        loaded.get("summaries", []),
        max_allowed_time_sec=NEWS_MAX_ALLOWED_TIME_SEC,
    )

    if display_table:
        print("\nBest News-specific CRAG configuration")
        news_df = build_news_best_config_table(NEWS_BEST_CONFIG_FOR_CURRENT_MODEL)

        if news_df.empty:
            print("No News-specific best config available.")
        else:
            display(news_df)

    return NEWS_BEST_CONFIG_FOR_CURRENT_MODEL


def refresh_best_configs_for_current_model(verbose=True, display_table=True):
    """
    Updated refresh function.

    It refreshes:
    - general best configs;
    - News-specific CRAG best config.
    """

    global BEST_CONFIGS_FOR_CURRENT_MODEL

    BEST_CONFIGS_FOR_CURRENT_MODEL = refresh_best_configs_for_current_model_original(
        verbose=verbose,
        display_table=display_table,
    )

    refresh_news_best_config_for_current_model(
        verbose=verbose,
        display_table=display_table,
    )

    return BEST_CONFIGS_FOR_CURRENT_MODEL


# ============================================================
# 7. Activate News-specific best CRAG config
# ============================================================

def activate_news_crag_config_from_best_row(best_row, debug=None):
    """
    Activates the automatically selected News-specific CRAG config.

    It reads all parameters from the cached News benchmark row.
    """

    if best_row is None:
        print("No News-specific best config available. Falling back to general CRAG selector.")
        return activate_selected_configuration_original(
            "crag",
            use_best_if_available=True,
        )

    config_name = best_row.get("config_name")
    pretty_name = best_row.get("pretty_name", config_name)

    if debug is None:
        debug = False

    print("Using automatic News-specific best CRAG configuration:")
    print(f"- Config: {config_name}")
    print(f"- Label: {pretty_name}")
    print(f"- Accuracy: {to_float(best_row.get('accuracy'), 0.0) * 100:.0f}%")
    print(f"- Avg time: {to_float(best_row.get('avg_time_sec'), 0.0):.2f}s")
    print(f"- Max time: {to_float(best_row.get('max_time_sec'), 0.0):.2f}s")
    print(f"- Selection rule: {best_row.get('selection_rule')}")
    print()

    activate_crag_rag_variant(
        force_web_for_news=to_bool(best_row.get("force_web_for_news"), True),
        news_comp_id=to_int(best_row.get("news_comp_id"), 5),

        use_semantic_retrieval=to_bool(best_row.get("use_semantic_retrieval"), True),
        use_bm25_retrieval=to_bool(best_row.get("use_bm25_retrieval"), True),
        use_cross_encoder_rerank=to_bool(best_row.get("use_cross_encoder_rerank"), False),

        local_answerability_min_confidence=to_float(
            best_row.get("local_answerability_min_confidence"),
            0.95,
        ),
        local_min_context_chars=to_int(
            best_row.get("local_min_context_chars"),
            999999,
        ),
        router_max_local_context_chars=to_int(
            best_row.get("router_max_local_context_chars"),
            3500,
        ),

        use_web_fallback=to_bool(best_row.get("use_web_fallback"), True),
        web_fallback_on_incorrect=to_bool(
            best_row.get("web_fallback_on_incorrect"),
            True,
        ),

        web_search_top_k=to_int(best_row.get("web_search_top_k"), 4),
        web_search_fetch_k=to_int(best_row.get("web_search_fetch_k"), 4),
        web_min_relevance_score=to_float(
            best_row.get("web_min_relevance_score"),
            0.0,
        ),

        web_use_llm_query_rewrite=to_bool(
            best_row.get("web_use_llm_query_rewrite"),
            False,
        ),
        web_use_option_wise_search=to_bool(
            best_row.get("web_use_option_wise_search"),
            True,
        ),

        fusion_semantic_query_mode=best_row.get(
            "fusion_semantic_query_mode",
            "question_options",
        ),
        fusion_bm25_query_mode=best_row.get(
            "fusion_bm25_query_mode",
            "question_only",
        ),

        top_k_context=to_int(best_row.get("top_k_context"), 12),
        final_top_k_context=to_int(best_row.get("final_top_k_context"), 3),

        rag_max_new_tokens=to_int(best_row.get("rag_max_new_tokens"), 64),
        crag_answer_max_retries=to_int(
            best_row.get("crag_answer_max_retries"),
            1,
        ),

        debug=debug,
    )

    if "BASE_RAG_CONFIG" in globals():
        BASE_RAG_CONFIG.update({
            "active_rag_variant": "crag",
            "active_configuration": config_name,
            "active_configuration_label": pretty_name,
            "active_configuration_source": "news_cached_best",
            "active_configuration_accuracy": to_float(best_row.get("accuracy"), 0.0),
            "active_configuration_avg_time_sec": to_float(best_row.get("avg_time_sec"), 0.0),
            "active_configuration_max_time_sec": to_float(best_row.get("max_time_sec"), 0.0),
            "debug": debug,
        })

    return {
        "name": "crag",
        "source": "news_cached_best",
        "best_row": best_row,
    }


# ============================================================
# 8. Updated public selector
# ============================================================

def activate_selected_configuration(
    configuration_name,
    use_best_if_available=True,
    competition_id=None,
    category_name=None,
    use_news_specific_if_available=True,
):
    """
    Updated selector.

    Behavior:
    - If configuration_name == "crag" and category is News:
        use automatic News-specific cached best config.
    - Otherwise:
        use the original general selector.
    """

    global BEST_CONFIGS_FOR_CURRENT_MODEL
    global NEWS_BEST_CONFIG_FOR_CURRENT_MODEL

    valid_names = set(RAG_CONFIGURATIONS.keys())

    if configuration_name not in valid_names:
        available = ", ".join(sorted(valid_names))
        raise ValueError(
            f"Unknown configuration '{configuration_name}'. Available: {available}"
        )

    is_news = is_news_competition(
        competition_id=competition_id,
        category_name=category_name,
    )

    if (
        use_best_if_available
        and use_news_specific_if_available
        and configuration_name == "crag"
        and is_news
    ):
        if NEWS_BEST_CONFIG_FOR_CURRENT_MODEL is None:
            refresh_news_best_config_for_current_model(
                verbose=False,
                display_table=False,
            )

        if NEWS_BEST_CONFIG_FOR_CURRENT_MODEL is not None:
            return activate_news_crag_config_from_best_row(
                NEWS_BEST_CONFIG_FOR_CURRENT_MODEL,
                debug=False,
            )

    return activate_selected_configuration_original(
        configuration_name,
        use_best_if_available=use_best_if_available,
    )


# ============================================================
# 9. Updated single-question API
# ============================================================

def answer_question_with_selected_configuration(
    question,
    selected_configuration="crag",
    tokenizer=None,
    model=None,
    use_best_if_available=True,
    competition_id=None,
    category_name=None,
    use_news_specific_if_available=True,
):
    """
    Updated single-question API.

    Used by speech/audio mode.

    It automatically uses News-specific CRAG best config when:
        selected_configuration == "crag"
        and competition_id == 5
    """

    if tokenizer is None:
        tokenizer = llm_tokenizer

    if model is None:
        model = llm_model

    if competition_id is None:
        competition_id = getattr(question, "competition_id", None)

    if category_name is None:
        category_name = getattr(question, "category", None)

    activate_selected_configuration(
        selected_configuration,
        use_best_if_available=use_best_if_available,
        competition_id=competition_id,
        category_name=category_name,
        use_news_specific_if_available=use_news_specific_if_available,
    )

    if selected_configuration == "llm":
        return get_llm_answer(
            question,
            tokenizer,
            model,
        )

    return get_rag_answer(
        question,
        tokenizer,
        model,
    )


# ============================================================
# 10. Updated play function
# ============================================================

def play_selected_configuration(
    configuration_name,
    use_best_if_available=True,
    competition_id=None,
    use_news_specific_if_available=True,
):
    """
    Updated play function.

    Automatically uses News-specific cached best CRAG config for News.
    """

    if competition_id is None:
        if "comp_id" in globals():
            competition_id = comp_id
        else:
            raise ValueError(
                "competition_id was not provided and global comp_id does not exist."
            )

    selected = activate_selected_configuration(
        configuration_name,
        use_best_if_available=use_best_if_available,
        competition_id=competition_id,
        use_news_specific_if_available=use_news_specific_if_available,
    )

    game = client.game.start(competition_id=competition_id)

    if configuration_name == "llm":
        return play_game_llm(
            game,
            llm_tokenizer,
            llm_model,
        )

    return play_game_rag(
        game,
        llm_tokenizer,
        llm_model,
    )


# ============================================================
# 11. Verification helper
# ============================================================

def inspect_active_rag_configuration():
    print("=" * 80)
    print("ACTIVE RAG CONFIGURATION")
    print("=" * 80)

    if "BASE_RAG_CONFIG" in globals():
        print("\nBASE_RAG_CONFIG:")
        for key in [
            "active_rag_variant",
            "active_configuration",
            "active_configuration_label",
            "active_configuration_source",
            "active_configuration_accuracy",
            "active_configuration_avg_time_sec",
            "active_configuration_max_time_sec",
            "debug",
        ]:
            print(f"{key}: {BASE_RAG_CONFIG.get(key)}")

    if "CRAG_CONFIG" in globals():
        print("\nCRAG_CONFIG important flags:")
        for key in [
            "force_web_for_news",
            "news_comp_id",
            "use_web_fallback",
            "web_fallback_on_incorrect",
            "web_search_top_k",
            "web_search_fetch_k",
            "web_use_llm_query_rewrite",
            "web_use_option_wise_search",
            "use_cross_encoder_rerank",
            "local_answerability_min_confidence",
            "local_min_context_chars",
            "router_max_local_context_chars",
            "top_k_context",
            "final_top_k_context",
        ]:
            print(f"{key}: {CRAG_CONFIG.get(key)}")

    print("\nget_rag_answer callable:")
    if "get_rag_answer" in globals():
        print(getattr(get_rag_answer, "__name__", str(get_rag_answer)))

    print("=" * 80)


# ============================================================
# 12. Optional immediate refresh
# ============================================================

print("News-aware automatic selector patch loaded.")
print("Run this to refresh both general and News-specific best configs:")
print("BEST_CONFIGS_FOR_CURRENT_MODEL = refresh_best_configs_for_current_model(verbose=True, display_table=True)")


News-aware automatic selector patch loaded.
Run this to refresh both general and News-specific best configs:
BEST_CONFIGS_FOR_CURRENT_MODEL = refresh_best_configs_for_current_model(verbose=True, display_table=True)


Debugger

In [ ]:
# Debugger

# ============================================================
# ADVANCED RAG DEBUGGER
# Clean debug for:
#   - adaptive_best_config_rag
#   - self_rag
#   - crag
#   - reliable_rag
#   - query_transform_rag
#   - hyde_rag
#   - rse_rag
#   - contextual_compression_rag
#   - rse_contextual_compression_rag
#   - fusion_rag
#   - fusion_cross_encoder_rag
#
# Paste this cell AFTER the Final setup / CRAG / Self-RAG cells.
# ============================================================

import re
import json
import time
import numpy as np


# ============================================================
# 1. Pretty-print helpers
# ============================================================

if "retrieve_local_wikipedia_context_with_options_candidates" not in globals():
    def retrieve_local_wikipedia_context_with_options_candidates(question, top_k=4):
        """
        Local fallback used by RSE/RSE-compression debug flows when the
        Reliable-RAG retrieval helper cell has not been executed.
        """
        question_text = question.text.strip()
        query_mode = BASE_RAG_CONFIG.get("retrieval_query_mode", "question_plus_options")

        used_options = query_mode != "question_only"
        if used_options and getattr(question, "options", None) is not None:
            options_text = " ".join([opt.text for opt in question.options])
            query = f"{question_text} {options_text}".strip()
        else:
            query = question_text

        query_embedding = semb_model.encode(query, convert_to_tensor=False).astype("float32")
        query_embedding = np.expand_dims(query_embedding, axis=0)
        faiss.normalize_L2(query_embedding)

        scores, indices = faiss_index.search(query_embedding, top_k)

        hits = []
        for rank, (corpus_id, score) in enumerate(zip(indices[0], scores[0]), start=1):
            corpus_id = int(corpus_id)
            if corpus_id < 0:
                continue

            hits.append({
                "corpus_id": corpus_id,
                "score": float(score),
                "query": query,
                "query_type": query_mode,
                "used_options": used_options,
                "source_label": chr(64 + rank),
            })

        return hits

def _dbg_line(char="=", width=90):
    print(char * width)


def _dbg_title(title, emoji="🔎"):
    print()
    _dbg_line("=")
    print(f"{emoji} {title}")
    _dbg_line("=")


def _dbg_subtitle(title, emoji="•"):
    print()
    print(f"{emoji} {title}")
    print("-" * 90)


def _dbg_clip(text, max_chars=900):
    text = str(text or "").strip()
    text = re.sub(r"\s+", " ", text)

    if len(text) <= max_chars:
        return text

    return text[:max_chars].rstrip() + "..."


def _dbg_multiline_clip(text, max_chars=1400):
    text = str(text or "").strip()

    if len(text) <= max_chars:
        return text

    return text[:max_chars].rstrip() + "\n..."


def _dbg_fmt_time(value):
    if isinstance(value, (int, float)):
        return f"{value:.2f}s"
    return "-"


def _dbg_get_option_text(question, answer_id):
    for opt in getattr(question, "options", []) or []:
        if int(opt.id) == int(answer_id):
            return opt.text
    return None


def _dbg_print_question(question):
    _dbg_subtitle("Question", "❓")
    print(question.text)

    print()
    print("Options:")
    for opt in question.options:
        print(f"  [{opt.id}] {opt.text}")


def _dbg_print_final_answer(question, answer_id, details):
    _dbg_subtitle("Final answer", "✅")

    answer_text = _dbg_get_option_text(question, answer_id)

    print(f"Selected option id: {answer_id}")

    if answer_text is not None:
        print(f"Selected option text: {answer_text}")

    if details.get("fallback_used"):
        print(f"Fallback used: YES")
        print(f"Fallback reason: {details.get('fallback_reason')}")
    elif details.get("parsing_failed"):
        print("Parsing failed: YES")
    else:
        print("Parsing failed: NO")


def _dbg_print_timings(details):
    timing_keys = [
        ("retrieval_decision_time_sec", "Retrieval decision"),
        ("retrieval_time_sec", "Retrieval"),
        ("relevance_time_sec", "Relevance check"),
        ("selection_time_sec", "Selection"),
        ("rerank_time_sec", "Cross-encoder rerank"),
        ("rse_time_sec", "RSE enrichment"),
        ("compression_time_sec", "Context compression"),
        ("context_time_sec", "Context construction"),
        ("generation_time_sec", "Answer generation"),
        ("support_time_sec", "Support critic"),
        ("utility_time_sec", "Utility critic"),
        ("total_answer_time_sec", "Total"),
    ]

    available = [
        (key, label)
        for key, label in timing_keys
        if isinstance(details.get(key), (int, float))
    ]

    if not available:
        return

    _dbg_subtitle("Timing", "⏱️")

    for key, label in available:
        print(f"{label:<26} {_dbg_fmt_time(details.get(key))}")


def _dbg_extract_sources_from_context(context):
    """
    Extract readable source blocks from a context string.
    Works for:
    - local Wikipedia Context Source A/B/C
    - web Context Source W1/W2
    - RSE Context Segment A/B/C
    """
    context = str(context or "").strip()

    if not context:
        return []

    parts = re.split(
        r"(?=Context\s+(?:Source|Segment)\s+[A-Z0-9]+)",
        context
    )

    sources = []

    for part in parts:
        part = part.strip()

        if not part.startswith("Context "):
            continue

        first_line = part.splitlines()[0].strip() if part.splitlines() else ""
        label_match = re.search(r"Context\s+(?:Source|Segment)\s+([A-Z0-9]+)", first_line)
        label = label_match.group(1) if label_match else "?"

        title_match = re.search(r"Title:\s*(.*?)(?:\||\n|$)", part)
        title = title_match.group(1).strip() if title_match else None

        url_match = re.search(r"URL:\s*(.*?)(?:\n|$)", part)
        url = url_match.group(1).strip() if url_match else None

        paragraph_match = re.search(r"Paragraph(?:s)?:\s*([^|\n]+)", part)
        paragraph = paragraph_match.group(1).strip() if paragraph_match else None

        score_match = re.search(
            r"(?:Score|Retrieval score|Fusion score|Segment score):\s*([0-9.\-]+)",
            part,
            flags=re.IGNORECASE,
        )
        score = score_match.group(1) if score_match else None

        sources.append({
            "label": label,
            "title": title,
            "url": url,
            "paragraph": paragraph,
            "score": score,
            "text": part,
        })

    return sources


def _dbg_print_sources_from_hits(
    hits,
    title="Retrieved passages",
    emoji="📄",
    max_sources=6,
    max_preview_chars=500,
):
    if not hits:
        return

    _dbg_subtitle(title, emoji)

    for idx, hit in enumerate(hits[:max_sources], start=1):
        label = hit.get("source_label", chr(64 + idx))
        title = hit.get("title", "Untitled")
        paragraph_id = hit.get("paragraph_id", "-")
        score = hit.get("score", None)

        score_text = f" | score={score:.4f}" if isinstance(score, (int, float)) else ""

        print(f"[{label}] {title} | paragraph {paragraph_id}{score_text}")

        if hit.get("semantic_rank") is not None or hit.get("bm25_rank") is not None:
            print(
                f"    semantic_rank={hit.get('semantic_rank')} | "
                f"bm25_rank={hit.get('bm25_rank')} | "
                f"rerank_score={hit.get('rerank_score')}"
            )

        text = hit.get("text") or hit.get("passage_text") or hit.get("snippet") or ""
        if text:
            print(f"    {_dbg_clip(text, max_preview_chars)}")

        query = hit.get("query")
        query_type = hit.get("query_type")

        if query or query_type:
            print(f"    query_type={query_type} | query={_dbg_clip(query, 250)}")

        print()

    if len(hits) > max_sources:
        print(f"... {len(hits) - max_sources} additional sources hidden.")


def _dbg_print_sources_from_context(
    context,
    title="Context passed to the answer prompt",
    emoji="📚",
    max_sources=5,
    max_preview_chars=700,
):
    sources = _dbg_extract_sources_from_context(context)

    if not sources:
        if context:
            _dbg_subtitle(title, emoji)
            print(_dbg_multiline_clip(context, max_preview_chars))
        return

    _dbg_subtitle(title, emoji)

    for src in sources[:max_sources]:
        label = src.get("label", "?")
        title_text = src.get("title") or "Untitled"
        paragraph = src.get("paragraph")
        score = src.get("score")
        url = src.get("url")

        meta = []
        if paragraph:
            meta.append(f"paragraph={paragraph}")
        if score:
            meta.append(f"score={score}")
        if url:
            meta.append(f"url={url}")

        meta_text = " | ".join(meta)

        if meta_text:
            print(f"[{label}] {title_text} | {meta_text}")
        else:
            print(f"[{label}] {title_text}")

        print(f"    {_dbg_clip(src.get('text'), max_preview_chars)}")
        print()

    if len(sources) > max_sources:
        print(f"... {len(sources) - max_sources} additional context sources hidden.")


def _dbg_print_attempts(details, max_output_chars=900):
    attempts = details.get("attempts") or []

    if not attempts:
        raw_output = details.get("raw_output")
        if raw_output:
            _dbg_subtitle("LLM final generation", "🤖")
            print(_dbg_multiline_clip(raw_output, max_output_chars))
        return

    _dbg_subtitle("LLM answer generation attempts", "🤖")

    for attempt in attempts:
        label = attempt.get("source_label")

        if label:
            print(f"Attempt {attempt.get('attempt')} from source {label}")
        else:
            print(f"Attempt {attempt.get('attempt')}")

        print(f"Parsed answer: {attempt.get('parsed_answer')}")
        print("Raw output:")
        print(_dbg_multiline_clip(attempt.get("raw_output", ""), max_output_chars))
        print()


def _dbg_print_query_items(details, title="Queries used for retrieval", emoji="🧾"):
    # Print the retrieval queries actually used by the RAG system.
    query_items = list(details.get("query_items") or [])

    if not query_items and (
        details.get("fusion_semantic_query")
        or details.get("fusion_bm25_query")
    ):
        if details.get("fusion_semantic_query"):
            query_items.append({
                "query": details.get("fusion_semantic_query"),
                "query_type": "fusion_semantic_query",
            })

        if details.get("fusion_bm25_query"):
            query_items.append({
                "query": details.get("fusion_bm25_query"),
                "query_type": "fusion_bm25_query",
            })

    # Some variants, such as Reliable-RAG, keep a single query instead.
    single_query = details.get("query")
    single_query_type = details.get("query_type") or details.get("hyde_query_type") or "retrieval_query"

    if not query_items and single_query:
        query_items = [{
            "query": single_query,
            "query_type": single_query_type,
        }]

    if not query_items:
        return

    _dbg_subtitle(title, emoji)

    for idx, item in enumerate(query_items, start=1):
        query_type = item.get("query_type", "query")
        query = item.get("query", "")

        print(f"[{idx}] {query_type}")
        print(f"    {_dbg_clip(query, 500)}")


def _dbg_details_with_fusion_query_items(question, details):
    details = dict(details or {})

    if (
        details.get("query_items")
        or details.get("fusion_semantic_query")
        or details.get("fusion_bm25_query")
        or "build_fusion_query" not in globals()
    ):
        return details

    query_items = []

    if BASE_RAG_CONFIG.get("fusion_enable_semantic", True):
        mode = BASE_RAG_CONFIG.get("fusion_semantic_query_mode", "question_options")
        query_items.append({
            "query": build_fusion_query(question, mode=mode),
            "query_type": f"fusion_semantic_{mode}",
        })

    if BASE_RAG_CONFIG.get("fusion_enable_bm25", True):
        mode = BASE_RAG_CONFIG.get("fusion_bm25_query_mode", "question_only")
        query_items.append({
            "query": build_fusion_query(question, mode=mode),
            "query_type": f"fusion_bm25_{mode}",
        })

    details["query_items"] = query_items
    return details


# ============================================================
# 2. Self-RAG detailed printer
# ============================================================

def _dbg_print_self_rag_details(question, answer_id, details):
    _dbg_title("Self-RAG debug flow", "🧠")
    _dbg_print_question(question)

    _dbg_subtitle("Step 1 — Retrieval decision", "🚦")
    print(f"Decision: {str(details.get('retrieval_decision')).upper()}")
    print(f"Parsing failed: {details.get('retrieval_decision_parsing_failed')}")
    print("LLM router output:")
    print(_dbg_multiline_clip(details.get("retrieval_decision_raw_output"), 700))

    hits = details.get("hits") or []
    filtered_hits = details.get("filtered_hits") or []

    if hits:
        _dbg_print_sources_from_hits(
            hits,
            title="Step 2 — Retrieved Wikipedia passages",
            emoji="📄",
            max_sources=6,
            max_preview_chars=450,
        )

    relevance_raw = details.get("relevance_check_raw_output")

    if relevance_raw:
        _dbg_subtitle("Step 3 — Relevance critic", "🧪")

        try:
            relevance_items = json.loads(relevance_raw)
        except Exception:
            relevance_items = []

        if relevance_items:
            for item in relevance_items:
                label = item.get("source_label", "?")
                parsed = item.get("parsed_relevant")

                if parsed is True:
                    status = "RELEVANT ✅"
                elif parsed is False:
                    status = "IRRELEVANT ❌"
                else:
                    status = "UNCLEAR ⚠️"

                print(f"[{label}] {status}")
                print(f"    LLM output: {_dbg_clip(item.get('raw_output'), 350)}")
        else:
            print(_dbg_multiline_clip(relevance_raw, 900))

    if filtered_hits:
        _dbg_print_sources_from_hits(
            filtered_hits,
            title="Step 4 — Selected contexts after filtering",
            emoji="📌",
            max_sources=4,
            max_preview_chars=450,
        )

    candidates = details.get("selfrag_candidates") or []

    if candidates:
        _dbg_subtitle("Step 5 — Candidate answers, support and utility", "🧩")

        for candidate in candidates:
            label = candidate.get("source_label", "?")
            print(f"Candidate from source {label}")
            print(f"  Parsed answer: {candidate.get('answer_id')}")
            print(f"  Support: {candidate.get('support_label')}")
            print(f"  Utility: {candidate.get('utility_score')}/5")
            print(f"  Retrieval score: {candidate.get('retrieval_score')}")
            print()
            print("  Candidate raw answer:")
            print("  " + _dbg_clip(candidate.get("raw_output"), 700))
            print()
            print("  Support critic raw output:")
            print("  " + _dbg_clip(candidate.get("support_raw_output"), 350))
            print()
            print("  Utility critic raw output:")
            print("  " + _dbg_clip(candidate.get("utility_raw_output"), 250))
            print("-" * 90)

    _dbg_print_final_answer(question, answer_id, details)
    _dbg_print_timings(details)


# ============================================================
# 3. CRAG router with raw-output logging
# ============================================================

def crag_run_local_answerability_router_logged(question, local_context, tokenizer, model, config):
    """
    Same logic as crag_run_local_answerability_router,
    but also returns the raw LLM router output and compact prompt info.
    """

    valid_ids = [opt.id for opt in question.options]
    options_text = format_options(question.options)

    max_context_chars = int(config.get("router_max_local_context_chars", 3500))
    compact_context = str(local_context or "")[:max_context_chars]

    system_msg = (
        "You are a routing judge for a multiple-choice RAG system.\n"
        "Your task is NOT to answer from general knowledge.\n"
        "Use only the provided local context.\n"
        "Decide whether the local context clearly supports exactly one option.\n"
        "Return JSON only."
    )

    user_msg = (
        f"Question:\n{question.text}\n\n"
        f"Options:\n{options_text}\n\n"
        f"Local context analyzed by router:\n{compact_context}\n\n"
        "Return exactly this JSON format:\n"
        "{\n"
        "  \"answerable\": true or false,\n"
        "  \"answer\": option_id_or_null,\n"
        "  \"confidence\": number_between_0_and_1,\n"
        "  \"reason\": \"short reason\"\n"
        "}\n\n"
        "Rules:\n"
        "- answerable=true only if the context clearly supports one option.\n"
        "- answerable=false if the context is generic, incomplete, contradictory, or supports multiple options.\n"
        "- Do not use outside knowledge."
    )

    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

    raw_output = generate_answer(
        prompt,
        tokenizer,
        model,
        do_sample=False,
        max_new_tokens=96,
        temperature=1.0,
    )

    parsed = crag_parse_answerability_router_output(raw_output, valid_ids)

    return parsed, {
        "router_system_msg": system_msg,
        "router_user_msg_preview": _dbg_multiline_clip(user_msg, 1800),
        "router_raw_output": raw_output,
        "router_context_chars": len(compact_context),
    }


def crag_repair_answer_with_local_llm_logged(question, raw_output, tokenizer, model, config):
    """
    Same role as crag_repair_answer_with_local_llm,
    but keeps the repair LLM output.
    """

    options_text = format_options(question.options)
    previous = str(raw_output or "").strip()[:1200]

    system_msg = (
        "You fix malformed multiple-choice answers. "
        "Return only one valid option ID. Do not explain."
    )

    user_msg = (
        f"Question:\n{question.text}\n\n"
        f"Options:\n{options_text}\n\n"
        f"Previous model output:\n{previous}\n\n"
        "Return only the option ID."
    )

    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

    repaired_output = generate_answer(
        prompt,
        tokenizer,
        model,
        do_sample=False,
        max_new_tokens=int(config.get("answer_repair_max_new_tokens", 12)),
        temperature=1.0,
    )

    repaired_answer = extract_option_id(
        repaired_output,
        [opt.id for opt in question.options],
    )

    return repaired_answer, repaired_output


# ============================================================
# 4. CRAG logged answer flow
#    This replaces the previous CRAG function that returned answer_id, {}
# ============================================================

def get_crag_rag_answer_logged(question, tokenizer, model, max_retries=1):
    start_total = time.perf_counter()

    config = dict(CRAG_CONFIG)
    valid_ids = [opt.id for opt in question.options]

    details = {
        "active_rag_variant": "crag",
        "prompt_mode": None,
        "context_status": None,
        "selection_mode": "crag_local_router_then_web_fallback",

        "forced_news_web": False,
        "route": None,
        "route_reason": None,

        "local_context": "",
        "web_context": "",
        "context": "",

        "router_payload": None,
        "router_raw_output": None,
        "router_user_msg_preview": None,

        "raw_output": "",
        "answer_repair_raw_output": None,
        "option_text_fallback_answer": None,

        "fallback_used": False,
        "fallback_reason": None,
        "parsing_failed": False,
    }

    forced_news_web = crag_should_force_web(question, config)
    details["forced_news_web"] = forced_news_web

    retrieval_start = time.perf_counter()

    if forced_news_web:
        details["route"] = "web"
        details["route_reason"] = "forced_web_for_news_category"

        try:
            context = crag_perform_web_search(question, tokenizer, model, config)
            details["web_context"] = context
            details["context_status"] = "web_context_forced_news"
        except Exception as exc:
            context = ""
            details["web_error"] = str(exc)
            details["context_status"] = "web_search_failed_forced_news"

        details["retrieval_time_sec"] = time.perf_counter() - retrieval_start

    else:
        local_context = crag_try_local_retrieval(question, config)
        details["local_context"] = local_context

        local_chars = len(local_context.strip())
        details["local_context_chars"] = local_chars

        local_min_chars = int(config.get("local_min_context_chars", 250))

        if local_chars < local_min_chars:
            route = "web"
            details["route_reason"] = (
                f"local_context_too_short_{local_chars}_chars_lt_{local_min_chars}"
            )
            details["router_payload"] = None

        else:
            router_start = time.perf_counter()

            router_payload, router_debug = crag_run_local_answerability_router_logged(
                question,
                local_context,
                tokenizer,
                model,
                config,
            )

            details["router_time_sec"] = time.perf_counter() - router_start
            details["router_payload"] = router_payload
            details["router_raw_output"] = router_debug.get("router_raw_output")
            details["router_user_msg_preview"] = router_debug.get("router_user_msg_preview")
            details["router_context_chars"] = router_debug.get("router_context_chars")

            confidence_score = float(router_payload.get("confidence", 0.0))
            local_router_answer = router_payload.get("answer")
            min_conf = float(config.get("local_answerability_min_confidence", 0.65))

            if (
                router_payload.get("answerable", False)
                and local_router_answer in valid_ids
                and confidence_score >= min_conf
            ):
                route = "local"
                details["route_reason"] = (
                    f"router_answerable_confidence_{confidence_score:.2f}_gte_{min_conf:.2f}"
                )
            else:
                route = "web"
                details["route_reason"] = (
                    f"router_rejected_local_confidence_{confidence_score:.2f}_lt_or_not_answerable"
                )

        details["route"] = route

        if route == "local":
            context = local_context
            details["context_status"] = "local_context_accepted_by_router"

        else:
            try:
                context = crag_perform_web_search(question, tokenizer, model, config)
                details["web_context"] = context
                details["context_status"] = "web_fallback_context"
            except Exception as exc:
                context = ""
                details["web_error"] = str(exc)
                details["context_status"] = "web_fallback_failed_empty_context"

        details["retrieval_time_sec"] = time.perf_counter() - retrieval_start

    details["context"] = context

    # --------------------------------------------------------
    # Final answer generation
    # --------------------------------------------------------

    generation_start = time.perf_counter()

    if context.strip():
        answer_id, raw_output = crag_answer_from_context(
            question,
            context,
            tokenizer,
            model,
            config,
        )
        details["prompt_mode"] = "crag_answer_from_context"
    else:
        answer_id, raw_output = crag_answer_without_context(
            question,
            tokenizer,
            model,
        )
        details["prompt_mode"] = "crag_llm_fallback_no_context"

    details["raw_output"] = raw_output
    details["generation_time_sec"] = time.perf_counter() - generation_start

    # --------------------------------------------------------
    # Parsing repair / fallback
    # --------------------------------------------------------

    if answer_id is None:
        details["parsing_failed"] = True

        if config.get("enable_answer_repair", True):
            repair_start = time.perf_counter()

            repaired, repaired_output = crag_repair_answer_with_local_llm_logged(
                question,
                raw_output,
                tokenizer,
                model,
                config,
            )

            details["answer_repair_time_sec"] = time.perf_counter() - repair_start
            details["answer_repair_raw_output"] = repaired_output

            if repaired in valid_ids:
                answer_id = repaired
                details["fallback_used"] = True
                details["fallback_reason"] = "answer_repair_llm"

        if answer_id is None and config.get("enable_direct_option_match", True):
            fallback = crag_option_text_fallback(question, context, raw_output)
            details["option_text_fallback_answer"] = fallback

            if fallback in valid_ids:
                answer_id = fallback
                details["fallback_used"] = True
                details["fallback_reason"] = "option_text_context_overlap"

        if answer_id is None:
            answer_id = valid_ids[0]
            details["fallback_used"] = True
            details["fallback_reason"] = "default_first_valid_option"

    details["answer_id"] = answer_id
    details["total_answer_time_sec"] = time.perf_counter() - start_total

    return answer_id, details


def get_crag_rag_answer(question, tokenizer, model, max_retries=1):
    answer_id, _ = get_crag_rag_answer_logged(
        question,
        tokenizer,
        model,
        max_retries=max_retries,
    )
    return answer_id


# ============================================================
# 5. CRAG pretty printer
# ============================================================

def _dbg_print_crag_details(question, answer_id, details):
    _dbg_title("CRAG debug flow", "🛠️")
    _dbg_print_question(question)

    _dbg_subtitle("Step 1 — Local retrieval / forced web decision", "🚦")

    print(f"Forced News web fallback: {details.get('forced_news_web')}")
    print(f"Route selected: {details.get('route')}")
    print(f"Route reason: {details.get('route_reason')}")
    print(f"Context status: {details.get('context_status')}")

    if details.get("local_context"):
        _dbg_print_sources_from_context(
            details.get("local_context"),
            title="Step 2 — Local Wikipedia context analyzed by CRAG",
            emoji="📄",
            max_sources=5,
            max_preview_chars=650,
        )

    if details.get("router_payload") is not None or details.get("router_raw_output"):
        _dbg_subtitle("Step 3 — Local answerability router", "🧭")

        router_payload = details.get("router_payload") or {}

        print("Parsed router decision:")
        print(json.dumps(router_payload, indent=2, ensure_ascii=False))

        print()
        print("Router LLM raw output:")
        print(_dbg_multiline_clip(details.get("router_raw_output"), 900))

    if details.get("web_context"):
        _dbg_print_sources_from_context(
            details.get("web_context"),
            title="Step 4 — Web fallback evidence",
            emoji="🌐",
            max_sources=5,
            max_preview_chars=650,
        )

    _dbg_print_sources_from_context(
        details.get("context"),
        title="Step 5 — Final context passed to answer generation",
        emoji="📚",
        max_sources=5,
        max_preview_chars=650,
    )

    _dbg_subtitle("Step 6 — Final answer generation", "🤖")
    print(f"Prompt mode: {details.get('prompt_mode')}")
    print("LLM raw output:")
    print(_dbg_multiline_clip(details.get("raw_output"), 1000))

    if details.get("answer_repair_raw_output"):
        _dbg_subtitle("Step 7 — Answer repair", "🩹")
        print("Repair LLM raw output:")
        print(_dbg_multiline_clip(details.get("answer_repair_raw_output"), 500))

    if details.get("option_text_fallback_answer") is not None:
        print()
        print(f"Option-text fallback answer: {details.get('option_text_fallback_answer')}")

    _dbg_print_final_answer(question, answer_id, details)
    _dbg_print_timings(details)


# ============================================================
# 6. Reliable-RAG pretty printer
# ============================================================

def _dbg_print_reliable_rag_details(question, answer_id, details):
    _dbg_title("Reliable-RAG debug flow", "🟦")
    _dbg_print_question(question)

    _dbg_subtitle("Step 1 — Retrieval query", "🔍")
    print(f"Query: {_dbg_clip(details.get('query'), 500)}")
    print(f"Used answer options in query: {details.get('used_options')}")

    _dbg_print_sources_from_hits(
        details.get("candidate_hits") or details.get("hits") or [],
        title="Step 2 — Retrieved Wikipedia passages",
        emoji="📄",
        max_sources=6,
        max_preview_chars=450,
    )

    _dbg_subtitle("Step 3 — LLM relevance filtering", "🧪")
    print(f"Relevant source labels: {details.get('relevant_source_labels')}")
    print(f"Relevance parsing failed: {details.get('relevance_parsing_failed')}")

    _dbg_print_sources_from_hits(
        details.get("selected_hits") or [],
        title="Step 4 — Selected passages after filtering",
        emoji="📌",
        max_sources=4,
        max_preview_chars=450,
    )

    _dbg_print_sources_from_context(
        details.get("context"),
        title="Step 5 — Final context passed to answer generation",
        emoji="📚",
        max_sources=5,
        max_preview_chars=650,
    )

    _dbg_subtitle("Step 6 — Final answer generation", "🤖")
    print(f"Prompt mode: {details.get('prompt_mode')}")
    _dbg_print_attempts(details)

    _dbg_print_final_answer(question, answer_id, details)
    _dbg_print_timings(details)


# ============================================================
# 7. Query Transformation RAG pretty printer
# ============================================================

def _dbg_print_query_transform_rag_details(question, answer_id, details):
    _dbg_title("Query Transformation RAG debug flow", "🔁")
    _dbg_print_question(question)

    _dbg_subtitle("Step 1 — Query transformations", "🔍")

    print(f"Fallback used for query transformation: {details.get('query_transform_fallback_used')}")

    raw_output = details.get("query_transform_raw_output")
    if raw_output:
        print()
        print("Raw transformation output:")
        print(_dbg_multiline_clip(raw_output, 900))

    transformations = details.get("query_transformations") or {}

    print()
    print("Parsed transformations:")
    print(json.dumps(transformations, indent=2, ensure_ascii=False))

    _dbg_subtitle("Step 2 — Final search queries", "🧾")

    query_items = details.get("query_items") or []

    if query_items:
        for item in query_items:
            print(f"- {item.get('query_type')}: {_dbg_clip(item.get('query'), 400)}")
    else:
        print("- No query items available.")

    _dbg_print_sources_from_hits(
        details.get("candidate_hits") or details.get("hits") or [],
        title="Step 3 — Retrieved Wikipedia passages",
        emoji="📄",
        max_sources=6,
        max_preview_chars=450,
    )

    _dbg_subtitle("Step 4 — LLM relevance filtering", "🧪")
    print(f"Relevant source labels: {details.get('relevant_source_labels')}")
    print(f"Relevance parsing failed: {details.get('relevance_parsing_failed')}")

    _dbg_print_sources_from_hits(
        details.get("selected_hits") or [],
        title="Step 5 — Selected passages after filtering",
        emoji="📌",
        max_sources=4,
        max_preview_chars=450,
    )

    _dbg_print_sources_from_context(
        details.get("context"),
        title="Step 6 — Final context passed to answer generation",
        emoji="📚",
        max_sources=5,
        max_preview_chars=650,
    )

    _dbg_subtitle("Step 7 — Final answer generation", "🤖")
    print(f"Prompt mode: {details.get('prompt_mode')}")
    _dbg_print_attempts(details)

    _dbg_print_final_answer(question, answer_id, details)
    _dbg_print_timings(details)


# ============================================================
# 8. RSE-RAG pretty printer
# ============================================================

def _dbg_print_rse_rag_details(question, answer_id, details):
    _dbg_title("RSE-RAG debug flow", "🧩")
    _dbg_print_question(question)

    _dbg_print_sources_from_hits(
        details.get("candidate_hits") or details.get("hits") or [],
        title="Step 1 — Retrieved candidate paragraphs",
        emoji="📄",
        max_sources=6,
        max_preview_chars=450,
    )

    _dbg_subtitle("Step 2 — Relevance / seed-hit selection", "🧪")
    print(f"Relevant source labels: {details.get('relevant_source_labels')}")
    print(f"Relevance parsing failed: {details.get('relevance_parsing_failed')}")

    _dbg_print_sources_from_hits(
        details.get("selected_hits") or [],
        title="Step 3 — Seed hits selected for RSE",
        emoji="📌",
        max_sources=5,
        max_preview_chars=450,
    )

    _dbg_subtitle("Step 4 — RSE segments", "🧱")

    rse_segments = details.get("rse_segments") or []

    if not rse_segments:
        print("No RSE segments available.")
    else:
        for segment in rse_segments:
            label = segment.get("source_label", "?")
            title = segment.get("title", "Untitled")
            start_pid = segment.get("start_paragraph_id")
            end_pid = segment.get("end_paragraph_id")
            score = segment.get("segment_score")
            score_text = f"{score:.4f}" if isinstance(score, (int, float)) else str(score)

            print(f"[{label}] {title} | paragraphs {start_pid}-{end_pid} | score={score_text}")

            for paragraph in segment.get("paragraphs", []):
                paragraph_id = paragraph.get("paragraph_id")
                marker = "retrieved" if paragraph.get("is_retrieved_hit") else "neighbor"

                print(f"    Paragraph {paragraph_id} ({marker})")
                print(f"    {_dbg_clip(paragraph.get('text'), 350)}")
                print()

    _dbg_print_sources_from_context(
        details.get("context"),
        title="Step 5 — Final RSE context passed to answer generation",
        emoji="📚",
        max_sources=5,
        max_preview_chars=650,
    )

    _dbg_subtitle("Step 6 — Final answer generation", "🤖")
    print(f"Prompt mode: {details.get('prompt_mode')}")
    _dbg_print_attempts(details)

    _dbg_print_final_answer(question, answer_id, details)
    _dbg_print_timings(details)


# ============================================================
# 9. RSE + Contextual Compression RAG pretty printer
# ============================================================

def _dbg_print_rse_compressed_rag_details(question, answer_id, details):
    _dbg_title("RSE + Contextual Compression RAG debug flow", "🗜️")
    _dbg_print_question(question)

    _dbg_print_sources_from_hits(
        details.get("candidate_hits") or details.get("hits") or [],
        title="Step 1 — Retrieved candidate paragraphs",
        emoji="📄",
        max_sources=6,
        max_preview_chars=450,
    )

    _dbg_subtitle("Step 2 — Relevance / seed-hit selection", "🧪")
    print(f"Relevant source labels: {details.get('relevant_source_labels')}")
    print(f"Relevance parsing failed: {details.get('relevance_parsing_failed')}")

    _dbg_print_sources_from_hits(
        details.get("selected_seed_hits") or details.get("selected_hits") or [],
        title="Step 3 — Seed hits selected",
        emoji="📌",
        max_sources=5,
        max_preview_chars=450,
    )

    _dbg_subtitle("Step 4 — Context before compression", "📚")
    print(_dbg_multiline_clip(details.get("context_before_compression"), 1200))

    if details.get("rse_segments"):
        _dbg_subtitle("Step 5 — RSE segments", "🧱")

        for segment in details.get("rse_segments") or []:
            label = segment.get("source_label", "?")
            title = segment.get("title", "Untitled")
            start_pid = segment.get("start_paragraph_id")
            end_pid = segment.get("end_paragraph_id")
            score = segment.get("segment_score")
            score_text = f"{score:.4f}" if isinstance(score, (int, float)) else str(score)

            print(f"[{label}] {title} | paragraphs {start_pid}-{end_pid} | score={score_text}")

            for paragraph in segment.get("paragraphs", []):
                paragraph_id = paragraph.get("paragraph_id")
                marker = "retrieved" if paragraph.get("is_retrieved_hit") else "neighbor"

                print(f"    Paragraph {paragraph_id} ({marker})")
                print(f"    {_dbg_clip(paragraph.get('text'), 350)}")
                print()

    _dbg_subtitle("Step 6 — Contextual compression", "🗜️")
    print(f"Compression used: {details.get('compression_used')}")
    print(f"Compression fallback used: {details.get('compression_fallback_used')}")

    raw_compression = details.get("compression_raw_output")
    if raw_compression:
        print()
        print("Raw compression output:")
        print(_dbg_multiline_clip(raw_compression, 900))

    _dbg_print_sources_from_context(
        details.get("context"),
        title="Step 7 — Final context passed to answer generation",
        emoji="📚",
        max_sources=5,
        max_preview_chars=650,
    )

    _dbg_subtitle("Step 8 — Final answer generation", "🤖")
    print(f"Prompt mode: {details.get('prompt_mode')}")
    _dbg_print_attempts(details)

    _dbg_print_final_answer(question, answer_id, details)
    _dbg_print_timings(details)


# ============================================================
# 10. HyDE-RAG pretty printer
# ============================================================

def _dbg_print_hyde_rag_details(question, answer_id, details):
    _dbg_title("HyDE-RAG debug flow", "🧬")
    _dbg_print_question(question)

    _dbg_subtitle("Step 1 — HyDE hypothetical document", "📝")
    print(f"HyDE fallback used: {details.get('hyde_fallback_used')}")
    print(f"HyDE query type: {details.get('hyde_query_type')}")

    raw_output = details.get("hyde_raw_output")
    if raw_output:
        print()
        print("Raw HyDE generation output:")
        print(_dbg_multiline_clip(raw_output, 900))

    print()
    print("Hypothetical document used for retrieval:")
    print(_dbg_multiline_clip(details.get("hyde_hypothetical_document"), 1000))

    _dbg_print_query_items(
        details,
        title="Step 2 — Query used for FAISS retrieval",
        emoji="🧾"
    )

    _dbg_print_sources_from_hits(
        details.get("candidate_hits") or details.get("hits") or [],
        title="Step 3 — Retrieved Wikipedia passages",
        emoji="📄",
        max_sources=6,
        max_preview_chars=450,
    )

    _dbg_subtitle("Step 4 — LLM relevance filtering", "🧪")
    print(f"Relevant source labels: {details.get('relevant_source_labels')}")
    print(f"Relevance parsing failed: {details.get('relevance_parsing_failed')}")

    _dbg_print_sources_from_hits(
        details.get("selected_hits") or [],
        title="Step 5 — Selected passages after filtering",
        emoji="📌",
        max_sources=4,
        max_preview_chars=450,
    )

    _dbg_print_sources_from_context(
        details.get("context"),
        title="Step 6 — Final context passed to answer generation",
        emoji="📚",
        max_sources=5,
        max_preview_chars=650,
    )

    _dbg_subtitle("Step 7 — Final answer generation", "🤖")
    print(f"Prompt mode: {details.get('prompt_mode')}")
    _dbg_print_attempts(details)

    _dbg_print_final_answer(question, answer_id, details)
    _dbg_print_timings(details)


# ============================================================
# 11. Fusion RAG pretty printer
# ============================================================

def _dbg_print_fusion_rag_details(question, answer_id, details):
    _dbg_title("Fusion RAG debug flow", "🔀")
    _dbg_print_question(question)
    query_details = _dbg_details_with_fusion_query_items(question, details)

    _dbg_subtitle("Step 1 — Fusion retrieval configuration", "⚙️")
    print(f"Semantic retrieval enabled: {BASE_RAG_CONFIG.get('fusion_enable_semantic')}")
    print(f"BM25 retrieval enabled: {BASE_RAG_CONFIG.get('fusion_enable_bm25')}")
    print(f"Semantic query mode: {BASE_RAG_CONFIG.get('fusion_semantic_query_mode')}")
    print(f"BM25 query mode: {BASE_RAG_CONFIG.get('fusion_bm25_query_mode')}")
    print(f"Semantic top-k: {BASE_RAG_CONFIG.get('fusion_semantic_top_k')}")
    print(f"BM25 top-k: {BASE_RAG_CONFIG.get('fusion_bm25_top_k')}")
    print(f"RRF k: {BASE_RAG_CONFIG.get('fusion_rrf_k')}")
    print(f"RSE enabled: {BASE_RAG_CONFIG.get('fusion_use_rse')}")

    _dbg_print_query_items(
        query_details,
        title="Step 1b — Queries used for retrieval",
        emoji="🧾",
    )

    _dbg_print_sources_from_hits(
        details.get("candidate_hits") or details.get("hits") or [],
        title="Step 2 — Fused candidate passages",
        emoji="📄",
        max_sources=8,
        max_preview_chars=450,
    )

    _dbg_subtitle("Step 3 — Relevance / source selection", "🧪")
    print(f"Relevant source labels: {details.get('relevant_source_labels')}")
    print(f"Relevance parsing failed: {details.get('relevance_parsing_failed')}")
    print(f"Selection mode: {details.get('selection_mode')}")

    _dbg_print_sources_from_hits(
        details.get("selected_hits") or [],
        title="Step 4 — Selected fused passages",
        emoji="📌",
        max_sources=5,
        max_preview_chars=450,
    )

    if details.get("rse_segments"):
        _dbg_subtitle("Step 5 — Optional RSE segments", "🧱")

        for segment in details.get("rse_segments") or []:
            label = segment.get("source_label", "?")
            title = segment.get("title", "Untitled")
            start_pid = segment.get("start_paragraph_id")
            end_pid = segment.get("end_paragraph_id")
            score = segment.get("segment_score")
            score_text = f"{score:.4f}" if isinstance(score, (int, float)) else str(score)

            print(f"[{label}] {title} | paragraphs {start_pid}-{end_pid} | score={score_text}")

            for paragraph in segment.get("paragraphs", []):
                paragraph_id = paragraph.get("paragraph_id")
                marker = "retrieved" if paragraph.get("is_retrieved_hit") else "neighbor"

                print(f"    Paragraph {paragraph_id} ({marker})")
                print(f"    {_dbg_clip(paragraph.get('text'), 350)}")
                print()

    _dbg_print_sources_from_context(
        details.get("context"),
        title="Step 6 — Final context passed to answer generation",
        emoji="📚",
        max_sources=5,
        max_preview_chars=650,
    )

    _dbg_subtitle("Step 7 — Final answer generation", "🤖")
    print(f"Prompt mode: {details.get('prompt_mode')}")
    print(f"Context status: {details.get('context_status')}")
    _dbg_print_attempts(details)

    _dbg_print_final_answer(question, answer_id, details)
    _dbg_print_timings(details)


# ============================================================
# 12. Fusion + Cross-Encoder RAG pretty printer
# ============================================================

def _dbg_print_fusion_cross_encoder_rag_details(question, answer_id, details):
    _dbg_title("Fusion + Cross-Encoder RAG debug flow", "🎯")
    _dbg_print_question(question)
    query_details = _dbg_details_with_fusion_query_items(question, details)

    _dbg_print_query_items(
        query_details,
        title="Step 1 — First-stage retrieval queries",
        emoji="🧾",
    )

    _dbg_print_sources_from_hits(
        details.get("candidate_hits") or details.get("hits") or [],
        title="Step 1 — First-stage Fusion retrieval",
        emoji="📄",
        max_sources=12,
        max_preview_chars=420,
    )

    _dbg_subtitle("Step 2 — Cross-Encoder selection", "🎯")
    print(f"Selection mode: {details.get('selection_mode')}")

    rerank_query = details.get("rerank_query")
    if rerank_query is None and "build_rerank_query" in globals():
        try:
            rerank_query = build_rerank_query(question)
        except Exception:
            rerank_query = None

    print(f"Rerank query: {_dbg_clip(rerank_query, 600)}")

    rerank_scores = details.get("rerank_scores") or []

    if not rerank_scores:
        rerank_scores = [
            {
                "source_label": hit.get("source_label"),
                "corpus_id": hit.get("corpus_id"),
                "rerank_score": hit.get("rerank_score"),
            }
            for hit in (details.get("selected_hits") or details.get("hits") or [])
            if hit.get("rerank_score") is not None
        ]

    if rerank_scores:
        print()
        print("Cross-encoder scores:")
        for item in rerank_scores[:12]:
            score = item.get("rerank_score")
            score_text = f"{score:.4f}" if isinstance(score, (int, float)) else str(score)
            print(
                f"- Source {item.get('source_label')} | "
                f"corpus_id={item.get('corpus_id')} | "
                f"rerank_score={score_text}"
            )

    _dbg_print_sources_from_hits(
        details.get("selected_hits") or [],
        title="Step 3 — Selected passages after reranking",
        emoji="📌",
        max_sources=5,
        max_preview_chars=500,
    )

    if details.get("rse_segments"):
        _dbg_subtitle("Step 4 — Optional RSE segments", "🧱")

        for segment in details.get("rse_segments") or []:
            label = segment.get("source_label", "?")
            title = segment.get("title", "Untitled")
            start_pid = segment.get("start_paragraph_id")
            end_pid = segment.get("end_paragraph_id")
            score = segment.get("segment_score")
            score_text = f"{score:.4f}" if isinstance(score, (int, float)) else str(score)

            print(f"[{label}] {title} | paragraphs {start_pid}-{end_pid} | score={score_text}")

            for paragraph in segment.get("paragraphs", []):
                paragraph_id = paragraph.get("paragraph_id")
                marker = "retrieved" if paragraph.get("is_retrieved_hit") else "neighbor"

                print(f"    Paragraph {paragraph_id} ({marker})")
                print(f"    {_dbg_clip(paragraph.get('text'), 350)}")
                print()

    _dbg_print_sources_from_context(
        details.get("context"),
        title="Step 5 — Final context passed to answer generation",
        emoji="📚",
        max_sources=5,
        max_preview_chars=650,
    )

    _dbg_subtitle("Step 6 — Final answer generation", "🤖")
    print(f"Prompt mode: {details.get('prompt_mode')}")
    _dbg_print_attempts(details)

    _dbg_print_final_answer(question, answer_id, details)
    _dbg_print_timings(details)


# ============================================================
# 13. Adaptive pretty printer
# ============================================================

def _dbg_print_adaptive_details(question, answer_id, details):
    _dbg_title("Adaptive Best-Config RAG debug flow", "🧭")
    _dbg_print_question(question)

    _dbg_subtitle("Step 1 — Question profile classification", "🧠")
    print(f"Adaptive profile: {details.get('adaptive_profile')}")
    print(f"Reason: {details.get('adaptive_reason')}")

    _dbg_subtitle("Step 2 — Candidate technologies", "🧰")
    candidates = details.get("adaptive_candidate_technologies") or []

    if candidates:
        for tech in candidates:
            print(f"- {tech}")
    else:
        print("- No candidate list available.")

    _dbg_subtitle("Step 3 — Selected configuration", "🏆")
    print(f"Selected technology: {details.get('adaptive_selected_technology')}")
    print(f"Selected config name: {details.get('adaptive_selected_config_name')}")
    print(f"Selected pretty name: {details.get('adaptive_selected_pretty_name')}")
    print(f"Expected accuracy: {details.get('adaptive_selection_accuracy')}")
    print(f"Expected avg time: {details.get('adaptive_selection_avg_time_sec')}")
    print(f"Actually activated config: {details.get('adaptive_actually_activated_config')}")
    print(f"Activation source: {details.get('adaptive_actually_activated_source')}")

    selected_technology = details.get("adaptive_selected_technology")

    _dbg_subtitle("Step 4 — Execution of selected technology", "⚙️")
    print(f"Selected technology executed: {selected_technology}")
    print(f"Underlying active RAG variant: {details.get('active_rag_variant')}")
    print(f"Prompt mode: {details.get('prompt_mode')}")
    print(f"Context status: {details.get('context_status')}")
    print(f"Selection mode: {details.get('selection_mode')}")

    # If Adaptive selected Self-RAG, print the Self-RAG internals too.
    if selected_technology == "self_rag":
        print()
        print("Adaptive selected Self-RAG, so the detailed Self-RAG flow follows.")
        _dbg_print_self_rag_details(question, answer_id, details)
        return

    # If Adaptive selected CRAG, print the CRAG internals too.
    if selected_technology == "crag":
        print()
        print("Adaptive selected CRAG, so the detailed CRAG flow follows.")
        _dbg_print_crag_details(question, answer_id, details)
        return

    # If Adaptive selected Reliable-RAG, print the Reliable-RAG internals too.
    if selected_technology == "reliable_rag":
        print()
        print("Adaptive selected Reliable-RAG, so the detailed Reliable-RAG flow follows.")
        _dbg_print_reliable_rag_details(question, answer_id, details)
        return

    # If Adaptive selected Query Transformation RAG, print its internals too.
    if selected_technology in {"query_transform_rag", "query_transformation_rag"}:
        print()
        print("Adaptive selected Query Transformation RAG, so the detailed Query Transformation flow follows.")
        _dbg_print_query_transform_rag_details(question, answer_id, details)
        return

    # If Adaptive selected HyDE-RAG, print its internals too.
    if selected_technology == "hyde_rag":
        print()
        print("Adaptive selected HyDE-RAG, so the detailed HyDE flow follows.")
        _dbg_print_hyde_rag_details(question, answer_id, details)
        return

    # If Adaptive selected RSE-RAG, print its internals too.
    if selected_technology == "rse_rag":
        print()
        print("Adaptive selected RSE-RAG, so the detailed RSE flow follows.")
        _dbg_print_rse_rag_details(question, answer_id, details)
        return

    # If Adaptive selected RSE + Contextual Compression RAG, print its internals too.
    if selected_technology == "rse_contextual_compression_rag":
        print()
        print("Adaptive selected RSE + Contextual Compression RAG, so the detailed flow follows.")
        _dbg_print_rse_compressed_rag_details(question, answer_id, details)
        return

    # If Adaptive selected plain Contextual Compression RAG, print the compression flow too.
    if selected_technology == "contextual_compression_rag":
        print()
        print("Adaptive selected Contextual Compression RAG, so the detailed compression flow follows.")
        _dbg_print_rse_compressed_rag_details(question, answer_id, details)
        return

    # If Adaptive selected Fusion RAG, print its internals too.
    if selected_technology == "fusion_rag":
        print()
        print("Adaptive selected Fusion RAG, so the detailed Fusion flow follows.")
        _dbg_print_fusion_rag_details(question, answer_id, details)
        return

    # If Adaptive selected Fusion + Cross-Encoder RAG, print its internals too.
    if selected_technology == "fusion_cross_encoder_rag":
        print()
        print("Adaptive selected Fusion + Cross-Encoder RAG, so the detailed Cross-Encoder flow follows.")
        _dbg_print_fusion_cross_encoder_rag_details(question, answer_id, details)
        return

    # Generic fallback for other selected technologies.
    _dbg_print_sources_from_hits(
        details.get("hits") or details.get("candidate_hits") or [],
        title="Retrieved passages from selected technology",
        emoji="📄",
        max_sources=6,
        max_preview_chars=450,
    )

    _dbg_print_sources_from_hits(
        details.get("selected_hits") or details.get("filtered_hits") or [],
        title="Selected passages",
        emoji="📌",
        max_sources=4,
        max_preview_chars=450,
    )

    _dbg_print_sources_from_context(
        details.get("context"),
        title="Final context",
        emoji="📚",
        max_sources=5,
        max_preview_chars=650,
    )

    _dbg_print_attempts(details)
    _dbg_print_final_answer(question, answer_id, details)
    _dbg_print_timings(details)


# ============================================================
# 14. Public advanced debugger
# ============================================================

_ADVANCED_DEBUG_SUPPORTED_CONFIGURATIONS = {
    "adaptive_best_config_rag",
    "self_rag",
    "crag",
    "reliable_rag",
    "query_transform_rag",
    "query_transformation_rag",
    "hyde_rag",
    "rse_rag",
    "contextual_compression_rag",
    "rse_contextual_compression_rag",
    "fusion_rag",
    "fusion_cross_encoder_rag",
}


def _dbg_normalize_selected_configuration(selected_configuration):
    if selected_configuration == "query_transform_rag":
        return "query_transformation_rag"
    return selected_configuration


def print_advanced_rag_debug(question, answer_id, details):
    """
    Clean debug printer for the advanced architectures:
    - adaptive_best_config_rag
    - self_rag
    - crag
    - reliable_rag
    - query_transform_rag
    - hyde_rag
    - rse_rag
    - contextual_compression_rag
    - rse_contextual_compression_rag
    - fusion_rag
    - fusion_cross_encoder_rag

    It also works as a generic fallback for other logged RAG variants.
    """

    details = details or {}

    active_variant = details.get("active_rag_variant")
    adaptive_selected = details.get("adaptive_selected_technology")

    if active_variant == "self_rag" or adaptive_selected == "self_rag":
        _dbg_print_self_rag_details(question, answer_id, details)
        return

    if active_variant == "crag" or adaptive_selected == "crag":
        _dbg_print_crag_details(question, answer_id, details)
        return

    if active_variant == "reliable_rag" or adaptive_selected == "reliable_rag":
        _dbg_print_reliable_rag_details(question, answer_id, details)
        return

    if active_variant in {"query_transform_rag", "query_transformation_rag"} or adaptive_selected in {
        "query_transform_rag",
        "query_transformation_rag",
    }:
        _dbg_print_query_transform_rag_details(question, answer_id, details)
        return

    if active_variant == "hyde_rag" or adaptive_selected == "hyde_rag":
        _dbg_print_hyde_rag_details(question, answer_id, details)
        return

    if active_variant == "rse_rag" or adaptive_selected == "rse_rag":
        _dbg_print_rse_rag_details(question, answer_id, details)
        return

    if (
        active_variant == "rse_contextual_compression_rag"
        or adaptive_selected == "rse_contextual_compression_rag"
    ):
        _dbg_print_rse_compressed_rag_details(question, answer_id, details)
        return

    if (
        active_variant == "contextual_compression_rag"
        or adaptive_selected == "contextual_compression_rag"
    ):
        _dbg_print_rse_compressed_rag_details(question, answer_id, details)
        return

    if active_variant == "fusion_rag" or adaptive_selected == "fusion_rag":
        _dbg_print_fusion_rag_details(question, answer_id, details)
        return

    if (
        active_variant == "fusion_cross_encoder_rag"
        or adaptive_selected == "fusion_cross_encoder_rag"
    ):
        _dbg_print_fusion_cross_encoder_rag_details(question, answer_id, details)
        return

    if (
        active_variant == "adaptive_best_config_rag"
        or details.get("adaptive_profile") is not None
        or details.get("adaptive_selected_technology") is not None
    ):
        _dbg_print_adaptive_details(question, answer_id, details)
        return

    # Generic fallback
    _dbg_title("Logged RAG debug flow", "🔎")
    _dbg_print_question(question)
    _dbg_print_sources_from_hits(details.get("hits") or [], "Retrieved passages", "📄")
    _dbg_print_sources_from_context(details.get("context"), "Final context", "📚")
    _dbg_print_attempts(details)
    _dbg_print_final_answer(question, answer_id, details)
    _dbg_print_timings(details)


def answer_question_with_advanced_debug(
    question,
    selected_configuration="adaptive_best_config_rag",
    tokenizer=None,
    model=None,
    use_best_if_available=True,
    max_retries=3,
    competition_id=None,
    category_name=None,
    use_news_specific_if_available=True,
):
    """
    Runs one question with one of the advanced architectures and prints
    a clean, structured debug trace.

    Recommended values:
        selected_configuration="adaptive_best_config_rag"
        selected_configuration="self_rag"
        selected_configuration="crag"
        selected_configuration="reliable_rag"
        selected_configuration="query_transformation_rag"
        selected_configuration="hyde_rag"
        selected_configuration="rse_rag"
        selected_configuration="contextual_compression_rag"
        selected_configuration="rse_contextual_compression_rag"
        selected_configuration="fusion_rag"
        selected_configuration="fusion_cross_encoder_rag"

    Returns:
        answer_id, details
    """

    if tokenizer is None:
        tokenizer = llm_tokenizer

    if model is None:
        model = llm_model

    if selected_configuration not in _ADVANCED_DEBUG_SUPPORTED_CONFIGURATIONS:
        raise ValueError(
            "This clean debugger is intended for: "
            "'adaptive_best_config_rag', 'self_rag', 'crag', "
            "'reliable_rag', 'query_transformation_rag', 'hyde_rag', "
            "'rse_rag', 'contextual_compression_rag', "
            "'rse_contextual_compression_rag', 'fusion_rag', or "
            "'fusion_cross_encoder_rag'."
        )

    activation_configuration = _dbg_normalize_selected_configuration(selected_configuration)

    old_debug = BASE_RAG_CONFIG.get("debug", True)
    old_crag_debug = CRAG_CONFIG.get("debug", True) if "CRAG_CONFIG" in globals() else True

    try:
        BASE_RAG_CONFIG["debug"] = False

        if "CRAG_CONFIG" in globals():
            CRAG_CONFIG["debug"] = False

        # Activate selected architecture.
        try:
            activate_selected_configuration(
                activation_configuration,
                use_best_if_available=use_best_if_available,
                competition_id=competition_id,
                category_name=category_name,
                use_news_specific_if_available=use_news_specific_if_available,
            )
        except TypeError:
            activate_selected_configuration(
                activation_configuration,
                use_best_if_available=use_best_if_available,
            )

        # Use logged path when available.
        if selected_configuration == "adaptive_best_config_rag":
            answer_id, details = get_adaptive_best_config_answer_logged(
                question,
                tokenizer,
                model,
                max_retries=max_retries,
            )

        elif selected_configuration == "self_rag":
            answer_id, details = get_self_rag_answer_logged(
                question,
                tokenizer,
                model,
                max_retries=max_retries,
            )

        elif selected_configuration == "crag":
            answer_id, details = get_crag_rag_answer_logged(
                question,
                tokenizer,
                model,
                max_retries=max_retries,
            )

        elif selected_configuration == "reliable_rag":
            answer_id, details = get_reliable_rag_answer_logged(
                question,
                tokenizer,
                model,
                max_retries=max_retries,
            )

        elif selected_configuration in {"query_transform_rag", "query_transformation_rag"}:
            answer_id, details = get_query_transform_rag_answer_logged(
                question,
                tokenizer,
                model,
                max_retries=max_retries,
            )

        elif selected_configuration == "hyde_rag":
            answer_id, details = get_rag_answer_logged(
                question,
                tokenizer,
                model,
                max_retries=max_retries,
            )

        elif selected_configuration == "rse_rag":
            answer_id, details = get_rse_rag_answer_logged(
                question,
                tokenizer,
                model,
                max_retries=max_retries,
            )

        elif selected_configuration == "rse_contextual_compression_rag":
            answer_id, details = get_rse_compressed_rag_answer_logged(
                question,
                tokenizer,
                model,
                max_retries=max_retries,
            )

        elif selected_configuration == "contextual_compression_rag":
            answer_id, details = get_rse_compressed_rag_answer_logged(
                question,
                tokenizer,
                model,
                max_retries=max_retries,
            )

        elif selected_configuration == "fusion_rag":
            answer_id, details = get_fusion_rag_answer_logged(
                question,
                tokenizer,
                model,
                max_retries=max_retries,
            )

        elif selected_configuration == "fusion_cross_encoder_rag":
            answer_id, details = get_fusion_rerank_rag_answer_logged(
                question,
                tokenizer,
                model,
                max_retries=max_retries,
            )

        print_advanced_rag_debug(question, answer_id, details)

        return answer_id, details

    finally:
        BASE_RAG_CONFIG["debug"] = old_debug

        if "CRAG_CONFIG" in globals():
            CRAG_CONFIG["debug"] = old_crag_debug


print("Advanced RAG debugger loaded.")
print("Use: answer_question_with_advanced_debug(question, selected_configuration='self_rag')")
print("Use: answer_question_with_advanced_debug(question, selected_configuration='crag')")
print("Use: answer_question_with_advanced_debug(question, selected_configuration='reliable_rag')")
print("Use: answer_question_with_advanced_debug(question, selected_configuration='query_transformation_rag')")
print("Use: answer_question_with_advanced_debug(question, selected_configuration='hyde_rag')")
print("Use: answer_question_with_advanced_debug(question, selected_configuration='rse_rag')")
print("Use: answer_question_with_advanced_debug(question, selected_configuration='contextual_compression_rag')")
print("Use: answer_question_with_advanced_debug(question, selected_configuration='rse_contextual_compression_rag')")
print("Use: answer_question_with_advanced_debug(question, selected_configuration='fusion_rag')")
print("Use: answer_question_with_advanced_debug(question, selected_configuration='fusion_cross_encoder_rag')")
print("Use: answer_question_with_advanced_debug(question, selected_configuration='adaptive_best_config_rag')")


# ============================================================
# PLAY SELECTED CONFIGURATION WITH ADVANCED DEBUG
# ============================================================

def _advanced_debug_answer_dispatcher(question, tokenizer, model, max_retries=3):
    """
    Dispatcher used during the game.
    It calls the active logged RAG function and prints the clean advanced debug.
    """

    answer_id, details = get_rag_answer_logged(
        question,
        tokenizer,
        model,
        max_retries=max_retries,
    )

    print_advanced_rag_debug(question, answer_id, details)

    return answer_id


def play_selected_configuration_debug(
    selected_configuration,
    use_best_if_available=True,
    competition_id=None,
    category_name=None,
    use_news_specific_if_available=True,
):
    """
    Same idea as play_selected_configuration(...),
    but with the clean advanced debug output.

    Recommended for:
        - "adaptive_best_config_rag"
        - "self_rag"
        - "crag"
        - "reliable_rag"
        - "query_transformation_rag"
        - "hyde_rag"
        - "rse_rag"
        - "contextual_compression_rag"
        - "rse_contextual_compression_rag"
        - "fusion_rag"
        - "fusion_cross_encoder_rag"
    """

    global get_rag_answer

    if competition_id is None and "comp_id" in globals():
        competition_id = comp_id

    if category_name is None and "chosen_comp" in globals():
        category_name = chosen_comp

    if selected_configuration not in _ADVANCED_DEBUG_SUPPORTED_CONFIGURATIONS:
        raise ValueError(
            "This debug play function is intended for: "
            "'adaptive_best_config_rag', 'self_rag', 'crag', "
            "'reliable_rag', 'query_transformation_rag', 'hyde_rag', "
            "'rse_rag', 'contextual_compression_rag', "
            "'rse_contextual_compression_rag', 'fusion_rag', or "
            "'fusion_cross_encoder_rag'."
        )

    activation_configuration = _dbg_normalize_selected_configuration(selected_configuration)

    old_debug = BASE_RAG_CONFIG.get("debug", True)
    old_crag_debug = CRAG_CONFIG.get("debug", True) if "CRAG_CONFIG" in globals() else True
    old_get_rag_answer = globals().get("get_rag_answer")

    try:
        # Disable the old noisy debug.
        BASE_RAG_CONFIG["debug"] = False

        if "CRAG_CONFIG" in globals():
            CRAG_CONFIG["debug"] = False

        # Activate the selected configuration.
        try:
            activate_selected_configuration(
                activation_configuration,
                use_best_if_available=use_best_if_available,
                competition_id=competition_id,
                category_name=category_name,
                use_news_specific_if_available=use_news_specific_if_available,
            )
        except TypeError:
            activate_selected_configuration(
                activation_configuration,
                use_best_if_available=use_best_if_available,
            )

        # Replace the normal answer function with the clean debug dispatcher.
        get_rag_answer = _advanced_debug_answer_dispatcher

        print("=" * 90)
        print("ADVANCED DEBUG PLAY MODE")
        print("=" * 90)
        print(f"Selected configuration: {selected_configuration}")
        if activation_configuration != selected_configuration:
            print(f"Activation configuration: {activation_configuration}")
        print(f"Use best if available: {use_best_if_available}")
        print(f"Competition id: {competition_id}")
        print(f"Category name: {category_name}")
        print("=" * 90)

        # Start game using the same logic as your normal RAG play function.
        game = client.game.start(competition_id=competition_id)

        play_game_rag(
            game,
            llm_tokenizer,
            llm_model,
        )

    finally:
        BASE_RAG_CONFIG["debug"] = old_debug

        if "CRAG_CONFIG" in globals():
            CRAG_CONFIG["debug"] = old_crag_debug

        if old_get_rag_answer is None:
            globals().pop("get_rag_answer", None)
        else:
            get_rag_answer = old_get_rag_answer


Advanced RAG debugger loaded.
Use: answer_question_with_advanced_debug(question, selected_configuration='self_rag')
Use: answer_question_with_advanced_debug(question, selected_configuration='crag')
Use: answer_question_with_advanced_debug(question, selected_configuration='reliable_rag')
Use: answer_question_with_advanced_debug(question, selected_configuration='query_transformation_rag')
Use: answer_question_with_advanced_debug(question, selected_configuration='hyde_rag')
Use: answer_question_with_advanced_debug(question, selected_configuration='rse_rag')
Use: answer_question_with_advanced_debug(question, selected_configuration='contextual_compression_rag')
Use: answer_question_with_advanced_debug(question, selected_configuration='rse_contextual_compression_rag')
Use: answer_question_with_advanced_debug(question, selected_configuration='fusion_rag')
Use: answer_question_with_advanced_debug(question, selected_configuration='fusion_cross_encoder_rag')
Use: answer_question_with_advanced_de

# 🎮 Play


In [ ]:
# 1. Load and display best configs for the current model
BEST_CONFIGS_FOR_CURRENT_MODEL = refresh_best_configs_for_current_model(
    verbose=True,
    display_table=True,
)

Current model:
Qwen/Qwen2.5-3B-Instruct

Benchmark hash:
208f033deb4f6166cd10ce46fc8eb4c2

Current model hash:
dc7c68033a18f81696395420280d55a4

Loaded 6 configs from rag_baseline_reliable
Exact current hash match: True
Summary path: /content/gdrive/MyDrive/NLP/PoliMillionaire/ASSO_RAG/Benchmark/benchmark_cache/rag_baseline_reliable/benchmark_208f033deb4f6166cd10ce46fc8eb4c2/model_dc7c68033a18f81696395420280d55a4/summary/rag_benchmark_summary.json

Loaded 8 configs from query_enhancement
Exact current hash match: True
Summary path: /content/gdrive/MyDrive/NLP/PoliMillionaire/ASSO_RAG/Benchmark/benchmark_cache/query_enhancement/benchmark_208f033deb4f6166cd10ce46fc8eb4c2/model_dc7c68033a18f81696395420280d55a4/summary/query_enhancement_benchmark_summary.json

Loaded 8 configs from context_enrichment
Exact current hash match: True
Summary path: /content/gdrive/MyDrive/NLP/PoliMillionaire/ASSO_RAG/Benchmark/benchmark_cache/context_enrichment/benchmark_208f033deb4f6166cd10ce46fc8eb4c2/model_

,RAG key,Technology,Best configuration,Config name,Correct,Accuracy,Avg. time
1,crag,CRAG,"CRAG, local-first + web fallback, semantic q+o...",crag_local_web_qopts,17/20,85%,9.89s
8,reliable_rag,Reliable-RAG,"Reliable-RAG, question-only, top-4",reliable_qonly_top4,16/20,80%,3.35s
3,fusion_rag,Fusion / Advanced Retrieval RAG,"Fusion Retrieval, semantic q+opts + BM25 q-onl...",fusion_sem_qopts_bm25_qonly_top8,16/20,80%,4.27s
0,contextual_compression_rag,Contextual Compression RAG,"Contextual Compression, no RSE, top-6, relevan...",compression_qopts_top6_rel_no_rse,16/20,80%,5.26s
9,rse_contextual_compression_rag,RSE + Contextual Compression RAG,"RSE + Compression, top-6, relevance filter",rse_compression_qopts_top6_rel_total5,16/20,80%,6.18s
4,hyde_rag,HyDE-RAG,"HyDE, with options, top-4, no relevance filter",hyde_with_options_top4_no_rel,16/20,80%,6.87s
5,query_transformation_rag,Query Transformation RAG,"Query Transformation, question + options, top-...",qt_qopts_top6_rel,16/20,80%,10.91s
7,rag_question_options,Baseline RAG question + options,"Baseline RAG, question + options, top-4",baseline_qopts_top4,15/20,75%,1.35s
6,rag_baseline_question_only,Baseline RAG question-only,"Baseline RAG, question-only, top-4",baseline_qonly_top4,15/20,75%,1.52s
11,self_rag,Self-RAG,"Self-RAG, top-6 retrieval, max 2 final contexts",self_rag_top6_ctx2,15/20,75%,4.26s



Saved best configs JSON:
/content/gdrive/MyDrive/NLP/PoliMillionaire/ASSO_RAG/Benchmark/best_configs/benchmark_208f033deb4f6166cd10ce46fc8eb4c2/model_dc7c68033a18f81696395420280d55a4/best_configs_by_play_key.json

Saved best configs CSV:
/content/gdrive/MyDrive/NLP/PoliMillionaire/ASSO_RAG/Benchmark/best_configs/benchmark_208f033deb4f6166cd10ce46fc8eb4c2/model_dc7c68033a18f81696395420280d55a4/best_configs_by_play_key.csv

Loaded News-specific benchmark summaries
Rows: 15
Exact current hash match: False
Summary path:
/content/gdrive/MyDrive/NLP/PoliMillionaire/ASSO_RAG/Benchmark/benchmark_cache/news_web_crag_eval/benchmark_4b830bd491accec0e984febd6b6cd7ef/model_4046e968d3f616312c3ad7b39ace3f26/summary/news_web_crag_all_configs_with_true_chunking_summary.json

Best News-specific CRAG configuration


,Category,RAG key,Best configuration,Config name,Correct,Accuracy,Avg. time,Max time,Selection rule
0,News,crag,"News force-web, LLM query rewrite, top-4",news_force_web_rewrite_top4,11/15,73%,6.95s,9.62s,"accuracy first, avg time tie-breaker, max_time..."


In [ ]:
# "llm"
# "rag_baseline_question_only"
# "rag_question_options"
# "reliable_rag"
# "query_transformation_rag"
# "hyde_rag"
# "rse_rag"
# "contextual_compression_rag"
# "rse_contextual_compression_rag"
# "fusion_rag"
# "fusion_cross_encoder_rag"
# "adaptive_best_config_rag"
# "self_rag"
# "crag"

## Match

In [ ]:
# 1. Choose what to play
SELECTED_CONFIGURATION = "crag"

# 2. Activate and play with clean debug
play_selected_configuration_debug(
    SELECTED_CONFIGURATION,
    use_best_if_available=True,
)

Using cached best configuration:
- RAG key: crag
- Config: crag_local_web_qopts
- Label: CRAG, local-first + web fallback, semantic q+opts + BM25 q-only
- Accuracy: 85%
- Avg time: 9.89s

Selected configuration: crag
Configuration source: cached_best
ADVANCED DEBUG PLAY MODE
Selected configuration: crag
Use best if available: True
Competition id: 4
Category name: Ancient History and Politics

--- Level 1 ---
Q: Which of the following best describes dehumanization?

  [0] The process of treating individuals or groups as less than human
  [1] The process of improving the social status of a group
  [2] The process of giving human traits to non-human entities
  [3] The process of teaching moral values to individuals

Time remaining: 29.9s
Active strategy: local_wikipedia_rag

🛠️ CRAG debug flow

❓ Question
------------------------------------------------------------------------------------------
Which of the following best describes dehumanization?

Options:
  [0] The process of treating i